In [1]:
#cutoff:1-47/48-77
from pathlib import Path
import csv

In [2]:
#import txt檔案
input_path = (
    Path.home()
    / "Desktop"
    / "summerintern"
    / "10_genotyping_data.txt"
)

output_folder = (
    Path.home()
    / "Desktop"
    / "summerintern"
    / "split_data"
)

output_folder.mkdir(parents=True, exist_ok=True)

In [3]:
#fornt_path:1/47
#back_path:48-77

front_path = output_folder / "genotype_front_1_47.txt"
back_path = output_folder / "genotype_back_48_77.txt"
metadata_path = output_folder / "metadata.txt"
report_path = output_folder / "split_report.txt"

In [4]:
import csv

# 設定原始 TXT 每一列理論上應有的欄位數
expected_columns = 77

# 設定前段要保留到第 47 欄
# Python 切片不包含結束位置，因此 row[:47] 會取得第 1–47 欄
front_end = 47

# 顯示設定結果，確認沒有輸入錯誤
print("每列預期欄位數：", expected_columns)
print("前段欄位數：", front_end)
print("後段欄位數：", expected_columns - front_end)

每列預期欄位數： 77
前段欄位數： 47
後段欄位數： 30


In [5]:
# metadata 列數
metadata_count = 0

#記錄表頭與 SNP 資料列數
data_count = 0

# 記錄欄位數不是 77 的異常列
bad_rows = []


In [6]:
# 同時開啟原始 TXT、前段輸出檔、後段輸出檔與 metadata 輸出檔
with input_path.open(
    "r",                    
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 遇到無法解碼的字元時，用替代符號取代
    newline=""              # 保留原始換行處理方式
) as infile, \
front_path.open(
    "w",                    # 以寫入模式建立前段檔案
    encoding="utf-8",
    newline=""
) as front_file, \
back_path.open(
    "w",                    # 以寫入模式建立後段檔案
    encoding="utf-8",
    newline=""
) as back_file, \
metadata_path.open(
    "w",                    # 以寫入模式建立 metadata 檔案
    encoding="utf-8",
    newline=""
) as metadata_file:

    # 建立 Tab 分隔的讀取器
    reader = csv.reader(
        infile,
        delimiter="\t",     # 指定 Tab 為欄位分隔符號
        quoting=csv.QUOTE_NONE
    )

    # 建立前段檔案的寫入器
    front_writer = csv.writer(
        front_file,
        delimiter="\t",     # 輸出仍使用 Tab 分隔
        quoting=csv.QUOTE_NONE,
        escapechar="\\",    # 特殊字元使用反斜線處理
        lineterminator="\n" # 每列結尾使用換行符號
    )

    # 建立後段檔案的寫入器
    back_writer = csv.writer(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 逐列讀取原始 TXT
    for physical_line_number, row in enumerate(reader, start=1):

        # 如果該列以 ## 開頭，代表是 metadata
        if row and row[0].startswith("##"):

            # 將 metadata 原樣寫入 metadata.txt
            metadata_file.write("\t".join(row) + "\n")

            # metadata 列數加 1
            metadata_count += 1

            # 跳過後面的切割步驟，直接處理下一列
            continue

        # 記錄表頭與 SNP 資料列數
        data_count += 1

        # 檢查該列欄位數是否為 77
        if len(row) != expected_columns:

            # 記錄異常列的實體列號、欄位數與第一欄內容
            bad_rows.append(
                (
                    physical_line_number,
                    len(row),
                    row[0] if row else ""
                )
            )

        # 將第 1–47 欄寫入前段檔案
        front_writer.writerow(row[:front_end])

        # 將第 48–77 欄寫入後段檔案
        back_writer.writerow(row[front_end:])

# 顯示切割結果
print("切割完成")
print("metadata 列數：", metadata_count)
print("表頭與資料列數：", data_count)
print("欄位數不是 77 的列數：", len(bad_rows))
print("前段檔案：", front_path)
print("後段檔案：", back_path)
print("metadata 檔案：", metadata_path)

切割完成
metadata 列數： 5
表頭與資料列數： 680866
欄位數不是 77 的列數： 0
前段檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_front_1_47.txt
後段檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_48_77.txt
metadata 檔案： /Users/sheena/Desktop/summerintern/split_data/metadata.txt


In [7]:
# 讀取前段檔案的第一列 欄位名稱
with front_path.open(
    "r",                    
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace"        # 無法解碼的字元以替代符號顯示
) as front_file:

    # 讀取第一列
    front_header_line = front_file.readline()

    # 移除換行符號，再用 Tab 切成欄位清單
    front_header = front_header_line.rstrip("\r\n").split("\t")


# 讀取後段檔案的第一列，也就是欄位名稱
with back_path.open(
    "r",                    # 以讀取模式開啟後段檔案
    encoding="utf-8",
    errors="replace"
) as back_file:

    # 讀取第一列
    back_header_line = back_file.readline()

    # 移除換行符號，再用 Tab 切成欄位清單
    back_header = back_header_line.rstrip("\r\n").split("\t")


# 顯示前段檢查結果
print("前段欄位數：", len(front_header))
print("前段第一欄：", front_header[0])
print("前段最後一欄：", front_header[-1])

# 空一行，讓輸出比較容易閱讀
print()

# 顯示後段檢查結果
print("後段欄位數：", len(back_header))
print("後段第一欄：", back_header[0])
print("後段最後一欄：", back_header[-1])

前段欄位數： 47
前段第一欄： probeset_id
前段最後一欄： CR

後段欄位數： 30
後段第一欄： FLD
後段最後一欄： Call Modified


In [8]:
#找出後段資料中常見的排列模式
# 建立空字典，用來統計後段資料中各種「欄位內容模式」出現的次數
pattern_counts = {}

# genotype_back_48_77.txt
with back_path.open(
    "r",                    # 以讀取模式開啟檔案
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 遇到無法解碼的字元時，用替代符號取代
    newline=""              # 保留原始換行處理
) as back_file:

    # 建立以 Tab 為分隔符號的讀取器
    reader = csv.reader(
        back_file,
        delimiter="\t",     # 指定 Tab 為欄位分隔符號
        quoting=csv.QUOTE_NONE
    )

    # 讀取第一列欄位名稱
    back_header = next(reader)

    # 逐列讀取後段 SNP 資料
    for row_number, row in enumerate(reader, start=1):

        # 如果該列欄位不足 30 欄，先補空字串
        # 這樣後面取固定位置時不會發生 IndexError
        if len(row) < 30:
            row = row + [""] * (30 - len(row))

        # 取出幾個具有辨識力的欄位內容
        # 後段第 9 欄原本對應 n_BB
        n_BB_value = row[8]

        # 後段第 10 欄原本對應 n_NC
        n_NC_value = row[9]

        # 後段第 11 欄原本對應 hemizygous
        hemizygous_value = row[10]

        # 後段第 12 欄原本對應 specialSNP_chr
        special_chr_value = row[11]

        # 後段第 13 欄原本對應 gender_metrics
        gender_value = row[12]

        # 後段第 14 欄原本對應 ConversionType
        conversion_value = row[13]

        # 將這六個欄位組合成一個 pattern
        pattern = (
            n_BB_value,
            n_NC_value,
            hemizygous_value,
            special_chr_value,
            gender_value,
            conversion_value
        )

        # 如果這個 pattern 第一次出現，就先設定次數為 0
        if pattern not in pattern_counts:
            pattern_counts[pattern] = 0

        # 該 pattern 出現次數加 1
        pattern_counts[pattern] += 1


# 將所有 pattern 依出現次數由大到小排序
sorted_patterns = sorted(
    pattern_counts.items(),
    key=lambda item: item[1],
    reverse=True
)


# 顯示前 20 種最常見的 pattern
print("最常見的 20 種後段欄位模式：")
print()

for rank, (pattern, count) in enumerate(
    sorted_patterns[:20],
    start=1
):
    print(f"第 {rank} 名，出現次數：{count}")

    print("n_BB位置：", repr(pattern[0]))
    print("n_NC位置：", repr(pattern[1]))
    print("hemizygous位置：", repr(pattern[2]))
    print("specialSNP_chr位置：", repr(pattern[3]))
    print("gender_metrics位置：", repr(pattern[4]))
    print("ConversionType位置：", repr(pattern[5]))

    print("-" * 50)

最常見的 20 種後段欄位模式：

第 1 名，出現次數：186383
n_BB位置： '0'
n_NC位置： '0'
hemizygous位置： 'autosomal'
specialSNP_chr位置： 'all'
gender_metrics位置： 'NoMinorHom'
ConversionType位置： '1'
--------------------------------------------------
第 2 名，出現次數：131659
n_BB位置： 'autosomal'
n_NC位置： 'all'
hemizygous位置： 'MonoHighResolution'
specialSNP_chr位置： '1'
gender_metrics位置： '1'
ConversionType位置： '0'
--------------------------------------------------
第 3 名，出現次數：26229
n_BB位置： '1'
n_NC位置： '0'
hemizygous位置： 'autosomal'
specialSNP_chr位置： 'all'
gender_metrics位置： 'NoMinorHom'
ConversionType位置： '1'
--------------------------------------------------
第 4 名，出現次數：17794
n_BB位置： '1'
n_NC位置： '0'
hemizygous位置： '0'
specialSNP_chr位置： 'autosomal'
gender_metrics位置： 'all'
ConversionType位置： 'PolyHighResolution'
--------------------------------------------------
第 5 名，出現次數：12292
n_BB位置： 'Y'
n_NC位置： 'male'
hemizygous位置： 'MonoHighResolution'
specialSNP_chr位置： '1'
gender_metrics位置： '1'
ConversionType位置： '0'
---------------------------------------

In [9]:
# 設定 specialSNP_chr 合理值
valid_chr_types = {"autosomal", "X", "Y", "MT"}

# 設定 gender_metrics 合理值
valid_gender_types = {"all", "male", "female"}

# 建立字典，統計不同位移量的列數
shift_counts = {}

# 建立字典，記錄無法辨識的列
unrecognized_rows = []

# 開啟後段資料
with back_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼字元以替代符號呈現
    newline=""
) as back_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取並跳過表頭
    back_header = next(reader)

    # specialSNP_chr 在正常後段資料中應位於第 12 欄
    # Python index 從 0 開始，所以位置是 11
    expected_special_chr_index = 11

    # 逐列檢查
    for row_number, row in enumerate(reader, start=1):

        # 若不足 30 欄，補空字串
        if len(row) < 30:
            row = row + [""] * (30 - len(row))

        # 找出此列中 autosomal、X、Y 或 MT 出現的位置
        chr_positions = [
            index
            for index, value in enumerate(row)
            if value in valid_chr_types
        ]

        # 只接受剛好找到一個 chromosome type 的列
        if len(chr_positions) == 1:

            # 取得 specialSNP_chr 實際出現位置
            actual_chr_index = chr_positions[0]

            # 計算向左位移幾欄
            # 例如應在 11，實際在 10，代表向左移 1 欄
            left_shift = expected_special_chr_index - actual_chr_index

            # 檢查 chromosome 後一欄是否為合理 gender_metrics
            gender_index = actual_chr_index + 1

            if gender_index < len(row):
                gender_value = row[gender_index]
            else:
                gender_value = ""

            # 判斷後一欄是否合理
            gender_ok = gender_value in valid_gender_types

            # 以「位移量 + gender是否合理」作為分類
            pattern_key = (left_shift, gender_ok)

            # 累計該模式列數
            shift_counts[pattern_key] = (
                shift_counts.get(pattern_key, 0) + 1
            )

        else:
            # 沒找到或找到多個 chromosome type 的列，先記錄
            unrecognized_rows.append(
                (
                    row_number,
                    len(chr_positions),
                    row[:15]
                )
            )


# 依出現次數由大到小排列
sorted_shift_counts = sorted(
    shift_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

# 顯示結果
print("依 specialSNP_chr 位置判斷的位移模式：")
print()

for (left_shift, gender_ok), count in sorted_shift_counts:

    print(
        "向左位移欄數：",
        left_shift,
        "| gender位置合理：",
        gender_ok,
        "| 列數：",
        count
    )

# 顯示無法辨識列數
print()
print("無法辨識的列數：", len(unrecognized_rows))

依 specialSNP_chr 位置判斷的位移模式：

向左位移欄數： 0 | gender位置合理： True | 列數： 297913
向左位移欄數： 1 | gender位置合理： True | 列數： 223971
向左位移欄數： 3 | gender位置合理： True | 列數： 155685
向左位移欄數： 2 | gender位置合理： True | 列數： 3296

無法辨識的列數： 0


In [10]:
# 設定 specialSNP_chr 可能出現的合法值
valid_chr_types = {"autosomal", "X", "Y", "MT"}

# 設定 specialSNP_chr 在正常後段資料中的位置
# 後段第 12 欄，Python index 從 0 開始，所以是 11
expected_special_chr_index = 11

# 設定新的輸出檔案路徑
classified_back_path = (
    output_folder
    / "genotype_back_48_77_with_shift.txt"
)

# 開啟原本的後段檔案與新的分類檔案
with back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as back_file, \
classified_back_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file:

    # 建立 Tab 分隔的讀取器
    reader = csv.reader(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立 Tab 分隔的寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 讀取原本後段表頭
    header = next(reader)

    # 在表頭最前面新增 left_shift
    new_header = ["left_shift"] + header

    # 寫入新的表頭
    writer.writerow(new_header)

    # 逐列讀取 SNP 後段資料
    for row_number, row in enumerate(reader, start=1):

        # 若不足 30 欄，補空字串
        if len(row) < 30:
            row = row + [""] * (30 - len(row))

        # 找出 autosomal、X、Y、MT 所在位置
        chr_positions = [
            index
            for index, value in enumerate(row)
            if value in valid_chr_types
        ]

        # 因為前一步已確認每列都只有一個合法位置
        actual_chr_index = chr_positions[0]

        # 計算向左位移欄數
        left_shift = (
            expected_special_chr_index
            - actual_chr_index
        )

        # 將 left_shift 放在每列最前面
        new_row = [left_shift] + row

        # 寫入新檔案
        writer.writerow(new_row)

# 顯示結果
print("分類檔案建立完成")
print("輸出位置：", classified_back_path)

分類檔案建立完成
輸出位置： /Users/sheena/Desktop/summerintern/split_data/genotype_back_48_77_with_shift.txt


In [11]:
# 開啟新的分類檔案
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace"
) as file:

    # 顯示前 5 列
    for i in range(5):
        line = file.readline()
        print(line.rstrip("\n"))

left_shift	FLD	HomFLD	HetSO	HomRO	nMinorAllele	Nclus	n_AA	n_AB	n_BB	n_NC	hemizygous	specialSNP_chr	gender_metrics	ConversionType	BestProbeset	BestandRecommended	HomHet	AA.meanX	AA.meanY	AB.meanX	AB.meanY	BB.meanX	BB.meanY	NC.meanX	NC.meanY	MMD	MinorAlleleFrequency	H.W.p-Value	H.W.chisquared.statistic	Call Modified
0	16.624	34.486	0.728	2.547	211	3	64	141	35	0	0	autosomal	all	PolyHighResolution	1	1	0	2.5472	9.6674	0.0158	10.3304	-2.7043	9.5337	NA	NA	41.545	0.44	0.00287	8.8852	FALSE
0	14.758	30.536	0.776	2.562	168	3	102	108	30	0	0	autosomal	all	PolyHighResolution	1	1	0	2.6958	9.4151	0.1546	10.1287	-2.5623	9.2856	NA	NA	50.182	0.35	0.865	0.029	FALSE
0	13.761	28.393	0.539	2.577	234	3	59	116	65	0	0	autosomal	all	PolyHighResolution	1	1	0	2.577	10.6359	-0.3268	11.3084	-3.0579	10.8953	NA	NA	50.462	0.488	0.612	0.2571	FALSE
0	14.287	29.199	0.571	1.952	236	3	60	124	56	0	0	autosomal	all	PolyHighResolution	1	1	0	1.9521	9.6865	-0.3371	10.3143	-2.5304	9.7971	NA	NA	61.263	0.492	0.602	0.2713	FALSE


In [12]:
# 建立字典：每種 left_shift 最多保存 3 列範例
examples_by_shift = {
    0: [],
    1: [],
    2: [],
    3: []
}

# 開啟已加入 left_shift 的後段檔案
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取表頭
    header = next(reader)

    # 逐列讀取資料
    for row_number, row in enumerate(reader, start=1):

        # 第 1 欄是 left_shift
        left_shift = int(row[0])

        # 每一類最多保留 3 列
        if len(examples_by_shift[left_shift]) < 3:
            examples_by_shift[left_shift].append(
                (row_number, row)
            )

        # 如果 4 種類型都已經各收集 3 列，就停止讀取
        if all(
            len(examples_by_shift[shift]) == 3
            for shift in examples_by_shift
        ):
            break


# 逐一顯示 4 種 left_shift 的範例
for shift in [0, 1, 2, 3]:

    print("=" * 70)
    print("left_shift =", shift)
    print("=" * 70)

    # 顯示這一類的 3 列範例
    for row_number, row in examples_by_shift[shift]:

        print("資料列號：", row_number)

        # 從第 2 欄開始，因為第 1 欄是新增的 left_shift
        for column_name, value in zip(
            header[1:],
            row[1:]
        ):
            print(
                f"{column_name}: {repr(value)}"
            )

        print("-" * 70)

left_shift = 0
資料列號： 1
FLD: '16.624'
HomFLD: '34.486'
HetSO: '0.728'
HomRO: '2.547'
nMinorAllele: '211'
Nclus: '3'
n_AA: '64'
n_AB: '141'
n_BB: '35'
n_NC: '0'
hemizygous: '0'
specialSNP_chr: 'autosomal'
gender_metrics: 'all'
ConversionType: 'PolyHighResolution'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '0'
AA.meanX: '2.5472'
AA.meanY: '9.6674'
AB.meanX: '0.0158'
AB.meanY: '10.3304'
BB.meanX: '-2.7043'
BB.meanY: '9.5337'
NC.meanX: 'NA'
NC.meanY: 'NA'
MMD: '41.545'
MinorAlleleFrequency: '0.44'
H.W.p-Value: '0.00287'
H.W.chisquared.statistic: '8.8852'
Call Modified: 'FALSE'
----------------------------------------------------------------------
資料列號： 2
FLD: '14.758'
HomFLD: '30.536'
HetSO: '0.776'
HomRO: '2.562'
nMinorAllele: '168'
Nclus: '3'
n_AA: '102'
n_AB: '108'
n_BB: '30'
n_NC: '0'
hemizygous: '0'
specialSNP_chr: 'autosomal'
gender_metrics: 'all'
ConversionType: 'PolyHighResolution'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '0'
AA.meanX: '2.6958'
AA.meanY: '9.4151'
AB.

In [14]:
# 指定這次要查看的位移類型
target_shift = 1

# 指定最多顯示幾筆範例
max_examples = 3

# 建立計數器，記錄目前已顯示幾筆
shown_count = 0

# 開啟已加入 left_shift 的後段檔案
with classified_back_path.open(
    "r",                    # 以讀取模式開啟
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 無法解碼的字元以替代符號顯示
    newline=""
) as file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取第一列欄位名稱
    header = next(reader)

    # 逐列讀取資料
    for row_number, row in enumerate(reader, start=1):

        # 第一欄是新增的 left_shift
        left_shift = int(row[0])

        # 只處理 left_shift = 1 的資料列
        if left_shift != target_shift:
            continue

        # 顯示資料列號
        print("=" * 70)
        print("資料列號：", row_number)
        print("left_shift：", left_shift)
        print("=" * 70)

        # 顯示原始後段 30 欄
        # header[1:] 跳過新增的 left_shift 欄
        # row[1:] 跳過該列的 left_shift 值
        for column_name, value in zip(
            header[1:],
            row[1:]
        ):
            print(f"{column_name}: {repr(value)}")

        # 每顯示一筆，計數器加 1
        shown_count += 1

        # 顯示滿 3 筆後停止
        if shown_count >= max_examples:
            break

# 顯示總共輸出幾筆
print()
print("已顯示範例數：", shown_count)

資料列號： 89
left_shift： 1
FLD: '4.462'
HomFLD: '?�數??-0.0327'
HetSO: '1.69'
HomRO: '13'
nMinorAllele: '2'
Nclus: '0'
n_AA: '13'
n_AB: '226'
n_BB: '1'
n_NC: '0'
hemizygous: 'autosomal'
specialSNP_chr: 'all'
gender_metrics: 'NoMinorHom'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '2.0892'
AA.meanX: '11.255'
AA.meanY: '-0.0631'
AB.meanX: '11.1512'
AB.meanY: '-1.6896'
BB.meanX: '11.1302'
BB.meanY: 'NA'
NC.meanX: 'NA'
NC.meanY: '?�數??0.0272'
MMD: '1'
MinorAlleleFrequency: 'NA'
H.W.p-Value: 'FALSE'
H.W.chisquared.statistic: ''
Call Modified: ''
資料列號： 91
left_shift： 1
FLD: '9.579'
HomFLD: '?�數??0.403'
HetSO: '1.612'
HomRO: '3'
nMinorAllele: '2'
Nclus: '0'
n_AA: '3'
n_AB: '237'
n_BB: '0'
n_NC: '0'
hemizygous: 'autosomal'
specialSNP_chr: 'all'
gender_metrics: 'NoMinorHom'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: '1'
HomHet: '2.4153'
AA.meanX: '9.8751'
AA.meanY: '0.4429'
AB.meanX: '10.1739'
AB.meanY: '-1.6123'
BB.meanX: '9.6633'
BB.meanY: 'NA'
NC.meanX: 'N

In [15]:
#單獨看HomFLD
# 建立空字典，用來統計 left_shift = 1 時
# HomFLD 欄位中每種原始值出現的次數
homfld_counts_shift1 = {}

# 開啟已加入 left_shift 分類的後段檔案
with classified_back_path.open(
    "r",                    # 以讀取模式開啟
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 無法解碼的字元用替代符號顯示
    newline=""
) as file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取表頭
    header = next(reader)

    # 找出 HomFLD 在目前檔案中的欄位位置
    # 因為前面多了一欄 left_shift，所以直接用欄名找最安全
    homfld_index = header.index("HomFLD")

    # 找出 left_shift 欄位位置
    shift_index = header.index("left_shift")

    # 逐列讀取資料
    for row in reader:

        # 只保留 left_shift = 1 的資料列
        if row[shift_index] != "1":
            continue

        # 取出目前 HomFLD 欄位的原始內容
        homfld_value = row[homfld_index]

        # 統計每個值出現次數
        homfld_counts_shift1[homfld_value] = (
            homfld_counts_shift1.get(homfld_value, 0) + 1
        )

# 依出現次數由多到少排序
sorted_homfld = sorted(
    homfld_counts_shift1.items(),
    key=lambda item: item[1],
    reverse=True
)

# 顯示前 30 種最常見的 HomFLD 原始值
print("left_shift = 1 中最常見的 HomFLD 值：")
print()

for value, count in sorted_homfld[:30]:
    print("出現次數：", count, "| HomFLD：", repr(value))

left_shift = 1 中最常見的 HomFLD 值：

出現次數： 613 | HomFLD： '?�數??0.329'
出現次數： 601 | HomFLD： '?�數??0.354'
出現次數： 584 | HomFLD： '?�數??0.331'
出現次數： 583 | HomFLD： '?�數??0.301'
出現次數： 583 | HomFLD： '?�數??0.287'
出現次數： 581 | HomFLD： '?�數??0.345'
出現次數： 581 | HomFLD： '?�數??0.338'
出現次數： 579 | HomFLD： '?�數??0.306'
出現次數： 579 | HomFLD： '?�數??0.361'
出現次數： 576 | HomFLD： '?�數??0.255'
出現次數： 575 | HomFLD： '?�數??0.377'
出現次數： 572 | HomFLD： '?�數??0.266'
出現次數： 567 | HomFLD： '?�數??0.262'
出現次數： 566 | HomFLD： '?�數??0.375'
出現次數： 566 | HomFLD： '?�數??0.347'
出現次數： 566 | HomFLD： '?�數??0.322'
出現次數： 564 | HomFLD： '?�數??0.359'
出現次數： 563 | HomFLD： '?�數??0.356'
出現次數： 562 | HomFLD： '?�數??0.303'
出現次數： 562 | HomFLD： '?�數??0.278'
出現次數： 560 | HomFLD： '?�數??0.269'
出現次數： 559 | HomFLD： '?�數??0.312'
出現次數： 559 | HomFLD： '?�數??0.292'
出現次數： 558 | HomFLD： '?�數??0.264'
出現次數： 557 | HomFLD： '?�數??0.326'
出現次數： 556 | HomFLD： '?�數??0.324'
出現次數： 554 | HomFLD： '?�數??0.35'
出現次數： 552 | HomFLD： '?�數??0.296'
出現次數： 551 | HomFLD： '?�數??0.28'
出現次數： 547 | H

In [16]:
#步驟（針對位移一個的）：
#1.先把FLD欄位之後 從HomFLD開始都先位移一個 這樣先空出HomFLD
#2.Backward 5個切割FLD 
#3.放入FLD

#  載入正規表示式模組
# 用來從含亂碼的 FLD 字串中擷取數字
import re
# 設定 left_shift = 1 修正版的輸出檔案
shift1_fixed_path = (
    output_folder
    / "genotype_back_shift1_fixed.txt"
)


# 用來記錄修復結果
total_shift1 = 0              # left_shift = 1 的總列數
success_count = 0             # 成功辨認 FLD 與 HomFLD 的列數
failed_count = 0              # 無法辨認兩個數值的列數
last_column_not_empty = 0     # 原始最後一欄不為空的列數

# 儲存前幾筆修復失敗的範例
failed_examples = []

# 儲存原始最後一欄不為空的範例
last_column_examples = []


# 這個 pattern 可以辨認：
# 4.462
# -0.0327
# 0.329
# .521
# -3
number_pattern = re.compile(
    r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
)


# 開啟已加入 left_shift 的後段檔案
with classified_back_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼的字元以替代符號保留
    newline=""
) as input_file, \
shift1_fixed_path.open(
    "w",                    # 建立新的修正版檔案
    encoding="utf-8",
    newline=""
) as output_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立 Tab 分隔寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 讀取欄位名稱
    header = next(reader)

    # 找出 left_shift 欄的位置
    shift_index = header.index("left_shift")

    # 後段原始 30 欄的欄位名稱
    back_columns = header[1:]

    # 輸出欄位：
    # left_shift + 修正後30欄 + repair_status
    output_header = (
        ["left_shift"]
        + back_columns
        + ["repair_status"]
    )

    # 寫入表頭
    writer.writerow(output_header)


    # 逐列讀取資料
    for row_number, row in enumerate(reader, start=1):

        # 只處理 left_shift = 1
        if row[shift_index] != "1":
            continue

        total_shift1 += 1

        # 取出原始後段30欄
        # 排除最前面的 left_shift
        original = row[1:]

        # 若不足30欄，補空字串
        if len(original) < 30:
            original = original + [""] * (30 - len(original))

        # 若超過30欄，只保留前30欄
        original = original[:30]

        # 保存原始 FLD 字串
        original_fld = original[0]

        # 從 FLD 字串中找出所有數值
        numbers = number_pattern.findall(original_fld)


        # 右移會捨棄原始最後一欄
        # 因此先確認原始 Call Modified 是否為空
        if original[-1].strip() != "":
            last_column_not_empty += 1

            if len(last_column_examples) < 10:
                last_column_examples.append(
                    (
                        row_number,
                        original[-1]
                    )
                )


        # 必須至少辨認到兩個數值：
        # 第一個視為真正 FLD
        # 最後一個視為真正 HomFLD
        if len(numbers) >= 2:

            corrected_fld = numbers[0]
            corrected_homfld = numbers[-1]

            # 先複製原始資料
            corrected = original.copy()

            # 將原本 HomFLD 到倒數第二欄
            # 全部向右移一格
            #
            # corrected[2:] 代表新的 HetSO 到 Call Modified
            # original[1:-1] 代表原始 HomFLD 到倒數第二欄
            corrected[2:] = original[1:-1]

            # 將清理後的 FLD 放回第一欄
            corrected[0] = corrected_fld

            # 將從 FLD 尾端擷取的數值放入 HomFLD
            corrected[1] = corrected_homfld

            repair_status = "success"

            success_count += 1

        else:
            # 如果 FLD 中找不到至少兩個數值
            # 暫時保留原始資料，不進行猜測性修正
            corrected = original.copy()

            repair_status = "failed_less_than_2_numbers"

            failed_count += 1

            # 最多記錄前10筆失敗範例
            if len(failed_examples) < 10:
                failed_examples.append(
                    (
                        row_number,
                        original_fld,
                        numbers
                    )
                )


        # 寫入：
        # left_shift + 修正後30欄 + 修復狀態
        writer.writerow(
            ["1"]
            + corrected
            + [repair_status]
        )


# 顯示處理結果
print("left_shift = 1 處理完成")
print("輸出檔案：", shift1_fixed_path)
print()
print("left_shift = 1 總列數：", total_shift1)
print("成功修復列數：", success_count)
print("無法辨認列數：", failed_count)
print("原始最後一欄不為空：", last_column_not_empty)

print()
print("前幾筆無法辨認的範例：")

for example in failed_examples:
    print(example)

print()
print("原始最後一欄不為空的範例：")

for example in last_column_examples:
    print(example)

left_shift = 1 處理完成
輸出檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_fixed.txt

left_shift = 1 總列數： 223971
成功修復列數： 0
無法辨認列數： 223971
原始最後一欄不為空： 0

前幾筆無法辨認的範例：
(89, '4.462', ['4.462'])
(91, '9.579', ['9.579'])
(93, '7.143', ['7.143'])
(98, '6.14', ['6.14'])
(101, '8.359', ['8.359'])
(103, '8.798', ['8.798'])
(105, '11.547', ['11.547'])
(107, '9.647', ['9.647'])
(108, '11.59', ['11.59'])
(109, '9.041', ['9.041'])

原始最後一欄不為空的範例：


In [17]:
# 載入正規表示式工具
import re

# 定義「完整純數字」格式
# 可接受 4.462、-0.0327、12、.35
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)$"
)

# 建立分類計數器
fld_numeric_homfld_numeric = 0
fld_numeric_homfld_garbled = 0
fld_garbled_homfld_numeric = 0
fld_garbled_homfld_garbled = 0

# 保存少量 FLD 含亂碼範例
fld_garbled_examples = []

# 開啟已分類的後段資料
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 讀取表頭
    header = next(reader)

    # 用欄名取得位置
    shift_index = header.index("left_shift")
    fld_index = header.index("FLD")
    homfld_index = header.index("HomFLD")

    # 逐列檢查
    for row_number, row in enumerate(reader, start=1):

        # 只檢查 left_shift = 1
        if row[shift_index] != "1":
            continue

        fld_value = row[fld_index].strip()
        homfld_value = row[homfld_index].strip()

        # 判斷是否為完整純數字
        fld_is_numeric = bool(
            full_number_pattern.fullmatch(fld_value)
        )

        homfld_is_numeric = bool(
            full_number_pattern.fullmatch(homfld_value)
        )

        # 分成四種組合
        if fld_is_numeric and homfld_is_numeric:
            fld_numeric_homfld_numeric += 1

        elif fld_is_numeric and not homfld_is_numeric:
            fld_numeric_homfld_garbled += 1

        elif not fld_is_numeric and homfld_is_numeric:
            fld_garbled_homfld_numeric += 1

        else:
            fld_garbled_homfld_garbled += 1

        # 保存前 20 筆 FLD 不是純數字的範例
        if not fld_is_numeric and len(fld_garbled_examples) < 20:
            fld_garbled_examples.append(
                (
                    row_number,
                    fld_value,
                    homfld_value
                )
            )

# 顯示統計
print("FLD純數字、HomFLD純數字：",
      fld_numeric_homfld_numeric)

print("FLD純數字、HomFLD含亂碼：",
      fld_numeric_homfld_garbled)

print("FLD含亂碼、HomFLD純數字：",
      fld_garbled_homfld_numeric)

print("FLD含亂碼、HomFLD含亂碼：",
      fld_garbled_homfld_garbled)

print()
print("FLD含亂碼的前20筆範例：")

for example in fld_garbled_examples:
    print(example)



FLD純數字、HomFLD純數字： 0
FLD純數字、HomFLD含亂碼： 223971
FLD含亂碼、HomFLD純數字： 0
FLD含亂碼、HomFLD含亂碼： 0

FLD含亂碼的前20筆範例：


發現位移一個是從HomFLD之後！！！

In [18]:
# 載入正規表示式模組
# 用來從 HomFLD 亂碼字串尾端擷取完整數值
import re


# 設定修正後的輸出檔案路徑
shift1_fixed_path = (
    output_folder
    / "genotype_back_shift1_fixed.txt"
)


# 設定修復檢查紀錄檔
shift1_audit_path = (
    output_folder
    / "genotype_back_shift1_audit.txt"
)


# 從字串尾端擷取完整數值
# 可辨認：
# 0.329
# -0.0327
# 12
# .35
# 1.2e-3
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立處理結果計數器
total_shift1 = 0                 # left_shift = 1 的總列數
success_count = 0                # 成功擷取 HomFLD 的列數
failed_count = 0                 # 無法擷取 HomFLD 的列數
last_column_not_empty = 0        # 原始最後一欄不是空白的列數
batch_total_240_count = 0        # 修正後 n_AA+n_AB+n_BB+n_NC = 240 的列數
batch_total_invalid_count = 0    # 修正後 genotype count 不等於 240 的列數


# 保存少量失敗範例
failed_examples = []


# 同時開啟：
# 1. 已加入 left_shift 的後段檔案
# 2. 修正後輸出檔案
# 3. 修復紀錄檔
with classified_back_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用 UTF-8
    errors="replace",            # 無法解碼字元以 � 保留
    newline=""
) as input_file, \
shift1_fixed_path.open(
    "w",                         # 寫入新的修正版
    encoding="utf-8",
    newline=""
) as output_file, \
shift1_audit_path.open(
    "w",                         # 寫入修復前後對照紀錄
    encoding="utf-8",
    newline=""
) as audit_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立修正版寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 建立修復紀錄寫入器
    audit_writer = csv.writer(
        audit_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 讀取原始表頭
    header = next(reader)

    # 取得後段原始 30 個欄位名稱
    # 跳過最前面的 left_shift
    back_columns = header[1:]


    # 修正版新增：
    # source_row_number：原始 SNP 資料列號
    # repair_status：修復是否成功
    writer.writerow(
        ["source_row_number", "left_shift"]
        + back_columns
        + ["repair_status"]
    )


    # 修復紀錄檔只保存關鍵對照資訊
    audit_writer.writerow(
        [
            "source_row_number",
            "original_FLD",
            "original_HomFLD",
            "fixed_HomFLD",
            "repair_status"
        ]
    )


    # 逐列讀取後段資料
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 第一欄是 left_shift
        # 只處理 left_shift = 1
        if row[0] != "1":
            continue

        total_shift1 += 1


        # 取得原始後段 30 欄
        # row[0] 是 left_shift，因此從 row[1] 開始
        original = row[1:]


        # 如果不足 30 欄，用空字串補足
        if len(original) < 30:
            original = original + [""] * (30 - len(original))


        # 如果意外超過 30 欄，只保留前 30 欄
        original = original[:30]


        # 原始欄位位置：
        # original[0] = FLD
        # original[1] = HomFLD
        # original[2] = HetSO
        original_fld = original[0]
        original_homfld = original[1]


        # 從原始 HomFLD 字串尾端尋找完整數值
        match = ending_number_pattern.search(
            original_homfld
        )


        # 修正時最後一欄會被移除
        # 因此先確認原始最後一欄是否為空
        if original[-1].strip() != "":
            last_column_not_empty += 1


        # 成功找到 HomFLD 尾端數值
        if match:

            # 取得真正的 HomFLD 數值
            fixed_homfld = match.group(1)


            # 建立新的 30 欄空白資料
            corrected = [""] * 30


            # FLD 保持原始值不變
            corrected[0] = original_fld


            # HomFLD 改成從亂碼字串尾端擷取的數值
            corrected[1] = fixed_homfld


            # HetSO 暫時留空
            # 因為這是此類資料缺少的欄位
            corrected[2] = ""


            # 將原始 HetSO 到倒數第二欄
            # 全部向右移一格
            #
            # 原始：
            # original[2] 放在 HetSO 位置
            #
            # 修正：
            # original[2] 移到新的 HomRO 位置 corrected[3]
            corrected[3:] = original[2:-1]


            # 標記修復成功
            repair_status = "success"
            success_count += 1


            # 修正後欄位位置：
            # n_AA = index 6
            # n_AB = index 7
            # n_BB = index 8
            # n_NC = index 9
            try:
                batch_total = (
                    int(corrected[6])
                    + int(corrected[7])
                    + int(corrected[8])
                    + int(corrected[9])
                )

                # 檢查四種 genotype count 是否合計 240
                if batch_total == 240:
                    batch_total_240_count += 1
                else:
                    batch_total_invalid_count += 1

            except ValueError:
                # 若任一 genotype count 無法轉成整數
                batch_total_invalid_count += 1


        # 無法從 HomFLD 尾端找到數值
        else:

            # 不進行猜測性修正
            corrected = original.copy()

            fixed_homfld = ""

            repair_status = "failed_no_ending_number"
            failed_count += 1


            # 最多保存前 10 筆失敗範例
            if len(failed_examples) < 10:
                failed_examples.append(
                    (
                        source_row_number,
                        original_fld,
                        original_homfld
                    )
                )


        # 寫入修正後資料
        writer.writerow(
            [
                source_row_number,
                "1"
            ]
            + corrected
            + [repair_status]
        )


        # 寫入修復前後對照紀錄
        audit_writer.writerow(
            [
                source_row_number,
                original_fld,
                original_homfld,
                fixed_homfld,
                repair_status
            ]
        )


# 顯示處理結果
print("left_shift = 1 修復完成")
print()

print("修正版檔案：", shift1_fixed_path)
print("修復紀錄檔：", shift1_audit_path)
print()

print("left_shift = 1 總列數：", total_shift1)
print("成功修復列數：", success_count)
print("無法擷取 HomFLD 列數：", failed_count)
print("原始最後一欄不為空：", last_column_not_empty)
print()

print(
    "修正後 genotype count 合計為 240：",
    batch_total_240_count
)

print(
    "修正後 genotype count 無法通過檢查：",
    batch_total_invalid_count
)

print()
print("前幾筆失敗範例：")

for example in failed_examples:
    print(example)

left_shift = 1 修復完成

修正版檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_fixed.txt
修復紀錄檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_audit.txt

left_shift = 1 總列數： 223971
成功修復列數： 223971
無法擷取 HomFLD 列數： 0
原始最後一欄不為空： 0

修正後 genotype count 合計為 240： 220896
修正後 genotype count 無法通過檢查： 3075

前幾筆失敗範例：


In [19]:
# 載入 Counter，用來統計各種異常原因與資料類型
from collections import Counter

# 統計未通過 count QC 的原因
invalid_reason_counts = Counter()

# 統計未通過者的 chromosome、gender與ConversionType組合
invalid_group_counts = Counter()

# 統計四個 genotype count 實際合計值
invalid_total_counts = Counter()

# 保存前20筆未通過的完整範例
invalid_examples = []


# 開啟剛才建立的 left_shift = 1 修正版檔案
with shift1_fixed_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取每一列
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查修正後的 genotype count
    for row in reader:

        # 取出四個 count 欄位的原始字串
        count_values = [
            row["n_AA"],
            row["n_AB"],
            row["n_BB"],
            row["n_NC"]
        ]

        try:
            # 嘗試將四個欄位轉成整數
            count_numbers = [
                int(value)
                for value in count_values
            ]

            # 計算四個 genotype count 的合計
            batch_total = sum(count_numbers)

            # 合計為240代表通過，不需要記錄
            if batch_total == 240:
                continue

            # 可以轉成整數，但合計不是240
            reason = "total_not_240"

            # 統計實際合計值
            invalid_total_counts[batch_total] += 1

        except (ValueError, TypeError):

            # 至少一個 count 是空白、NA或不能轉成整數
            reason = "non_integer_or_missing"
            batch_total = None

        # 統計異常原因
        invalid_reason_counts[reason] += 1

        # 依特殊染色體、性別指標與ConversionType分類
        group_key = (
            row["specialSNP_chr"],
            row["gender_metrics"],
            row["ConversionType"]
        )

        invalid_group_counts[group_key] += 1

        # 保存前20筆範例
        if len(invalid_examples) < 20:
            invalid_examples.append(
                {
                    "source_row_number": row["source_row_number"],
                    "n_AA": row["n_AA"],
                    "n_AB": row["n_AB"],
                    "n_BB": row["n_BB"],
                    "n_NC": row["n_NC"],
                    "batch_total": batch_total,
                    "specialSNP_chr": row["specialSNP_chr"],
                    "gender_metrics": row["gender_metrics"],
                    "ConversionType": row["ConversionType"]
                }
            )


# 顯示未通過原因
print("未通過原因：")

for reason, count in invalid_reason_counts.most_common():
    print(reason, "：", count)


# 顯示最常見的實際合計值
print()
print("最常見的 genotype count 合計值：")

for total, count in invalid_total_counts.most_common(20):
    print("合計 =", total, "| 列數 =", count)


# 顯示最常見的資料類型組合
print()
print("最常見的 chromosome／gender／ConversionType 組合：")

for group, count in invalid_group_counts.most_common(20):
    print(
        "specialSNP_chr =", group[0],
        "| gender_metrics =", group[1],
        "| ConversionType =", group[2],
        "| 列數 =", count
    )


# 顯示前20筆範例
print()
print("前20筆未通過範例：")

for example in invalid_examples:
    print(example)

未通過原因：
total_not_240 ： 3075

最常見的 genotype count 合計值：
合計 = 163 | 列數 = 3075

最常見的 chromosome／gender／ConversionType 組合：
specialSNP_chr = X | gender_metrics = female | ConversionType = PolyHighResolution | 列數 = 2696
specialSNP_chr = X | gender_metrics = female | ConversionType = NoMinorHom | 列數 = 379

前20筆未通過範例：
{'source_row_number': '216', 'n_AA': '0', 'n_AB': '11', 'n_BB': '152', 'n_NC': '0', 'batch_total': 163, 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution'}
{'source_row_number': '242', 'n_AA': '0', 'n_AB': '8', 'n_BB': '155', 'n_NC': '0', 'batch_total': 163, 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution'}
{'source_row_number': '255', 'n_AA': '153', 'n_AB': '10', 'n_BB': '0', 'n_NC': '0', 'batch_total': 163, 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution'}
{'source_row_number': '804', 'n_AA': '152', 'n_AB': '10', 'n_BB': '0', 'n_NC': '1', 'batch_total': 163, '

##位移2個##

In [20]:
# 指定這次要查看的位移類型
target_shift = 2

# 設定最多顯示幾筆範例
max_examples = 3

# 建立計數器，記錄目前已顯示幾筆
shown_count = 0


# 開啟已加入 left_shift 分類的後段檔案
with classified_back_path.open(
    "r",                    # 以讀取模式開啟檔案
    encoding="utf-8",       # 使用 UTF-8 編碼
    errors="replace",       # 無法解碼的字元以替代符號顯示
    newline=""
) as file:

    # 建立 Tab 分隔的讀取器
    reader = csv.reader(
        file,
        delimiter="\t",     # 指定 Tab 為欄位分隔符號
        quoting=csv.QUOTE_NONE
    )

    # 讀取第一列欄位名稱
    header = next(reader)


    # 逐列讀取資料
    for row_number, row in enumerate(
        reader,
        start=1
    ):

        # 第一欄是新增的 left_shift
        left_shift = int(row[0])

        # 只查看 left_shift = 2 的資料列
        if left_shift != target_shift:
            continue


        # 顯示這筆資料的基本資訊
        print("=" * 70)
        print("資料列號：", row_number)
        print("left_shift：", left_shift)
        print("=" * 70)


        # header[1:] 跳過 left_shift 欄位名稱
        # row[1:] 跳過該列的 left_shift 數值
        for column_name, value in zip(
            header[1:],
            row[1:]
        ):

            # repr() 可以清楚顯示空字串與亂碼
            print(
                f"{column_name}: {repr(value)}"
            )


        # 每顯示一筆，計數器加 1
        shown_count += 1


        # 顯示滿指定筆數後停止
        if shown_count >= max_examples:
            break


# 顯示這次實際輸出的範例數
print()
print("已顯示範例數：", shown_count)

資料列號： 88
left_shift： 2
FLD: '?�數???�數??0.548'
HomFLD: '3.205'
HetSO: '4'
HomRO: '2'
nMinorAllele: '159'
Nclus: '4'
n_AA: '0'
n_AB: '0'
n_BB: '0'
n_NC: 'X'
hemizygous: 'female'
specialSNP_chr: 'PolyHighResolution'
gender_metrics: '1'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: '3.2047'
HomHet: '10.7486'
AA.meanX: '0.3084'
AA.meanY: '11.334'
AB.meanX: 'NA'
AB.meanY: 'NA'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
資料列號： 141
left_shift： 2
FLD: '?�數???�數??0.43'
HomFLD: '2.379'
HetSO: '3'
HomRO: '2'
nMinorAllele: '0'
Nclus: '3'
n_AA: '160'
n_AB: '0'
n_BB: '0'
n_NC: 'X'
hemizygous: 'female'
specialSNP_chr: 'PolyHighResolution'
gender_metrics: '1'
ConversionType: '1'
BestProbeset: '1'
BestandRecommended: 'NA'
HomHet: 'NA'
AA.meanX: '0.0985'
AA.meanY: '10.1475'
AB.meanX: '-2.3795'
AB.meanY: '9.5921'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0092'
NC

利用逐一欄位檢查 可以知道這些位移兩個 共3296個 分別在每個欄位中的情形也許就可以正確改正

In [21]:
# 載入需要的工具
import csv
import re

# Counter 用來統計不同值或不同模式的出現次數
from collections import Counter

# median 用來計算數值欄位的中位數
from statistics import median


# 設定 left_shift = 2 的逐欄分析報告路徑
shift2_column_report_path = (
    output_folder
    / "shift2_column_summary.txt"
)

# 設定 left_shift = 2 的子模式報告路徑
shift2_pattern_report_path = (
    output_folder
    / "shift2_subpattern_summary.txt"
)


# 定義完整數字格式
# 可以辨認：
# 4
# 4.462
# -0.0327
# .35
# 1.2e-3
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義已知的類別值
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}

valid_gender_values = {
    "all",
    "male",
    "female"
}

valid_boolean_values = {
    "TRUE",
    "FALSE"
}


# 建立函數：判斷每一個欄位值屬於哪一種型態
def classify_value(value):

    # 去除字串前後空白
    value = value.strip()

    # 空字串
    if value == "":
        return "empty"

    # NA
    if value.upper() == "NA":
        return "NA"

    # 完整純數字
    if full_number_pattern.fullmatch(value):
        return "numeric"

    # 染色體類型
    if value in valid_chr_values:
        return "chromosome"

    # 性別分析群組
    if value in valid_gender_values:
        return "gender"

    # TRUE 或 FALSE
    if value in valid_boolean_values:
        return "boolean"

    # 字串中含有常見解碼異常符號
    if "�" in value or "?" in value:
        return "garbled"

    # 其他文字
    return "text"


# 先開啟檔案取得表頭
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 取得全部欄位名稱
    all_columns = reader.fieldnames


# 排除新增的 left_shift
# 剩下的是原始後段30欄
back_columns = [
    column
    for column in all_columns
    if column != "left_shift"
]


# 每一欄的值統計
column_value_counts = {
    column: Counter()
    for column in back_columns
}

# 每一欄的資料型態統計
column_type_counts = {
    column: Counter()
    for column in back_columns
}

# 每一欄中可轉成數字的值
column_numeric_values = {
    column: []
    for column in back_columns
}


# 用來統計不同的欄位型態模式
pattern_counts = Counter()

# 保存每種模式的前3筆資料列範例
pattern_examples = {}

# left_shift = 2 的總列數
total_shift2 = 0


# 這些欄位先用來建立子模式
# 主要涵蓋 FLD 到 ConversionType
pattern_columns = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType"
]


# 重新開啟分類後的後段檔案
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只保留 left_shift = 2
        if row["left_shift"] != "2":
            continue

        total_shift2 += 1


        # 逐一檢查後段30欄
        for column in back_columns:

            # 取得此列、此欄的原始值
            value = row[column]

            # 統計該原始值出現次數
            column_value_counts[column][value] += 1

            # 判斷欄位值型態
            value_type = classify_value(value)

            # 統計該型態出現次數
            column_type_counts[column][value_type] += 1

            # 若是純數字，轉成 float 保存
            if value_type == "numeric":
                column_numeric_values[column].append(
                    float(value)
                )


        # 建立這一列的型態模式
        #
        # 例如：
        # garbled、numeric、numeric、numeric...
        pattern_signature = tuple(
            classify_value(row[column])
            for column in pattern_columns
        )

        # 統計此模式出現次數
        pattern_counts[pattern_signature] += 1


        # 保存每種模式前3筆範例
        if pattern_signature not in pattern_examples:
            pattern_examples[pattern_signature] = []

        if len(pattern_examples[pattern_signature]) < 3:
            pattern_examples[pattern_signature].append(
                {
                    "source_row_number": source_row_number,
                    "values": {
                        column: row[column]
                        for column in pattern_columns
                    }
                }
            )


# 建立逐欄摘要報告
with shift2_column_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    # 寫入總列數
    report.write(
        f"left_shift = 2 total rows: "
        f"{total_shift2}\n\n"
    )


    # 逐欄輸出結果
    for column in back_columns:

        report.write("=" * 70 + "\n")
        report.write(f"Column: {column}\n")
        report.write("=" * 70 + "\n")


        # 取得該欄各種型態統計
        type_counts = column_type_counts[column]

        report.write(
            f"numeric: {type_counts['numeric']}\n"
        )

        report.write(
            f"garbled: {type_counts['garbled']}\n"
        )

        report.write(
            f"empty: {type_counts['empty']}\n"
        )

        report.write(
            f"NA: {type_counts['NA']}\n"
        )

        report.write(
            f"chromosome: {type_counts['chromosome']}\n"
        )

        report.write(
            f"gender: {type_counts['gender']}\n"
        )

        report.write(
            f"boolean: {type_counts['boolean']}\n"
        )

        report.write(
            f"other text: {type_counts['text']}\n"
        )


        # 若該欄有數字，輸出數值摘要
        numeric_values = column_numeric_values[column]

        if numeric_values:

            report.write(
                f"numeric minimum: "
                f"{min(numeric_values)}\n"
            )

            report.write(
                f"numeric median: "
                f"{median(numeric_values)}\n"
            )

            report.write(
                f"numeric maximum: "
                f"{max(numeric_values)}\n"
            )


        # 找出前10種最常出現的原始值
        report.write("\nTop 10 original values:\n")

        for value, count in (
            column_value_counts[column].most_common(10)
        ):

            report.write(
                f"{repr(value)}\t{count}\n"
            )

        report.write("\n")


# 建立子模式報告
with shift2_pattern_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        f"left_shift = 2 total rows: "
        f"{total_shift2}\n"
    )

    report.write(
        f"Number of value-type patterns: "
        f"{len(pattern_counts)}\n\n"
    )


    # 依出現次數由多到少顯示前30種模式
    for pattern_rank, (
        pattern_signature,
        pattern_count
    ) in enumerate(
        pattern_counts.most_common(30),
        start=1
    ):

        report.write("=" * 80 + "\n")
        report.write(
            f"Pattern rank: {pattern_rank}\n"
        )

        report.write(
            f"Count: {pattern_count}\n"
        )

        report.write("-" * 80 + "\n")


        # 顯示每一欄對應的值型態
        for column, value_type in zip(
            pattern_columns,
            pattern_signature
        ):

            report.write(
                f"{column}: {value_type}\n"
            )


        # 顯示此模式的前3筆實際範例
        report.write("\nExamples:\n")

        for example in pattern_examples[
            pattern_signature
        ]:

            report.write(
                f"source_row_number: "
                f"{example['source_row_number']}\n"
            )

            for column in pattern_columns:

                value = example["values"][column]

                report.write(
                    f"  {column}: {repr(value)}\n"
                )

            report.write("\n")


# 顯示完成訊息
print("left_shift = 2 逐欄分析完成")
print()
print("left_shift = 2 總列數：", total_shift2)
print("發現的子模式數：", len(pattern_counts))
print()
print("逐欄摘要報告：", shift2_column_report_path)
print("子模式摘要報告：", shift2_pattern_report_path)

left_shift = 2 逐欄分析完成

left_shift = 2 總列數： 3296
發現的子模式數： 2

逐欄摘要報告： /Users/sheena/Desktop/summerintern/split_data/shift2_column_summary.txt
子模式摘要報告： /Users/sheena/Desktop/summerintern/split_data/shift2_subpattern_summary.txt


位移兩個：要分段除理
1.先處理nminor allele, nclus是缺失的 以及位移 
2.後面30欄（NC.meanX 到 Call Modified

In [22]:
# 載入處理 Tab 分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從含亂碼的字串尾端擷取數值
import re

# 設定第 1 階段修正後的輸出檔案
# 此檔只清理 FLD 與 HomFLD，不移動其他欄位
shift2_stage1_path = (
    output_folder
    / "genotype_back_shift2_stage1_clean_FLD_HomFLD.txt"
)


# 設定修正前後對照紀錄檔
shift2_stage1_audit_path = (
    output_folder
    / "genotype_back_shift2_stage1_audit.txt"
)


# 定義「完整純數字」格式
# 可辨認：
# 0.548
# -0.0327
# 12
# .35
# 1.2e-3
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義「字串尾端數字」格式
# 例如：
# '?�數???�數??0.548' → 擷取 0.548
# '?�數??-0.0327'     → 擷取 -0.0327
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_shift2 = 0

# FLD 成功擷取的列數
fld_success_count = 0

# FLD 無法擷取的列數
fld_failed_count = 0

# HomFLD 原本就是純數字的列數
homfld_already_numeric_count = 0

# HomFLD 從亂碼尾端成功擷取的列數
homfld_extracted_count = 0

# HomFLD 無法擷取的列數
homfld_failed_count = 0


# 保存少量失敗範例
failed_examples = []


# 開啟：
# 1. 已加入 left_shift 的後段資料
# 2. 第 1 階段修正版
# 3. 修正前後對照紀錄
with classified_back_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用 UTF-8
    errors="replace",            # 無法解碼字元以 � 保留
    newline=""
) as input_file, \
shift2_stage1_path.open(
    "w",                         # 建立新的修正版
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_stage1_audit_path.open(
    "w",                         # 建立修正紀錄檔
    encoding="utf-8",
    newline=""
) as audit_file:

    # 建立 Tab 分隔讀取器
    reader = csv.reader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 建立修正版寫入器
    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 建立對照紀錄寫入器
    audit_writer = csv.writer(
        audit_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 讀取原始表頭
    header = next(reader)


    # 修正版增加 source_row_number 與 repair_status
    writer.writerow(
        ["source_row_number"]
        + header
        + ["repair_status"]
    )


    # 對照檔只保留這次修復相關的欄位
    audit_writer.writerow(
        [
            "source_row_number",
            "original_FLD",
            "fixed_FLD",
            "original_HomFLD",
            "fixed_HomFLD",
            "repair_status"
        ]
    )


    # 逐列讀取資料
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 第一欄是 left_shift
        # 只處理 left_shift = 2
        if row[0] != "2":
            continue

        total_shift2 += 1


        # 複製整列資料
        # 避免直接修改原始 row
        corrected_row = row.copy()


        # 找出 FLD 與 HomFLD 在表頭中的位置
        fld_index = header.index("FLD")
        homfld_index = header.index("HomFLD")


        # 取得原始值
        original_fld = row[fld_index]
        original_homfld = row[homfld_index]


        # -------------------------
        # 清理 FLD
        # -------------------------

        # 從 FLD 字串尾端擷取數字
        fld_match = ending_number_pattern.search(
            original_fld.strip()
        )

        if fld_match:

            # 取得擷取出的 FLD 數值
            fixed_fld = fld_match.group(1)

            # 放回原本 FLD 欄位
            corrected_row[fld_index] = fixed_fld

            fld_success_count += 1

        else:

            # 無法擷取時保留原始內容
            fixed_fld = original_fld

            fld_failed_count += 1


        # -------------------------
        # 清理 HomFLD
        # -------------------------

        # 如果 HomFLD 本身已是完整純數字
        if full_number_pattern.fullmatch(
            original_homfld.strip()
        ):

            # 保留原始純數字
            fixed_homfld = original_homfld.strip()

            homfld_already_numeric_count += 1

        else:

            # 若 HomFLD 含亂碼，從尾端擷取數字
            homfld_match = ending_number_pattern.search(
                original_homfld.strip()
            )

            if homfld_match:

                # 取得尾端數字
                fixed_homfld = homfld_match.group(1)

                homfld_extracted_count += 1

            else:

                # 無法擷取時保留原始內容
                fixed_homfld = original_homfld

                homfld_failed_count += 1


        # 將清理後的 HomFLD 放回原欄位
        corrected_row[homfld_index] = fixed_homfld


        # 判斷本列是否成功
        if (
            fld_match is not None
            and (
                full_number_pattern.fullmatch(
                    fixed_homfld.strip()
                )
                is not None
            )
        ):
            repair_status = "success"

        else:
            repair_status = "failed"


        # 保存少量失敗範例
        if (
            repair_status == "failed"
            and len(failed_examples) < 20
        ):
            failed_examples.append(
                {
                    "source_row_number": source_row_number,
                    "original_FLD": original_fld,
                    "fixed_FLD": fixed_fld,
                    "original_HomFLD": original_homfld,
                    "fixed_HomFLD": fixed_homfld
                }
            )


        # 寫入第 1 階段修正版
        writer.writerow(
            [source_row_number]
            + corrected_row
            + [repair_status]
        )


        # 寫入修正前後對照
        audit_writer.writerow(
            [
                source_row_number,
                original_fld,
                fixed_fld,
                original_homfld,
                fixed_homfld,
                repair_status
            ]
        )


# 顯示處理結果
print("left_shift = 2，第 1 階段處理完成")
print()

print("修正版檔案：", shift2_stage1_path)
print("修正對照檔：", shift2_stage1_audit_path)
print()

print("left_shift = 2 總列數：", total_shift2)
print("FLD 成功擷取：", fld_success_count)
print("FLD 無法擷取：", fld_failed_count)
print()

print(
    "HomFLD 原本為純數字：",
    homfld_already_numeric_count
)

print(
    "HomFLD 從亂碼成功擷取：",
    homfld_extracted_count
)

print(
    "HomFLD 無法擷取：",
    homfld_failed_count
)

print()
print("失敗範例：")

for example in failed_examples:
    print(example)

left_shift = 2，第 1 階段處理完成

修正版檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage1_clean_FLD_HomFLD.txt
修正對照檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage1_audit.txt

left_shift = 2 總列數： 3296
FLD 成功擷取： 3296
FLD 無法擷取： 0

HomFLD 原本為純數字： 3058
HomFLD 從亂碼成功擷取： 238
HomFLD 無法擷取： 0

失敗範例：


In [23]:
# 載入 csv 模組，用來讀取 Tab 分隔檔案
import csv

# 載入 re 模組，用來判斷欄位是否為完整數字
import re


# 定義完整數字格式
# 可接受：
# 0.548
# -0.0327
# 12
# .35
# 1.2e-3
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 建立檢查計數器
total_rows = 0                 # 總資料列數
fld_numeric_count = 0          # FLD 為純數字的列數
homfld_numeric_count = 0       # HomFLD 為純數字的列數
both_numeric_count = 0         # FLD 與 HomFLD 都為純數字
repair_success_count = 0       # repair_status 為 success

# 保存前 20 筆仍有問題的資料
problem_examples = []

# 保存前 5 筆清理後範例
clean_examples = []


# 開啟第 1 階段清理後的檔案
with shift2_stage1_path.open(
    "r",                       # 讀取模式
    encoding="utf-8",          # 使用 UTF-8
    errors="replace",          # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        total_rows += 1

        # 取得清理後的 FLD 與 HomFLD
        fld_value = row["FLD"].strip()
        homfld_value = row["HomFLD"].strip()

        # 判斷 FLD 是否為完整純數字
        fld_is_numeric = bool(
            full_number_pattern.fullmatch(fld_value)
        )

        # 判斷 HomFLD 是否為完整純數字
        homfld_is_numeric = bool(
            full_number_pattern.fullmatch(homfld_value)
        )

        # 分別累計
        if fld_is_numeric:
            fld_numeric_count += 1

        if homfld_is_numeric:
            homfld_numeric_count += 1

        # 兩欄都為純數字
        if fld_is_numeric and homfld_is_numeric:
            both_numeric_count += 1

        # 檢查程式先前標記的修復狀態
        if row["repair_status"] == "success":
            repair_success_count += 1

        # 保存前 5 筆正常範例
        if (
            fld_is_numeric
            and homfld_is_numeric
            and len(clean_examples) < 5
        ):
            clean_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],
                    "FLD": fld_value,
                    "HomFLD": homfld_value,
                    "specialSNP_chr":
                        row["specialSNP_chr"],
                    "gender_metrics":
                        row["gender_metrics"]
                }
            )

        # 保存前 20 筆仍有問題的範例
        if (
            not fld_is_numeric
            or not homfld_is_numeric
        ):
            if len(problem_examples) < 20:
                problem_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],
                        "FLD": fld_value,
                        "HomFLD": homfld_value,
                        "repair_status":
                            row["repair_status"]
                    }
                )


# 顯示整體驗證結果
print("left_shift = 2，第 1 階段驗證結果")
print()

print("總列數：", total_rows)
print("FLD 為純數字：", fld_numeric_count)
print("HomFLD 為純數字：", homfld_numeric_count)
print("FLD 與 HomFLD 都為純數字：", both_numeric_count)
print("repair_status = success：", repair_success_count)

print()
print("前 5 筆清理後範例：")

for example in clean_examples:
    print(example)

print()
print("仍有問題的範例：")

for example in problem_examples:
    print(example)

left_shift = 2，第 1 階段驗證結果

總列數： 3296
FLD 為純數字： 3296
HomFLD 為純數字： 3296
FLD 與 HomFLD 都為純數字： 3296
repair_status = success： 3296

前 5 筆清理後範例：
{'source_row_number': '88', 'FLD': '0.548', 'HomFLD': '3.205', 'specialSNP_chr': 'PolyHighResolution', 'gender_metrics': '1'}
{'source_row_number': '141', 'FLD': '0.43', 'HomFLD': '2.379', 'specialSNP_chr': 'PolyHighResolution', 'gender_metrics': '1'}
{'source_row_number': '459', 'FLD': '0.702', 'HomFLD': '1.583', 'specialSNP_chr': 'NoMinorHom', 'gender_metrics': '1'}
{'source_row_number': '843', 'FLD': '8.928', 'HomFLD': '0.723', 'specialSNP_chr': 'PolyHighResolution', 'gender_metrics': '1'}
{'source_row_number': '932', 'FLD': '0.441', 'HomFLD': '2.365', 'specialSNP_chr': 'NoMinorHom', 'gender_metrics': '1'}

仍有問題的範例：


In [24]:
# 載入 Counter
# 用來統計不同群組、ConversionType 與 genotype 總數
from collections import Counter

# 設定不同 gender_metrics 對應的預期樣本數
expected_total_by_gender = {
    "all": 240,       # 全部240位樣本
    "female": 163,    # 女性163位
    "male": 77        # 男性77位
}

# 合理的染色體分類
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}

# 合理的 gender_metrics
valid_gender_values = {
    "all",
    "female",
    "male"
}

# 合理的 hemizygous 值
valid_hemizygous_values = {
    "0",
    "1"
}


# 建立檢查計數器
total_rows = 0                    # left_shift = 2 總列數
structure_pass_count = 0          # 文字欄位結構合理
genotype_qc_pass_count = 0        # genotype count 合計合理
genotype_qc_fail_count = 0        # genotype count 合計不合理
best_columns_pass_count = 0       # BestProbeset等欄位合理

# 統計不同 genotype count 合計值
batch_total_counts = Counter()

# 統計不同候選資料群組
group_counts = Counter()

# 統計候選 ConversionType
conversion_counts = Counter()

# 保存前20筆未通過範例
failed_examples = []


# 開啟位移2、第1階段清理後的檔案
with shift2_stage1_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        total_rows += 1

        # -------------------------------------------------
        # 假設缺少 nMinorAllele 與 Nclus
        # 因此目前這些欄位中的值應向右移兩格
        # -------------------------------------------------

        # 目前 nMinorAllele 的值，候選對應真正的 n_AA
        candidate_n_AA = row["nMinorAllele"]

        # 目前 Nclus 的值，候選對應真正的 n_AB
        candidate_n_AB = row["Nclus"]

        # 目前 n_AA 的值，候選對應真正的 n_BB
        candidate_n_BB = row["n_AA"]

        # 目前 n_AB 的值，候選對應真正的 n_NC
        candidate_n_NC = row["n_AB"]

        # 目前 n_BB 的值，候選對應真正的 hemizygous
        candidate_hemizygous = row["n_BB"]

        # 目前 n_NC 的值，候選對應真正的 specialSNP_chr
        candidate_chr = row["n_NC"]

        # 目前 hemizygous 的值，候選對應真正的 gender_metrics
        candidate_gender = row["hemizygous"]

        # 目前 specialSNP_chr 的值，候選對應真正的 ConversionType
        candidate_conversion = row["specialSNP_chr"]

        # 目前 gender_metrics 的值，候選對應真正的 BestProbeset
        candidate_best_probeset = row["gender_metrics"]

        # 目前 ConversionType 的值，候選對應真正的 BestandRecommended
        candidate_best_recommended = row["ConversionType"]

        # 目前 BestProbeset 的值，候選對應真正的 HomHet
        candidate_homhet = row["BestProbeset"]


        # -------------------------------------------------
        # 檢查文字欄位位置是否合理
        # -------------------------------------------------

        structure_ok = (
            candidate_hemizygous in valid_hemizygous_values
            and candidate_chr in valid_chr_values
            and candidate_gender in valid_gender_values
            and candidate_conversion
            in {"PolyHighResolution", "NoMinorHom"}
        )

        if structure_ok:
            structure_pass_count += 1


        # -------------------------------------------------
        # 檢查 BestProbeset 等欄位是否合理
        # -------------------------------------------------

        best_columns_ok = (
            candidate_best_probeset in {"0", "1"}
            and candidate_best_recommended in {"0", "1"}
            and candidate_homhet in {"0", "1"}
        )

        if best_columns_ok:
            best_columns_pass_count += 1


        # -------------------------------------------------
        # 統計候選群組
        # -------------------------------------------------

        group_counts[
            (
                candidate_chr,
                candidate_gender,
                candidate_conversion
            )
        ] += 1

        conversion_counts[
            candidate_conversion
        ] += 1


        # -------------------------------------------------
        # 檢查 genotype count
        # -------------------------------------------------

        try:
            # 將四個候選 count 轉成整數
            candidate_counts = [
                int(candidate_n_AA),
                int(candidate_n_AB),
                int(candidate_n_BB),
                int(candidate_n_NC)
            ]

            # 計算四個 genotype count 合計
            batch_total = sum(candidate_counts)

            # 統計實際合計值
            batch_total_counts[batch_total] += 1

            # 根據 candidate_gender 決定合理樣本數
            expected_total = expected_total_by_gender.get(
                candidate_gender
            )

            # 實際總數符合該群組預期樣本數
            count_ok = (
                expected_total is not None
                and batch_total == expected_total
            )

        except (ValueError, TypeError):

            # 無法轉為整數
            batch_total = None
            expected_total = expected_total_by_gender.get(
                candidate_gender
            )
            count_ok = False


        # 通過 genotype count QC
        if count_ok:
            genotype_qc_pass_count += 1

        # 未通過 genotype count QC
        else:
            genotype_qc_fail_count += 1

            # 最多保存前20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_n_AA":
                            candidate_n_AA,

                        "candidate_n_AB":
                            candidate_n_AB,

                        "candidate_n_BB":
                            candidate_n_BB,

                        "candidate_n_NC":
                            candidate_n_NC,

                        "batch_total":
                            batch_total,

                        "expected_total":
                            expected_total,

                        "candidate_hemizygous":
                            candidate_hemizygous,

                        "candidate_chr":
                            candidate_chr,

                        "candidate_gender":
                            candidate_gender,

                        "candidate_conversion":
                            candidate_conversion
                    }
                )


# 顯示整體檢查結果
print("left_shift = 2 候選欄位對應驗證")
print()

print("總列數：", total_rows)

print(
    "文字欄位結構合理：",
    structure_pass_count
)

print(
    "BestProbeset相關欄位合理：",
    best_columns_pass_count
)

print(
    "genotype count QC 通過：",
    genotype_qc_pass_count
)

print(
    "genotype count QC 未通過：",
    genotype_qc_fail_count
)


# 顯示 genotype count 合計分布
print()
print("genotype count 合計分布：")

for batch_total, count in batch_total_counts.most_common():
    print(
        "合計 =",
        batch_total,
        "| 列數 =",
        count
    )


# 顯示主要資料群組
print()
print("候選 chromosome／gender／ConversionType 組合：")

for group, count in group_counts.most_common():
    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| ConversionType =",
        group[2],
        "| 列數 =",
        count
    )


# 顯示 ConversionType 分布
print()
print("候選 ConversionType 分布：")

for conversion, count in conversion_counts.most_common():
    print(
        conversion,
        "：",
        count
    )


# 顯示未通過範例
print()
print("前20筆未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2 候選欄位對應驗證

總列數： 3296
文字欄位結構合理： 3296
BestProbeset相關欄位合理： 3296
genotype count QC 通過： 3296
genotype count QC 未通過： 0

genotype count 合計分布：
合計 = 163 | 列數 = 3058
合計 = 240 | 列數 = 238

候選 chromosome／gender／ConversionType 組合：
specialSNP_chr = X | gender_metrics = female | ConversionType = NoMinorHom | 列數 = 1681
specialSNP_chr = X | gender_metrics = female | ConversionType = PolyHighResolution | 列數 = 1377
specialSNP_chr = MT | gender_metrics = all | ConversionType = PolyHighResolution | 列數 = 238

候選 ConversionType 分布：
NoMinorHom ： 1681
PolyHighResolution ： 1615

前20筆未通過範例：


In [25]:
#n_AA 到 BB.meanY向右修正兩欄
# 設定第 2 階段的輸出檔案
# partial 表示目前只修正到 BB.meanY
shift2_stage2_path = (
    output_folder
    / "genotype_back_shift2_stage2_partial_fixed.txt"
)


# 設定修正前後的對照紀錄檔
shift2_stage2_audit_path = (
    output_folder
    / "genotype_back_shift2_stage2_audit.txt"
)


# 原始欄位中的值，依序應移到右邊兩格的目標欄位
source_fields = [
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY"
]


# 上面 source_fields 中的值，修正後應放入這些欄位
target_fields = [
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY"
]


# 建立處理結果計數器
total_rows = 0
qc_pass_count = 0
qc_fail_count = 0

# 保存前 20 筆未通過檢查的資料
failed_examples = []


# 開啟第 1 階段已清理 FLD、HomFLD 的檔案
with shift2_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift2_stage2_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_stage2_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 保留原始欄位順序
    output_columns = reader.fieldnames

    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_columns,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 寫入修正版表頭
    writer.writeheader()


    # 對照檔保存修正前後的關鍵欄位
    audit_columns = (
        ["source_row_number"]
        + ["raw_" + field for field in source_fields]
        + ["fixed_nMinorAllele", "fixed_Nclus"]
        + ["fixed_" + field for field in target_fields]
    )

    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_columns,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 寫入對照檔表頭
    audit_writer.writeheader()


    # 逐列處理 3,296 筆 left_shift = 2
    for row in reader:

        total_rows += 1

        # 先保存所有要搬移的原始值
        # 必須先保存，避免移動過程中覆蓋後面的值
        original_values = {
            field: row[field]
            for field in source_fields
        }


        # nMinorAllele 與 Nclus 沒有可靠的原始值
        # 暫時設為空白，不猜測填值
        row["nMinorAllele"] = ""
        row["Nclus"] = ""


        # 將原始值依照對應關係放入右邊兩格的欄位
        for source_field, target_field in zip(
            source_fields,
            target_fields
        ):
            row[target_field] = original_values[source_field]


        # ----------------------------------------
        # 修正後的合理性檢查
        # ----------------------------------------

        try:
            # 計算修正後四個 genotype count
            batch_total = (
                int(row["n_AA"])
                + int(row["n_AB"])
                + int(row["n_BB"])
                + int(row["n_NC"])
            )

            # 根據 gender_metrics 決定預期樣本數
            expected_total_by_gender = {
                "all": 240,
                "female": 163,
                "male": 77
            }

            expected_total = expected_total_by_gender.get(
                row["gender_metrics"]
            )

            # 同時檢查 count 與文字錨點
            qc_ok = (
                batch_total == expected_total
                and row["hemizygous"] in {"0", "1"}
                and row["specialSNP_chr"]
                in {"autosomal", "X", "Y", "MT"}
                and row["gender_metrics"]
                in {"all", "female", "male"}
                and row["ConversionType"]
                in {"PolyHighResolution", "NoMinorHom"}
                and row["BestProbeset"] in {"0", "1"}
                and row["BestandRecommended"] in {"0", "1"}
                and row["HomHet"] in {"0", "1"}
            )

        except (ValueError, TypeError):
            batch_total = None
            expected_total = None
            qc_ok = False


        # 累計 QC 結果
        if qc_ok:
            qc_pass_count += 1

        else:
            qc_fail_count += 1

            # 保存前 20 筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],
                        "batch_total":
                            batch_total,
                        "expected_total":
                            expected_total,
                        "specialSNP_chr":
                            row["specialSNP_chr"],
                        "gender_metrics":
                            row["gender_metrics"],
                        "ConversionType":
                            row["ConversionType"]
                    }
                )


        # 寫入部分修正後的資料
        writer.writerow(row)


        # 建立修正前後對照紀錄
        audit_row = {
            "source_row_number":
                row["source_row_number"],

            "fixed_nMinorAllele":
                row["nMinorAllele"],

            "fixed_Nclus":
                row["Nclus"]
        }

        # 加入修正前的值
        for field in source_fields:
            audit_row["raw_" + field] = (
                original_values[field]
            )

        # 加入修正後的值
        for field in target_fields:
            audit_row["fixed_" + field] = row[field]

        # 寫入對照紀錄
        audit_writer.writerow(audit_row)


# 顯示結果
print("left_shift = 2，第 2 階段處理完成")
print()

print("輸出檔案：", shift2_stage2_path)
print("對照紀錄：", shift2_stage2_audit_path)
print()

print("總列數：", total_rows)
print("修正後 QC 通過：", qc_pass_count)
print("修正後 QC 未通過：", qc_fail_count)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，第 2 階段處理完成

輸出檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage2_partial_fixed.txt
對照紀錄： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_stage2_audit.txt

總列數： 3296
修正後 QC 通過： 3296
修正後 QC 未通過： 0

未通過範例：


處理NC.meanX 到 Call Modified

In [26]:
# 載入 csv 模組
# 用來讀取 Tab 分隔的 TXT 檔案
import csv

# 載入正規表示式模組
# 用來判斷某個值是否為純數字
import re

# Counter 用來統計各種欄位排列模式出現的次數
from collections import Counter


# 設定這次要分析的尾端欄位
tail_columns = [
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 設定分析報告的輸出位置
shift2_tail_report_path = (
    output_folder
    / "shift2_tail_pattern_summary.txt"
)


# 定義完整數字格式
# 可辨認：
# 0
# 0.0031
# -0.0327
# 58.27
# 1.32E-05
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 建立函數：判斷欄位值的資料型態
def classify_tail_value(value):

    # 避免值為 None
    if value is None:
        return "missing"

    # 去除字串前後空白
    value = value.strip()

    # 空字串
    if value == "":
        return "empty"

    # NA
    if value.upper() == "NA":
        return "NA"

    # TRUE 或 FALSE
    if value.upper() in {"TRUE", "FALSE"}:
        return "boolean"

    # 完整純數字
    if full_number_pattern.fullmatch(value):
        return "numeric"

    # 出現解碼替代符號或問號
    if "�" in value or "?" in value:
        return "garbled"

    # 其他文字
    return "text"


# 統計尾端欄位型態模式
tail_pattern_counts = Counter()

# 保存每種模式的前 3 筆實際範例
tail_pattern_examples = {}

# 統計模式和染色體、性別、ConversionType、HomRO的關係
group_pattern_counts = Counter()

# 記錄總列數
total_rows = 0


# 開啟位移 2、第 2 階段部分修正後的檔案
with shift2_stage2_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼的字元以 � 顯示
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 確認尾端欄位都存在
    missing_columns = [
        column
        for column in tail_columns
        if column not in reader.fieldnames
    ]

    # 若有欄位不存在，停止執行並顯示錯誤
    if missing_columns:
        raise ValueError(
            f"找不到以下欄位：{missing_columns}"
        )


    # 逐列分析
    for row in reader:

        total_rows += 1


        # 建立此列的尾端欄位型態模式
        #
        # 例如：
        # garbled, numeric, NA, boolean, empty, empty, empty
        pattern_signature = tuple(
            classify_tail_value(row[column])
            for column in tail_columns
        )


        # 統計此模式出現次數
        tail_pattern_counts[pattern_signature] += 1


        # 第一次遇到此模式時建立範例清單
        if pattern_signature not in tail_pattern_examples:
            tail_pattern_examples[pattern_signature] = []


        # 每種模式最多保存前 3 筆實際資料
        if len(tail_pattern_examples[pattern_signature]) < 3:

            tail_pattern_examples[pattern_signature].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "HomRO":
                        row["HomRO"],

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "tail_values": {
                        column: row[column]
                        for column in tail_columns
                    }
                }
            )


        # 同時統計此模式與前面欄位的關係
        group_key = (
            row["specialSNP_chr"],
            row["gender_metrics"],
            row["ConversionType"],
            row["HomRO"],
            pattern_signature
        )

        group_pattern_counts[group_key] += 1


# 將結果寫入報告檔
with shift2_tail_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    # 寫入基本摘要
    report.write(
        f"left_shift = 2 total rows: {total_rows}\n"
    )

    report.write(
        f"Number of tail patterns: "
        f"{len(tail_pattern_counts)}\n\n"
    )


    # 依出現次數由多到少輸出所有模式
    for rank, (
        pattern_signature,
        count
    ) in enumerate(
        tail_pattern_counts.most_common(),
        start=1
    ):

        report.write("=" * 80 + "\n")
        report.write(f"Pattern rank: {rank}\n")
        report.write(f"Count: {count}\n")
        report.write("-" * 80 + "\n")


        # 顯示每個尾端欄位的資料型態
        for column, value_type in zip(
            tail_columns,
            pattern_signature
        ):
            report.write(
                f"{column}: {value_type}\n"
            )


        # 顯示此模式的實際範例
        report.write("\nExamples:\n")

        for example in tail_pattern_examples[
            pattern_signature
        ]:

            report.write(
                f"source_row_number: "
                f"{example['source_row_number']}\n"
            )

            report.write(
                f"HomRO: {repr(example['HomRO'])}\n"
            )

            report.write(
                f"specialSNP_chr: "
                f"{repr(example['specialSNP_chr'])}\n"
            )

            report.write(
                f"gender_metrics: "
                f"{repr(example['gender_metrics'])}\n"
            )

            report.write(
                f"ConversionType: "
                f"{repr(example['ConversionType'])}\n"
            )


            # 顯示尾端 7 欄實際內容
            for column in tail_columns:

                value = example["tail_values"][column]

                report.write(
                    f"  {column}: {repr(value)}\n"
                )

            report.write("\n")


    # 顯示模式與前段特徵的組合
    report.write("\n" + "=" * 80 + "\n")
    report.write("Pattern grouping summary\n")
    report.write("=" * 80 + "\n\n")


    for group_key, count in (
        group_pattern_counts.most_common()
    ):

        chromosome = group_key[0]
        gender = group_key[1]
        conversion = group_key[2]
        homro = group_key[3]
        pattern_signature = group_key[4]

        report.write(
            f"specialSNP_chr={chromosome} | "
            f"gender_metrics={gender} | "
            f"ConversionType={conversion} | "
            f"HomRO={homro} | "
            f"pattern={pattern_signature} | "
            f"count={count}\n"
        )


# 顯示執行結果
print("left_shift = 2 尾端欄位分析完成")
print()

print("總列數：", total_rows)
print("尾端型態模式數：", len(tail_pattern_counts))
print("報告位置：", shift2_tail_report_path)

print()
print("各尾端模式列數：")

for rank, (
    pattern_signature,
    count
) in enumerate(
    tail_pattern_counts.most_common(),
    start=1
):
    print(
        "模式",
        rank,
        "| 列數 =",
        count,
        "|",
        pattern_signature
    )

left_shift = 2 尾端欄位分析完成

總列數： 3296
尾端型態模式數： 2
報告位置： /Users/sheena/Desktop/summerintern/split_data/shift2_tail_pattern_summary.txt

各尾端模式列數：
模式 1 | 列數 = 3190 | ('garbled', 'numeric', 'NA', 'boolean', 'empty', 'empty', 'empty')
模式 2 | 列數 = 106 | ('numeric', 'numeric', 'numeric', 'NA', 'boolean', 'empty', 'empty')


In [28]:
# 指定要查看的尾端欄位
tail_columns = [
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 儲存兩種模式的範例
pattern_examples = {
    1: [],
    2: []
}


# 開啟位移2、第2階段部分修正後的檔案
with shift2_stage2_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",       # 無法解碼字元以 � 顯示
    newline=""
) as file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # 判斷目前屬於哪一種尾端模式
        #
        # 模式1：
        # NC.meanX 含亂碼
        # MMD = NA
        # MinorAlleleFrequency = FALSE
        if (
            ("�" in row["NC.meanX"] or "?" in row["NC.meanX"])
            and row["MMD"] == "NA"
            and row["MinorAlleleFrequency"] in {"TRUE", "FALSE"}
        ):
            pattern_id = 1

        # 模式2：
        # NC.meanX、NC.meanY、MMD 為數字
        # MinorAlleleFrequency = NA
        # H.W.p-Value = FALSE
        elif (
            full_number_pattern.fullmatch(
                row["NC.meanX"].strip()
            )
            and full_number_pattern.fullmatch(
                row["NC.meanY"].strip()
            )
            and full_number_pattern.fullmatch(
                row["MMD"].strip()
            )
            and row["MinorAlleleFrequency"] == "NA"
            and row["H.W.p-Value"] in {"TRUE", "FALSE"}
        ):
            pattern_id = 2

        # 不符合目前兩種模式時先跳過
        else:
            continue


        # 每種模式最多保存5筆
        if len(pattern_examples[pattern_id]) < 5:

            pattern_examples[pattern_id].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "values": {
                        column: row[column]
                        for column in tail_columns
                    }
                }
            )


        # 兩種模式都各取得5筆後停止
        if (
            len(pattern_examples[1]) == 5
            and len(pattern_examples[2]) == 5
        ):
            break


# 顯示兩種模式的實際內容
for pattern_id in [1, 2]:

    print("=" * 80)
    print("尾端模式：", pattern_id)
    print("=" * 80)

    for example in pattern_examples[pattern_id]:

        print(
            "source_row_number：",
            example["source_row_number"]
        )

        print(
            "specialSNP_chr：",
            example["specialSNP_chr"]
        )

        print(
            "gender_metrics：",
            example["gender_metrics"]
        )

        print(
            "ConversionType：",
            example["ConversionType"]
        )

        # 顯示尾端欄位內容
        for column in tail_columns:
            print(
                f"{column}: "
                f"{repr(example['values'][column])}"
            )

        print("-" * 80)

尾端模式： 1
source_row_number： 88
specialSNP_chr： X
gender_metrics： female
ConversionType： PolyHighResolution
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 141
specialSNP_chr： X
gender_metrics： female
ConversionType： PolyHighResolution
BB.meanX: '-2.3795'
BB.meanY: '9.5921'
NC.meanX: '?�數??0.0092'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 459
specialSNP_chr： X
gender_metrics： female
ConversionType： NoMinorHom
BB.meanX: '-1.5826'
BB.meanY: '10.1036'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
----------

In [29]:
# Counter 用來統計 FALSE 出現在各欄位的次數
from collections import Counter

# 設定要檢查的尾端欄位
tail_columns = [
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]

# 統計 FALSE 所在位置
false_position_counts = Counter()

# 統計 TRUE 所在位置
true_position_counts = Counter()

# 統計每列中布林值出現在哪些欄位
boolean_pattern_counts = Counter()

# 保存每種模式前 3 筆範例
boolean_pattern_examples = {}

# 開啟位移 2、第 2 階段部分修正後的檔案
with shift2_stage2_path.open(
    "r",                    # 讀取模式
    encoding="utf-8",       # 使用 UTF-8
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # 找出這一列 FALSE 出現在哪些欄位
        false_columns = tuple(
            column
            for column in tail_columns
            if row[column].strip().upper() == "FALSE"
        )

        # 找出這一列 TRUE 出現在哪些欄位
        true_columns = tuple(
            column
            for column in tail_columns
            if row[column].strip().upper() == "TRUE"
        )

        # 分別統計 FALSE 的欄位位置
        for column in false_columns:
            false_position_counts[column] += 1

        # 分別統計 TRUE 的欄位位置
        for column in true_columns:
            true_position_counts[column] += 1

        # 用 FALSE 與 TRUE 的位置組成子模式
        boolean_pattern = (
            false_columns,
            true_columns
        )

        # 統計子模式列數
        boolean_pattern_counts[boolean_pattern] += 1

        # 保存每種子模式前 3 筆範例
        if boolean_pattern not in boolean_pattern_examples:
            boolean_pattern_examples[boolean_pattern] = []

        if len(boolean_pattern_examples[boolean_pattern]) < 3:
            boolean_pattern_examples[boolean_pattern].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "values": {
                        column: row[column]
                        for column in tail_columns
                    }
                }
            )


# 顯示 FALSE 各位置的列數
print("FALSE 出現位置統計：")

for column, count in false_position_counts.most_common():
    print(column, "：", count)


# 顯示 TRUE 各位置的列數
print()
print("TRUE 出現位置統計：")

for column, count in true_position_counts.most_common():
    print(column, "：", count)


# 顯示布林值位置子模式
print()
print("布林值位置子模式：")

for rank, (pattern, count) in enumerate(
    boolean_pattern_counts.most_common(),
    start=1
):
    print("=" * 70)
    print("模式：", rank)
    print("列數：", count)
    print("FALSE 欄位：", pattern[0])
    print("TRUE 欄位：", pattern[1])

    # 顯示該模式的實際範例
    for example in boolean_pattern_examples[pattern]:

        print(
            "source_row_number：",
            example["source_row_number"]
        )

        print(
            "specialSNP_chr：",
            example["specialSNP_chr"]
        )

        # 顯示尾端 7 欄原始內容
        for column in tail_columns:
            print(
                f"{column}: "
                f"{repr(example['values'][column])}"
            )

        print("-" * 50)

FALSE 出現位置統計：
MinorAlleleFrequency ： 3190
H.W.p-Value ： 106

TRUE 出現位置統計：

布林值位置子模式：
模式： 1
列數： 3190
FALSE 欄位： ('MinorAlleleFrequency',)
TRUE 欄位： ()
source_row_number： 88
specialSNP_chr： X
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------
source_row_number： 141
specialSNP_chr： X
NC.meanX: '?�數??0.0092'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------
source_row_number： 459
specialSNP_chr： X
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------
模式： 2
列數： 106
FALSE 欄位： ('H.W.p-Value',)
TRUE 欄位： ()
source_row_number： 9558
specialSNP_chr： X
NC.meanX: '41.438'
NC.meanY: '0.0184'
MMD: '0.0457'
MinorAl

In [30]:
# 設定要查看的欄位
tail_check_columns = [
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]

# 分別保存 X 和 MT 的前 3 筆範例
examples = {
    "X": [],
    "MT": []
}

# 必須從 stage1 檔案讀取
# 因為 stage1 只清理 FLD、HomFLD，其他欄位仍保留原始狀態
with shift2_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取 Tab 分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列讀取
    for row in reader:

        # 在 stage1 尚未修正位移
        # 原始 n_NC 位置放的是實際 specialSNP_chr
        chromosome = row["n_NC"]

        # 只查看 X 和 MT
        if chromosome not in examples:
            continue

        # 每一類最多保存 3 筆
        if len(examples[chromosome]) < 3:
            examples[chromosome].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "values": {
                        column: row[column]
                        for column in tail_check_columns
                    }
                }
            )

        # X 與 MT 都各取得 3 筆後停止
        if (
            len(examples["X"]) == 3
            and len(examples["MT"]) == 3
        ):
            break


# 顯示結果
for chromosome in ["X", "MT"]:

    print("=" * 80)
    print("原始 specialSNP_chr：", chromosome)
    print("=" * 80)

    for example in examples[chromosome]:

        print(
            "source_row_number：",
            example["source_row_number"]
        )

        for column in tail_check_columns:
            print(
                f"{column}: "
                f"{repr(example['values'][column])}"
            )

        print("-" * 80)

原始 specialSNP_chr： X
source_row_number： 88
AB.meanX: 'NA'
AB.meanY: 'NA'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 141
AB.meanX: '-2.3795'
AB.meanY: '9.5921'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0092'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
source_row_number： 459
AB.meanX: '-1.5826'
AB.meanY: '10.1036'
BB.meanX: 'NA'
BB.meanY: 'NA'
NC.meanX: '?�數??0.0123'
NC.meanY: '1'
MMD: 'NA'
MinorAlleleFrequency: 'FALSE'
H.W.p-Value: ''
H.W.chisquared.statistic: ''
Call Modified: ''
--------------------------------------------------------------------------------
原始 specialSNP_chr： MT
source_row_number： 843
A

In [31]:
# 載入處理 Tab 分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從亂碼字串尾端擷取數字
import re


# 定義完整數字格式
# 可辨認 0、1、0.0123、-0.5、1.2E-5
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端數字格式
# 例如亂碼加0.0123會擷取0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_x_rows = 0

# 成功從原始 NC.meanX 擷取候選 MAF
maf_extract_success = 0

# 候選 MAF 落在0到0.5
maf_range_pass = 0

# 候選 H.W.p-Value 落在0到1
hwp_range_pass = 0

# 候選 H.W.chisquared.statistic為NA或非負數
hwchi_pass = 0

# 候選 Call Modified為TRUE或FALSE
call_modified_pass = 0

# NC.meanX與NC.meanY候選值為數字或NA
nc_mean_pass = 0

# 所有條件同時通過
all_conditions_pass = 0

# 保存前20筆未通過範例
failed_examples = []

# 保存前5筆候選對應範例
candidate_examples = []


# 必須讀取stage1檔案
# stage1只清理FLD與HomFLD，其他欄位仍保留原始內容
with shift2_stage1_path.open(
    "r",                    # 以讀取模式開啟
    encoding="utf-8",       # 使用UTF-8
    errors="replace",       # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # stage1尚未移位
        # 原始n_NC欄位中的X才是真正的specialSNP_chr
        if row["n_NC"] != "X":
            continue

        total_x_rows += 1


        # -----------------------------------
        # 候選NC cluster mean
        # -----------------------------------

        # 原始BB.meanX候選對應真正NC.meanX
        candidate_nc_mean_x = row["BB.meanX"].strip()

        # 原始BB.meanY候選對應真正NC.meanY
        candidate_nc_mean_y = row["BB.meanY"].strip()


        # 判斷兩個候選值是否為數字或NA
        nc_x_ok = (
            candidate_nc_mean_x.upper() == "NA"
            or full_number_pattern.fullmatch(
                candidate_nc_mean_x
            )
        )

        nc_y_ok = (
            candidate_nc_mean_y.upper() == "NA"
            or full_number_pattern.fullmatch(
                candidate_nc_mean_y
            )
        )

        nc_mean_ok = bool(nc_x_ok and nc_y_ok)

        if nc_mean_ok:
            nc_mean_pass += 1


        # -----------------------------------
        # 候選MinorAlleleFrequency
        # -----------------------------------

        # 從原始NC.meanX亂碼字串尾端擷取數字
        maf_match = ending_number_pattern.search(
            row["NC.meanX"].strip()
        )

        if maf_match:

            # 取得候選MAF
            candidate_maf = maf_match.group(1)

            maf_extract_success += 1

            # 轉成float檢查值域
            maf_value = float(candidate_maf)

            # Minor allele frequency理論上介於0與0.5
            maf_ok = 0 <= maf_value <= 0.5

        else:

            candidate_maf = ""
            maf_value = None
            maf_ok = False

        if maf_ok:
            maf_range_pass += 1


        # -----------------------------------
        # 候選H.W.p-Value
        # -----------------------------------

        # 原始NC.meanY候選對應真正H.W.p-Value
        candidate_hwp = row["NC.meanY"].strip()

        try:
            hwp_value = float(candidate_hwp)

            # p-value應介於0與1
            hwp_ok = 0 <= hwp_value <= 1

        except ValueError:
            hwp_value = None
            hwp_ok = False

        if hwp_ok:
            hwp_range_pass += 1


        # -----------------------------------
        # 候選H.W.chisquared.statistic
        # -----------------------------------

        # 原始MMD候選對應真正H.W.chisquared.statistic
        candidate_hwchi = row["MMD"].strip()

        # NA視為可接受
        if candidate_hwchi.upper() == "NA":

            hwchi_ok = True

        else:

            try:
                hwchi_value = float(candidate_hwchi)

                # 卡方統計量應為非負值
                hwchi_ok = hwchi_value >= 0

            except ValueError:
                hwchi_ok = False

        if hwchi_ok:
            hwchi_pass += 1


        # -----------------------------------
        # 候選Call Modified
        # -----------------------------------

        # 原始MinorAlleleFrequency候選對應Call Modified
        candidate_call_modified = (
            row["MinorAlleleFrequency"]
            .strip()
            .upper()
        )

        call_modified_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )

        if call_modified_ok:
            call_modified_pass += 1


        # -----------------------------------
        # 綜合判斷
        # -----------------------------------

        all_ok = (
            nc_mean_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
        )

        if all_ok:
            all_conditions_pass += 1

        else:

            # 最多保存前20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_NC.meanX":
                            candidate_nc_mean_x,

                        "candidate_NC.meanY":
                            candidate_nc_mean_y,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified
                    }
                )


        # 保存前5筆候選修正範例
        if len(candidate_examples) < 5:
            candidate_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "fixed_NC.meanX":
                        candidate_nc_mean_x,

                    "fixed_NC.meanY":
                        candidate_nc_mean_y,

                    "fixed_MMD":
                        "",

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    "fixed_H.W.chisquared.statistic":
                        candidate_hwchi,

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# 顯示驗證結果
print("left_shift = 2，X chromosome尾端候選對應驗證")
print()

print("X資料總列數：", total_x_rows)
print("候選NC mean合理：", nc_mean_pass)
print("MAF尾端數值擷取成功：", maf_extract_success)
print("候選MAF值域合理：", maf_range_pass)
print("候選H.W.p-Value合理：", hwp_range_pass)
print("候選H.W.chisquared合理：", hwchi_pass)
print("候選Call Modified合理：", call_modified_pass)
print("所有條件同時通過：", all_conditions_pass)

print()
print("前5筆候選修正範例：")

for example in candidate_examples:
    print(example)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，X chromosome尾端候選對應驗證

X資料總列數： 3058
候選NC mean合理： 3058
MAF尾端數值擷取成功： 3058
候選MAF值域合理： 2961
候選H.W.p-Value合理： 3058
候選H.W.chisquared合理： 3058
候選Call Modified合理： 2952
所有條件同時通過： 2952

前5筆候選修正範例：
{'source_row_number': '88', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0123', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '141', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0092', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '459', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0123', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '932', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_Mino

In [31]:
#先修復x染色體
# 載入 Tab 分隔檔案與正規表示式工具
import csv
import re


# 定義完整數字格式
# 可以辨認整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 建立檢查計數器
total_pattern_b = 0              # X-B模式總列數
nc_mean_pass = 0                 # NC.meanX、NC.meanY合理
mmd_pass = 0                     # 候選MMD合理
maf_pass = 0                     # 候選MAF合理
hwp_pass = 0                     # 候選H.W.p-Value合理
hwchi_pass = 0                   # 候選卡方值合理
call_modified_pass = 0           # 候選Call Modified合理
all_pass = 0                     # 所有條件同時通過

# 保存未通過資料的前20筆
failed_examples = []

# 保存前5筆候選對應
candidate_examples = []


# 必須從stage1讀取
# stage1只清理FLD、HomFLD，其他欄仍是原始位置
with shift2_stage1_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用UTF-8
    errors="replace",            # 無法解碼字元以替代符號保留
    newline=""
) as file:

    # 使用欄名讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # stage1中，真正的specialSNP_chr仍位於n_NC
        # 只處理X染色體
        if row["n_NC"] != "X":
            continue

        # X-B模式的特徵：
        # 目前MinorAlleleFrequency為NA
        # 目前H.W.p-Value為TRUE或FALSE
        if not (
            row["MinorAlleleFrequency"].strip().upper() == "NA"
            and row["H.W.p-Value"].strip().upper()
            in {"TRUE", "FALSE"}
        ):
            continue

        total_pattern_b += 1


        # -----------------------------------
        # 候選NC cluster mean
        # -----------------------------------

        # 原始BB.meanX、BB.meanY
        # 候選對應真正的NC.meanX、NC.meanY
        candidate_nc_x = row["BB.meanX"].strip()
        candidate_nc_y = row["BB.meanY"].strip()

        # 數字或NA都視為合理
        nc_x_ok = (
            candidate_nc_x.upper() == "NA"
            or bool(
                full_number_pattern.fullmatch(
                    candidate_nc_x
                )
            )
        )

        nc_y_ok = (
            candidate_nc_y.upper() == "NA"
            or bool(
                full_number_pattern.fullmatch(
                    candidate_nc_y
                )
            )
        )

        nc_ok = nc_x_ok and nc_y_ok

        if nc_ok:
            nc_mean_pass += 1


        # -----------------------------------
        # 候選MMD
        # -----------------------------------

        # 原始NC.meanX候選對應真正MMD
        candidate_mmd = row["NC.meanX"].strip()

        try:
            mmd_value = float(candidate_mmd)

            # 目前先以非負值作為基本合理性檢查
            mmd_ok = mmd_value >= 0

        except ValueError:
            mmd_value = None
            mmd_ok = False

        if mmd_ok:
            mmd_pass += 1


        # -----------------------------------
        # 候選MinorAlleleFrequency
        # -----------------------------------

        # 原始NC.meanY候選對應真正MAF
        candidate_maf = row["NC.meanY"].strip()

        try:
            maf_value = float(candidate_maf)

            # Minor allele frequency應介於0與0.5
            maf_ok = 0 <= maf_value <= 0.5

        except ValueError:
            maf_value = None
            maf_ok = False

        if maf_ok:
            maf_pass += 1


        # -----------------------------------
        # 候選H.W.p-Value
        # -----------------------------------

        # 原始MMD候選對應真正H.W.p-Value
        candidate_hwp = row["MMD"].strip()

        try:
            hwp_value = float(candidate_hwp)

            # p-value應介於0與1
            hwp_ok = 0 <= hwp_value <= 1

        except ValueError:
            hwp_value = None
            hwp_ok = False

        if hwp_ok:
            hwp_pass += 1


        # -----------------------------------
        # 候選H.W.chisquared.statistic
        # -----------------------------------

        # 原始MinorAlleleFrequency候選對應卡方值
        candidate_hwchi = (
            row["MinorAlleleFrequency"].strip()
        )

        # NA可以接受
        if candidate_hwchi.upper() == "NA":
            hwchi_ok = True

        else:
            try:
                hwchi_value = float(candidate_hwchi)

                # 卡方統計量應為非負數
                hwchi_ok = hwchi_value >= 0

            except ValueError:
                hwchi_ok = False

        if hwchi_ok:
            hwchi_pass += 1


        # -----------------------------------
        # 候選Call Modified
        # -----------------------------------

        # 原始H.W.p-Value候選對應Call Modified
        candidate_call_modified = (
            row["H.W.p-Value"].strip().upper()
        )

        call_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )

        if call_ok:
            call_modified_pass += 1


        # -----------------------------------
        # 綜合檢查
        # -----------------------------------

        row_all_ok = (
            nc_ok
            and mmd_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_ok
        )

        if row_all_ok:
            all_pass += 1

        else:
            # 最多保存20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_NC.meanX":
                            candidate_nc_x,

                        "candidate_NC.meanY":
                            candidate_nc_y,

                        "candidate_MMD":
                            candidate_mmd,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified
                    }
                )


        # 保存前5筆候選對應
        if len(candidate_examples) < 5:
            candidate_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "fixed_NC.meanX":
                        candidate_nc_x,

                    "fixed_NC.meanY":
                        candidate_nc_y,

                    "fixed_MMD":
                        candidate_mmd,

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    "fixed_H.W.chisquared.statistic":
                        candidate_hwchi,

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# 顯示驗證結果
print("left_shift = 2，X chromosome尾端模式B驗證")
print()

print("模式B總列數：", total_pattern_b)
print("候選NC mean合理：", nc_mean_pass)
print("候選MMD合理：", mmd_pass)
print("候選MAF合理：", maf_pass)
print("候選H.W.p-Value合理：", hwp_pass)
print("候選H.W.chisquared合理：", hwchi_pass)
print("候選Call Modified合理：", call_modified_pass)
print("所有條件同時通過：", all_pass)

print()
print("前5筆候選修正範例：")

for example in candidate_examples:
    print(example)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，X chromosome尾端模式B驗證

模式B總列數： 106
候選NC mean合理： 106
候選MMD合理： 106
候選MAF合理： 106
候選H.W.p-Value合理： 106
候選H.W.chisquared合理： 106
候選Call Modified合理： 106
所有條件同時通過： 106

前5筆候選修正範例：
{'source_row_number': '9558', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '41.438', 'fixed_MinorAlleleFrequency': '0.0184', 'fixed_H.W.p-Value': '0.0457', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '11324', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '62.254', 'fixed_MinorAlleleFrequency': '0.0215', 'fixed_H.W.p-Value': '0.0636', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '20011', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '61.486', 'fixed_MinorAlleleFrequency': '0.0215', 'fixed_H.W.p-Value': '0.0636', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '20869', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'f

In [32]:
# 載入處理 Tab 分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從 X-A 模式的亂碼字串尾端擷取 MAF
import re


# 設定修復後的 X chromosome 輸出檔案
shift2_x_fixed_path = (
    output_folder
    / "genotype_back_shift2_X_fixed.txt"
)


# 設定修復前後對照紀錄檔
shift2_x_audit_path = (
    output_folder
    / "genotype_back_shift2_X_fixed_audit.txt"
)


# 定義字串尾端數字格式
# 例如：
# '?�數??0.0123' → 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立處理結果計數器
total_x_rows = 0              # X chromosome 總列數
pattern_xa_count = 0          # X-A 模式列數
pattern_xb_count = 0          # X-B 模式列數
unrecognized_count = 0        # 無法分類的列數
qc_pass_count = 0             # 修復後通過 QC
qc_fail_count = 0             # 修復後未通過 QC


# 保存少量無法辨識或 QC 失敗的範例
problem_examples = []


# 中段欄位對應關係
# key 是修正後欄位
# value 是原始 stage1 中提供資料的欄位
middle_mapping = {
    "n_AA": "nMinorAllele",
    "n_AB": "Nclus",
    "n_BB": "n_AA",
    "n_NC": "n_AB",
    "hemizygous": "n_BB",
    "specialSNP_chr": "n_NC",
    "gender_metrics": "hemizygous",
    "ConversionType": "specialSNP_chr",
    "BestProbeset": "gender_metrics",
    "BestandRecommended": "ConversionType",
    "HomHet": "BestProbeset",
    "AA.meanX": "BestandRecommended",
    "AA.meanY": "HomHet",
    "AB.meanX": "AA.meanX",
    "AB.meanY": "AA.meanY",
    "BB.meanX": "AB.meanX",
    "BB.meanY": "AB.meanY"
}


# 開啟 stage1 檔案
# stage1 只清理 FLD 與 HomFLD，尚未移動其他欄位
with shift2_stage1_path.open(
    "r",                         # 讀取模式
    encoding="utf-8",            # 使用 UTF-8
    errors="replace",            # 無法解碼字元以替代符號保留
    newline=""
) as input_file, \
shift2_x_fixed_path.open(
    "w",                         # 建立 X chromosome 修正版
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_x_audit_path.open(
    "w",                         # 建立修復紀錄檔
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 保留原始欄位，再新增兩個修復紀錄欄位
    output_fields = (
        reader.fieldnames
        + [
            "shift2_subpattern",
            "final_qc_status"
        ]
    )


    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    # 寫入修正版表頭
    writer.writeheader()


    # 對照紀錄檔的欄位
    audit_fields = [
        "source_row_number",
        "shift2_subpattern",
        "raw_NC.meanX",
        "raw_NC.meanY",
        "raw_MMD",
        "raw_MinorAlleleFrequency",
        "raw_H.W.p-Value",
        "fixed_NC.meanX",
        "fixed_NC.meanY",
        "fixed_MMD",
        "fixed_MinorAlleleFrequency",
        "fixed_H.W.p-Value",
        "fixed_H.W.chisquared.statistic",
        "fixed_Call Modified",
        "final_qc_status"
    ]


    # 建立對照檔寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    # 寫入對照檔表頭
    audit_writer.writeheader()


    # 逐列處理 stage1 資料
    for row in reader:

        # stage1 尚未修正位移
        # 真正的 specialSNP_chr 仍放在原始 n_NC
        # 只處理 X chromosome
        if row["n_NC"] != "X":
            continue

        total_x_rows += 1


        # 完整保存修復前的資料
        # 後面所有欄位都從 raw 取值，避免互相覆蓋
        raw = row.copy()


        # 建立修正後資料
        fixed = row.copy()


        # -----------------------------------------
        # 第一部分：nMinorAllele 到 BB.meanY
        # -----------------------------------------

        # nMinorAllele 與 Nclus 沒有可靠輸出
        # 暫時設為空白
        fixed["nMinorAllele"] = ""
        fixed["Nclus"] = ""


        # 依照已驗證的對應關係移動中段欄位
        for target_field, source_field in middle_mapping.items():
            fixed[target_field] = raw[source_field]


        # -----------------------------------------
        # 第二部分：分類 X-A 或 X-B 尾端模式
        # -----------------------------------------

        raw_maf_position = (
            raw["MinorAlleleFrequency"]
            .strip()
            .upper()
        )

        raw_hwp_position = (
            raw["H.W.p-Value"]
            .strip()
            .upper()
        )


        # X-A 模式：
        # 原始 MinorAlleleFrequency 位置為 TRUE/FALSE
        if raw_maf_position in {"TRUE", "FALSE"}:

            shift2_subpattern = "X-A"

            pattern_xa_count += 1


            # 原始 BB.meanX、BB.meanY
            # 修正後對應 NC.meanX、NC.meanY
            fixed["NC.meanX"] = raw["BB.meanX"]
            fixed["NC.meanY"] = raw["BB.meanY"]


            # 此模式沒有可靠 MMD
            fixed["MMD"] = ""


            # 從原始 NC.meanX 亂碼尾端擷取 MAF
            maf_match = ending_number_pattern.search(
                raw["NC.meanX"].strip()
            )

            if maf_match:
                fixed["MinorAlleleFrequency"] = (
                    maf_match.group(1)
                )
            else:
                fixed["MinorAlleleFrequency"] = ""


            # 原始 NC.meanY 對應 H.W.p-Value
            fixed["H.W.p-Value"] = raw["NC.meanY"]


            # 原始 MMD 對應卡方統計量
            fixed["H.W.chisquared.statistic"] = (
                raw["MMD"]
            )


            # 原始 MinorAlleleFrequency 位置的布林值
            # 對應 Call Modified
            fixed["Call Modified"] = raw_maf_position


        # X-B 模式：
        # 原始 H.W.p-Value 位置為 TRUE/FALSE
        elif (
            raw["MinorAlleleFrequency"]
            .strip()
            .upper() == "NA"
            and raw_hwp_position in {"TRUE", "FALSE"}
        ):

            shift2_subpattern = "X-B"

            pattern_xb_count += 1


            # 原始 BB.meanX、BB.meanY
            # 修正後對應 NC.meanX、NC.meanY
            fixed["NC.meanX"] = raw["BB.meanX"]
            fixed["NC.meanY"] = raw["BB.meanY"]


            # 原始 NC.meanX 對應 MMD
            fixed["MMD"] = raw["NC.meanX"]


            # 原始 NC.meanY 對應 MAF
            fixed["MinorAlleleFrequency"] = (
                raw["NC.meanY"]
            )


            # 原始 MMD 對應 H.W.p-Value
            fixed["H.W.p-Value"] = raw["MMD"]


            # 原始 MinorAlleleFrequency 對應卡方值
            fixed["H.W.chisquared.statistic"] = (
                raw["MinorAlleleFrequency"]
            )


            # 原始 H.W.p-Value 的布林值
            # 對應 Call Modified
            fixed["Call Modified"] = raw_hwp_position


        # 不符合 X-A 或 X-B
        else:

            shift2_subpattern = "unrecognized"

            unrecognized_count += 1


        # -----------------------------------------
        # 第三部分：修復後 QC
        # -----------------------------------------

        try:
            # X/female 的 genotype count 應合計為 163
            batch_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            # MAF 應介於 0 到 0.5
            maf_value = float(
                fixed["MinorAlleleFrequency"]
            )

            maf_ok = 0 <= maf_value <= 0.5


            # H.W.p-Value 應介於 0 到 1
            hwp_value = float(
                fixed["H.W.p-Value"]
            )

            hwp_ok = 0 <= hwp_value <= 1


            # Call Modified 應為 TRUE 或 FALSE
            call_modified_ok = (
                fixed["Call Modified"]
                .strip()
                .upper()
                in {"TRUE", "FALSE"}
            )


            # 文字欄位與 genotype count 同時驗證
            qc_ok = (
                shift2_subpattern in {"X-A", "X-B"}
                and batch_total == 163
                and fixed["hemizygous"] in {"0", "1"}
                and fixed["specialSNP_chr"] == "X"
                and fixed["gender_metrics"] == "female"
                and fixed["ConversionType"]
                in {
                    "PolyHighResolution",
                    "NoMinorHom"
                }
                and maf_ok
                and hwp_ok
                and call_modified_ok
            )

        except (ValueError, TypeError):

            batch_total = None
            qc_ok = False


        # 設定 QC 結果
        if qc_ok:

            final_qc_status = "pass"

            qc_pass_count += 1

        else:

            final_qc_status = "fail"

            qc_fail_count += 1

            # 最多保存前20筆問題範例
            if len(problem_examples) < 20:
                problem_examples.append(
                    {
                        "source_row_number":
                            raw["source_row_number"],

                        "shift2_subpattern":
                            shift2_subpattern,

                        "batch_total":
                            batch_total,

                        "MAF":
                            fixed.get(
                                "MinorAlleleFrequency",
                                ""
                            ),

                        "H.W.p-Value":
                            fixed.get(
                                "H.W.p-Value",
                                ""
                            ),

                        "Call Modified":
                            fixed.get(
                                "Call Modified",
                                ""
                            )
                    }
                )


        # 將模式與 QC 狀態放入修正版
        fixed["shift2_subpattern"] = (
            shift2_subpattern
        )

        fixed["final_qc_status"] = (
            final_qc_status
        )


        # 寫入修正後資料
        writer.writerow(fixed)


        # 寫入修復前後對照
        audit_writer.writerow(
            {
                "source_row_number":
                    raw["source_row_number"],

                "shift2_subpattern":
                    shift2_subpattern,

                "raw_NC.meanX":
                    raw["NC.meanX"],

                "raw_NC.meanY":
                    raw["NC.meanY"],

                "raw_MMD":
                    raw["MMD"],

                "raw_MinorAlleleFrequency":
                    raw["MinorAlleleFrequency"],

                "raw_H.W.p-Value":
                    raw["H.W.p-Value"],

                "fixed_NC.meanX":
                    fixed.get("NC.meanX", ""),

                "fixed_NC.meanY":
                    fixed.get("NC.meanY", ""),

                "fixed_MMD":
                    fixed.get("MMD", ""),

                "fixed_MinorAlleleFrequency":
                    fixed.get(
                        "MinorAlleleFrequency",
                        ""
                    ),

                "fixed_H.W.p-Value":
                    fixed.get(
                        "H.W.p-Value",
                        ""
                    ),

                "fixed_H.W.chisquared.statistic":
                    fixed.get(
                        "H.W.chisquared.statistic",
                        ""
                    ),

                "fixed_Call Modified":
                    fixed.get(
                        "Call Modified",
                        ""
                    ),

                "final_qc_status":
                    final_qc_status
            }
        )


# 顯示處理結果
print("left_shift = 2，X chromosome正式修復完成")
print()

print("修正版檔案：", shift2_x_fixed_path)
print("修復對照檔：", shift2_x_audit_path)
print()

print("X chromosome總列數：", total_x_rows)
print("X-A模式：", pattern_xa_count)
print("X-B模式：", pattern_xb_count)
print("無法辨識模式：", unrecognized_count)
print("最終QC通過：", qc_pass_count)
print("最終QC未通過：", qc_fail_count)

print()
print("問題範例：")

for example in problem_examples:
    print(example)

left_shift = 2，X chromosome正式修復完成

修正版檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_X_fixed.txt
修復對照檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_X_fixed_audit.txt

X chromosome總列數： 3058
X-A模式： 2952
X-B模式： 106
無法辨識模式： 0
最終QC通過： 3058
最終QC未通過： 0

問題範例：


處理剩下238列MT

In [33]:
# 載入需要的套件
import csv
import re


# 定義完整數字格式
# 可辨認整數、小數、負數及科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端數字格式
# 例如：
# '?�數??0.0123' → 擷取 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_mt_rows = 0

# Stage1中的FLD與HomFLD是否已清理成功
fld_homfld_pass = 0

# 四個genotype count合計是否為240
genotype_count_pass = 0

# 中段文字欄位對應是否合理
structure_pass = 0

# 候選NC.meanX與NC.meanY是否為數字或NA
nc_mean_pass = 0

# 是否能從原始NC.meanX尾端擷取MAF
maf_extract_success = 0

# 候選MAF是否位於0到0.5
maf_range_pass = 0

# 候選H.W.p-Value是否位於0到1
hwp_pass = 0

# 候選H.W.chisquared是否為NA或非負數
hwchi_pass = 0

# 候選Call Modified是否為TRUE或FALSE
call_modified_pass = 0

# 所有條件同時通過
all_conditions_pass = 0


# 保存前20筆未通過範例
failed_examples = []

# 保存前5筆候選修正範例
candidate_examples = []


# 從stage1檔案讀取
# stage1只有清理FLD與HomFLD，其他欄位仍保留原始位置
with shift2_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )

    # 逐列檢查
    for row in reader:

        # stage1尚未修正中段位移
        # 原始n_NC欄位中的MT才是真正的specialSNP_chr
        if row["n_NC"] != "MT":
            continue

        total_mt_rows += 1


        # =========================================
        # 1. 檢查FLD與HomFLD
        # =========================================

        fld_ok = bool(
            full_number_pattern.fullmatch(
                row["FLD"].strip()
            )
        )

        homfld_ok = bool(
            full_number_pattern.fullmatch(
                row["HomFLD"].strip()
            )
        )

        fld_homfld_ok = fld_ok and homfld_ok

        if fld_homfld_ok:
            fld_homfld_pass += 1


        # =========================================
        # 2. 檢查中段欄位候選對應
        # =========================================

        # 原始nMinorAllele到n_AB
        # 候選對應真正的四個genotype count
        try:
            candidate_n_AA = int(
                row["nMinorAllele"]
            )

            candidate_n_AB = int(
                row["Nclus"]
            )

            candidate_n_BB = int(
                row["n_AA"]
            )

            candidate_n_NC = int(
                row["n_AB"]
            )

            genotype_total = (
                candidate_n_AA
                + candidate_n_AB
                + candidate_n_BB
                + candidate_n_NC
            )

            # MT使用all共240位樣本
            count_ok = genotype_total == 240

        except (ValueError, TypeError):
            genotype_total = None
            count_ok = False

        if count_ok:
            genotype_count_pass += 1


        # 原始欄位中的候選結構
        candidate_hemizygous = row["n_BB"].strip()
        candidate_chr = row["n_NC"].strip()
        candidate_gender = row["hemizygous"].strip()
        candidate_conversion = (
            row["specialSNP_chr"].strip()
        )

        structure_ok = (
            candidate_hemizygous in {"0", "1"}
            and candidate_chr == "MT"
            and candidate_gender == "all"
            and candidate_conversion
            in {
                "PolyHighResolution",
                "NoMinorHom"
            }
        )

        if structure_ok:
            structure_pass += 1


        # =========================================
        # 3. 檢查候選NC.meanX與NC.meanY
        # =========================================

        # 向右修正兩欄後：
        # 原始BB.meanX → 真正NC.meanX
        # 原始BB.meanY → 真正NC.meanY
        candidate_nc_x = row["BB.meanX"].strip()
        candidate_nc_y = row["BB.meanY"].strip()

        nc_x_ok = (
            candidate_nc_x.upper() == "NA"
            or bool(
                full_number_pattern.fullmatch(
                    candidate_nc_x
                )
            )
        )

        nc_y_ok = (
            candidate_nc_y.upper() == "NA"
            or bool(
                full_number_pattern.fullmatch(
                    candidate_nc_y
                )
            )
        )

        nc_ok = nc_x_ok and nc_y_ok

        if nc_ok:
            nc_mean_pass += 1


        # =========================================
        # 4. 檢查候選MinorAlleleFrequency
        # =========================================

        # MT目前看起來與X-A相同
        # 從原始NC.meanX的亂碼尾端擷取MAF
        maf_match = ending_number_pattern.search(
            row["NC.meanX"].strip()
        )

        if maf_match:

            candidate_maf = maf_match.group(1)

            maf_extract_success += 1

            try:
                maf_value = float(candidate_maf)

                # MAF應介於0到0.5
                maf_ok = 0 <= maf_value <= 0.5

            except ValueError:
                maf_value = None
                maf_ok = False

        else:
            candidate_maf = ""
            maf_value = None
            maf_ok = False

        if maf_ok:
            maf_range_pass += 1


        # =========================================
        # 5. 檢查候選H.W.p-Value
        # =========================================

        # 原始NC.meanY候選對應真正H.W.p-Value
        candidate_hwp = row["NC.meanY"].strip()

        try:
            hwp_value = float(candidate_hwp)

            hwp_ok = 0 <= hwp_value <= 1

        except ValueError:
            hwp_value = None
            hwp_ok = False

        if hwp_ok:
            hwp_pass += 1


        # =========================================
        # 6. 檢查候選H.W.chisquared
        # =========================================

        # 原始MMD候選對應真正H.W.chisquared
        candidate_hwchi = row["MMD"].strip()

        if candidate_hwchi.upper() == "NA":

            hwchi_ok = True

        else:

            try:
                hwchi_value = float(candidate_hwchi)

                hwchi_ok = hwchi_value >= 0

            except ValueError:
                hwchi_ok = False

        if hwchi_ok:
            hwchi_pass += 1


        # =========================================
        # 7. 檢查候選Call Modified
        # =========================================

        # 原始MinorAlleleFrequency位置中的布林值
        # 候選對應真正Call Modified
        candidate_call_modified = (
            row["MinorAlleleFrequency"]
            .strip()
            .upper()
        )

        call_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )

        if call_ok:
            call_modified_pass += 1


        # =========================================
        # 8. 綜合判斷
        # =========================================

        all_ok = (
            fld_homfld_ok
            and count_ok
            and structure_ok
            and nc_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_ok
        )

        if all_ok:

            all_conditions_pass += 1

        else:

            # 最多保存前20筆未通過範例
            if len(failed_examples) < 20:
                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "FLD":
                            row["FLD"],

                        "HomFLD":
                            row["HomFLD"],

                        "genotype_total":
                            genotype_total,

                        "candidate_chr":
                            candidate_chr,

                        "candidate_gender":
                            candidate_gender,

                        "candidate_NC.meanX":
                            candidate_nc_x,

                        "candidate_NC.meanY":
                            candidate_nc_y,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified
                    }
                )


        # 保存前5筆候選修正結果
        if len(candidate_examples) < 5:
            candidate_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "fixed_NC.meanX":
                        candidate_nc_x,

                    "fixed_NC.meanY":
                        candidate_nc_y,

                    "fixed_MMD":
                        "",

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    "fixed_H.W.chisquared.statistic":
                        candidate_hwchi,

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# 顯示驗證結果
print("left_shift = 2，MT尾端候選對應驗證")
print()

print("MT資料總列數：", total_mt_rows)
print("FLD與HomFLD為純數字：", fld_homfld_pass)
print("genotype count合計為240：", genotype_count_pass)
print("中段文字結構合理：", structure_pass)
print("候選NC mean合理：", nc_mean_pass)
print("MAF尾端數值擷取成功：", maf_extract_success)
print("候選MAF值域合理：", maf_range_pass)
print("候選H.W.p-Value合理：", hwp_pass)
print("候選H.W.chisquared合理：", hwchi_pass)
print("候選Call Modified合理：", call_modified_pass)
print("所有條件同時通過：", all_conditions_pass)

print()
print("前5筆候選修正範例：")

for example in candidate_examples:
    print(example)

print()
print("未通過範例：")

for example in failed_examples:
    print(example)

left_shift = 2，MT尾端候選對應驗證

MT資料總列數： 238
FLD與HomFLD為純數字： 238
genotype count合計為240： 238
中段文字結構合理： 238
候選NC mean合理： 238
MAF尾端數值擷取成功： 238
候選MAF值域合理： 238
候選H.W.p-Value合理： 238
候選H.W.chisquared合理： 238
候選Call Modified合理： 238
所有條件同時通過： 238

前5筆候選修正範例：
{'source_row_number': '843', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0042', 'fixed_H.W.p-Value': '0.00209', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '4833', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0167', 'fixed_H.W.p-Value': '2.04E-09', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '5335', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0.0084', 'fixed_H.W.p-Value': '1.32E-05', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '7843', 'fixed_NC

In [34]:
# 載入需要的套件
import csv
import re


# 設定MT修正版輸出位置
shift2_mt_fixed_path = (
    output_folder
    / "genotype_back_shift2_MT_fixed.txt"
)


# 設定MT修復對照檔
shift2_mt_audit_path = (
    output_folder
    / "genotype_back_shift2_MT_fixed_audit.txt"
)


# 定義完整數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端數字格式
# 例如：
# '?�數??0.0123' → 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 中段欄位修正對應
# 修正後欄位：stage1中目前存放該值的欄位
middle_mapping = {
    "n_AA": "nMinorAllele",
    "n_AB": "Nclus",
    "n_BB": "n_AA",
    "n_NC": "n_AB",
    "hemizygous": "n_BB",
    "specialSNP_chr": "n_NC",
    "gender_metrics": "hemizygous",
    "ConversionType": "specialSNP_chr",
    "BestProbeset": "gender_metrics",
    "BestandRecommended": "ConversionType",
    "HomHet": "BestProbeset",
    "AA.meanX": "BestandRecommended",
    "AA.meanY": "HomHet",
    "AB.meanX": "AA.meanX",
    "AB.meanY": "AA.meanY",
    "BB.meanX": "AB.meanX",
    "BB.meanY": "AB.meanY"
}


# 建立計數器
total_mt_rows = 0
repair_success_count = 0
repair_failed_count = 0

# 保存前20筆問題資料
problem_examples = []


# 從stage1安全基準檔案開始
with shift2_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift2_mt_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift2_mt_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 讀取Tab分隔資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 修正版保留原始欄位
    # 另外新增子模式與QC結果
    output_fields = (
        reader.fieldnames
        + [
            "shift2_subpattern",
            "final_qc_status"
        ]
    )


    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 修復對照檔欄位
    audit_fields = [
        "source_row_number",
        "shift2_subpattern",

        "raw_FLD",
        "raw_HomFLD",

        "raw_NC.meanX",
        "raw_NC.meanY",
        "raw_MMD",
        "raw_MinorAlleleFrequency",

        "fixed_n_AA",
        "fixed_n_AB",
        "fixed_n_BB",
        "fixed_n_NC",

        "fixed_NC.meanX",
        "fixed_NC.meanY",
        "fixed_MMD",
        "fixed_MinorAlleleFrequency",
        "fixed_H.W.p-Value",
        "fixed_H.W.chisquared.statistic",
        "fixed_Call Modified",

        "genotype_total",
        "final_qc_status"
    ]


    # 建立對照檔寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # 逐列處理
    for row in reader:

        # stage1尚未移動中段欄位
        # 真正的specialSNP_chr仍位於原始n_NC
        if row["n_NC"] != "MT":
            continue

        total_mt_rows += 1


        # 完整保存原始資料
        # 避免修正時覆蓋後面仍需要使用的值
        raw = row.copy()

        # 建立修正版
        fixed = row.copy()


        # =========================================
        # 1. 修復中段欄位
        # =========================================

        # nMinorAllele與Nclus沒有可靠的原始資料
        # 暫時設為空白
        fixed["nMinorAllele"] = ""
        fixed["Nclus"] = ""


        # 將原本提前兩格的資料移回正確欄位
        for target_field, source_field in middle_mapping.items():
            fixed[target_field] = raw[source_field]


        # =========================================
        # 2. 修復MT尾端欄位
        # =========================================

        # MT使用與X-A相同的尾端模式
        shift2_subpattern = "MT-A"


        # 原始BB.meanX、BB.meanY
        # 修正後對應NC.meanX、NC.meanY
        fixed["NC.meanX"] = raw["BB.meanX"]
        fixed["NC.meanY"] = raw["BB.meanY"]


        # 此模式沒有可靠的MMD輸出
        fixed["MMD"] = ""


        # 從原始NC.meanX亂碼字串尾端擷取MAF
        maf_match = ending_number_pattern.search(
            raw["NC.meanX"].strip()
        )

        if maf_match:
            fixed["MinorAlleleFrequency"] = (
                maf_match.group(1)
            )
        else:
            fixed["MinorAlleleFrequency"] = ""


        # 原始NC.meanY對應真正H.W.p-Value
        fixed["H.W.p-Value"] = raw["NC.meanY"]


        # 原始MMD對應真正H.W.chisquared
        fixed["H.W.chisquared.statistic"] = (
            raw["MMD"]
        )


        # 原始MinorAlleleFrequency欄位中的FALSE
        # 對應真正Call Modified
        fixed["Call Modified"] = (
            raw["MinorAlleleFrequency"]
            .strip()
            .upper()
        )


        # =========================================
        # 3. 修復後QC
        # =========================================

        try:
            # MT資料使用all，共240位樣本
            genotype_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            # FLD必須為純數字
            fld_ok = bool(
                full_number_pattern.fullmatch(
                    fixed["FLD"].strip()
                )
            )


            # HomFLD必須為純數字
            homfld_ok = bool(
                full_number_pattern.fullmatch(
                    fixed["HomFLD"].strip()
                )
            )


            # MAF必須介於0至0.5
            maf_value = float(
                fixed["MinorAlleleFrequency"]
            )

            maf_ok = 0 <= maf_value <= 0.5


            # HWE p-value必須介於0至1
            hwp_value = float(
                fixed["H.W.p-Value"]
            )

            hwp_ok = 0 <= hwp_value <= 1


            # 卡方統計量可為NA或非負數
            hwchi_value = (
                fixed["H.W.chisquared.statistic"]
                .strip()
            )

            if hwchi_value.upper() == "NA":
                hwchi_ok = True

            else:
                hwchi_ok = float(hwchi_value) >= 0


            # Call Modified必須為布林值
            call_modified_ok = (
                fixed["Call Modified"]
                in {"TRUE", "FALSE"}
            )


            # 綜合檢查
            qc_ok = (
                genotype_total == 240
                and fld_ok
                and homfld_ok
                and fixed["hemizygous"] in {"0", "1"}
                and fixed["specialSNP_chr"] == "MT"
                and fixed["gender_metrics"] == "all"
                and fixed["ConversionType"]
                in {
                    "PolyHighResolution",
                    "NoMinorHom"
                }
                and fixed["BestProbeset"] in {"0", "1"}
                and fixed["BestandRecommended"] in {"0", "1"}
                and fixed["HomHet"] in {"0", "1"}
                and maf_ok
                and hwp_ok
                and hwchi_ok
                and call_modified_ok
            )

        except (ValueError, TypeError):

            genotype_total = None
            qc_ok = False


        # =========================================
        # 4. 記錄QC結果
        # =========================================

        if qc_ok:

            final_qc_status = "pass"
            repair_success_count += 1

        else:

            final_qc_status = "fail"
            repair_failed_count += 1

            # 保存前20筆問題資料
            if len(problem_examples) < 20:
                problem_examples.append(
                    {
                        "source_row_number":
                            raw["source_row_number"],

                        "genotype_total":
                            genotype_total,

                        "FLD":
                            fixed["FLD"],

                        "HomFLD":
                            fixed["HomFLD"],

                        "specialSNP_chr":
                            fixed["specialSNP_chr"],

                        "gender_metrics":
                            fixed["gender_metrics"],

                        "MAF":
                            fixed["MinorAlleleFrequency"],

                        "H.W.p-Value":
                            fixed["H.W.p-Value"],

                        "H.W.chisquared":
                            fixed[
                                "H.W.chisquared.statistic"
                            ],

                        "Call Modified":
                            fixed["Call Modified"]
                    }
                )


        # 新增模式與QC結果
        fixed["shift2_subpattern"] = shift2_subpattern
        fixed["final_qc_status"] = final_qc_status


        # 寫入MT修正版
        writer.writerow(fixed)


        # 寫入修復前後對照
        audit_writer.writerow(
            {
                "source_row_number":
                    raw["source_row_number"],

                "shift2_subpattern":
                    shift2_subpattern,

                "raw_FLD":
                    raw["FLD"],

                "raw_HomFLD":
                    raw["HomFLD"],

                "raw_NC.meanX":
                    raw["NC.meanX"],

                "raw_NC.meanY":
                    raw["NC.meanY"],

                "raw_MMD":
                    raw["MMD"],

                "raw_MinorAlleleFrequency":
                    raw["MinorAlleleFrequency"],

                "fixed_n_AA":
                    fixed["n_AA"],

                "fixed_n_AB":
                    fixed["n_AB"],

                "fixed_n_BB":
                    fixed["n_BB"],

                "fixed_n_NC":
                    fixed["n_NC"],

                "fixed_NC.meanX":
                    fixed["NC.meanX"],

                "fixed_NC.meanY":
                    fixed["NC.meanY"],

                "fixed_MMD":
                    fixed["MMD"],

                "fixed_MinorAlleleFrequency":
                    fixed["MinorAlleleFrequency"],

                "fixed_H.W.p-Value":
                    fixed["H.W.p-Value"],

                "fixed_H.W.chisquared.statistic":
                    fixed[
                        "H.W.chisquared.statistic"
                    ],

                "fixed_Call Modified":
                    fixed["Call Modified"],

                "genotype_total":
                    genotype_total,

                "final_qc_status":
                    final_qc_status
            }
        )


# 顯示修復結果
print("left_shift = 2，MT正式修復完成")
print()

print("MT修正版檔案：", shift2_mt_fixed_path)
print("MT修復對照檔：", shift2_mt_audit_path)
print()

print("MT資料總列數：", total_mt_rows)
print("修復後QC通過：", repair_success_count)
print("修復後QC未通過：", repair_failed_count)

print()
print("問題範例：")

for example in problem_examples:
    print(example)

left_shift = 2，MT正式修復完成

MT修正版檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_MT_fixed.txt
MT修復對照檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_MT_fixed_audit.txt

MT資料總列數： 238
修復後QC通過： 238
修復後QC未通過： 0

問題範例：


原本把X和MT分開處理
現在要合併

In [35]:
# 載入需要的套件
import csv
from collections import Counter


# 設定合併後的輸出檔案
shift2_all_fixed_path = (
    output_folder
    / "genotype_back_shift2_all_fixed.txt"
)


# 建立清單，用來保存X與MT的修復後資料
all_fixed_rows = []


# 指定要合併的兩個檔案
shift2_fixed_files = [
    shift2_x_fixed_path,
    shift2_mt_fixed_path
]


# 保存兩個檔案的表頭
reference_fieldnames = None


# 逐一讀取X與MT修正版
for fixed_path in shift2_fixed_files:

    with fixed_path.open(
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as file:

        # 使用欄位名稱讀取Tab分隔資料
        reader = csv.DictReader(
            file,
            delimiter="\t",
            quoting=csv.QUOTE_NONE
        )

        # 第一個檔案的表頭作為標準
        if reference_fieldnames is None:

            reference_fieldnames = reader.fieldnames

        # 確認第二個檔案的欄位完全相同
        elif reader.fieldnames != reference_fieldnames:

            raise ValueError(
                f"欄位不一致：{fixed_path}"
            )

        # 將資料加入合併清單
        for row in reader:
            all_fixed_rows.append(row)


# 依照原始source_row_number排序
# 讓X與MT資料回到原始TXT中的順序
all_fixed_rows.sort(
    key=lambda row: int(
        row["source_row_number"]
    )
)


# 建立驗證計數器
total_rows = len(all_fixed_rows)

# 統計不同修復子模式
subpattern_counts = Counter(
    row["shift2_subpattern"]
    for row in all_fixed_rows
)

# 統計QC狀態
qc_status_counts = Counter(
    row["final_qc_status"]
    for row in all_fixed_rows
)

# 統計left_shift
left_shift_counts = Counter(
    row["left_shift"]
    for row in all_fixed_rows
)

# 取得所有source_row_number
source_row_numbers = [
    int(row["source_row_number"])
    for row in all_fixed_rows
]

# 檢查是否有重複的source_row_number
duplicate_source_rows = [
    source_row_number
    for source_row_number, count
    in Counter(source_row_numbers).items()
    if count > 1
]


# 建立最終問題清單
problem_examples = []


# 逐列做合併後的基本檢查
for row in all_fixed_rows:

    problem_reasons = []

    # 必須全部是left_shift = 2
    if row["left_shift"] != "2":
        problem_reasons.append(
            "left_shift不是2"
        )

    # 必須通過先前的正式QC
    if row["final_qc_status"] != "pass":
        problem_reasons.append(
            "final_qc_status不是pass"
        )

    # 子模式只能是三種
    if row["shift2_subpattern"] not in {
        "X-A",
        "X-B",
        "MT-A"
    }:
        problem_reasons.append(
            "無法辨識的shift2_subpattern"
        )

    # X資料的結構檢查
    if row["shift2_subpattern"] in {
        "X-A",
        "X-B"
    }:

        if row["specialSNP_chr"] != "X":
            problem_reasons.append(
                "X模式但specialSNP_chr不是X"
            )

        if row["gender_metrics"] != "female":
            problem_reasons.append(
                "X模式但gender_metrics不是female"
            )

        expected_genotype_total = 163

    # MT資料的結構檢查
    elif row["shift2_subpattern"] == "MT-A":

        if row["specialSNP_chr"] != "MT":
            problem_reasons.append(
                "MT模式但specialSNP_chr不是MT"
            )

        if row["gender_metrics"] != "all":
            problem_reasons.append(
                "MT模式但gender_metrics不是all"
            )

        expected_genotype_total = 240

    else:
        expected_genotype_total = None


    # 檢查四個genotype count
    try:
        genotype_total = (
            int(row["n_AA"])
            + int(row["n_AB"])
            + int(row["n_BB"])
            + int(row["n_NC"])
        )

        if (
            expected_genotype_total is not None
            and genotype_total
            != expected_genotype_total
        ):
            problem_reasons.append(
                "genotype count合計錯誤"
            )

    except (ValueError, TypeError):

        genotype_total = None

        problem_reasons.append(
            "genotype count無法轉為整數"
        )


    # 保存前20筆有問題的資料
    if (
        problem_reasons
        and len(problem_examples) < 20
    ):
        problem_examples.append(
            {
                "source_row_number":
                    row["source_row_number"],

                "shift2_subpattern":
                    row["shift2_subpattern"],

                "specialSNP_chr":
                    row["specialSNP_chr"],

                "gender_metrics":
                    row["gender_metrics"],

                "genotype_total":
                    genotype_total,

                "problem_reasons":
                    problem_reasons
            }
        )


# 將合併且排序後的資料寫入新檔案
with shift2_all_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file:

    writer = csv.DictWriter(
        output_file,
        fieldnames=reference_fieldnames,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    # 寫入表頭
    writer.writeheader()

    # 寫入全部3,296列
    writer.writerows(all_fixed_rows)


# 顯示合併結果
print("left_shift = 2 合併完成")
print()

print("完整修正版：", shift2_all_fixed_path)
print()

print("合併後總列數：", total_rows)
print("left_shift分布：", dict(left_shift_counts))
print("子模式分布：", dict(subpattern_counts))
print("QC狀態分布：", dict(qc_status_counts))
print("重複source_row_number：", len(duplicate_source_rows))
print("合併後問題列數：", len(problem_examples))

print()
print("問題範例：")

for example in problem_examples:
    print(example)

left_shift = 2 合併完成

完整修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_all_fixed.txt

合併後總列數： 3296
left_shift分布： {'2': 3296}
子模式分布： {'X-A': 2952, 'MT-A': 238, 'X-B': 106}
QC狀態分布： {'pass': 3296}
重複source_row_number： 0
合併後問題列數： 0

問題範例：


看左移3個

In [36]:
# 載入需要的套件
import csv
import re
from collections import Counter, defaultdict


# 設定分析報告輸出位置
shift3_structure_report_path = (
    output_folder
    / "shift3_structure_analysis.txt"
)


# 定義完整數字格式
# 可辨認整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端數字格式
# 用來測試亂碼字串尾端是否仍保留可用數值
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 判斷欄位值的型態
def classify_value(value):

    if value is None:
        return "missing"

    value = value.strip()

    if value == "":
        return "empty"

    if value.upper() == "NA":
        return "NA"

    if value.upper() in {"TRUE", "FALSE"}:
        return "boolean"

    if full_number_pattern.fullmatch(value):
        return "numeric"

    if "�" in value or "?" in value:
        return "garbled"

    return "text"


# 從純數字或亂碼尾端擷取數字
def extract_numeric_tail(value):

    if value is None:
        return None

    value = value.strip()

    # 本身就是純數字
    if full_number_pattern.fullmatch(value):
        return value

    # 嘗試從字串尾端擷取數字
    match = ending_number_pattern.search(value)

    if match:
        return match.group(1)

    return None


# 預期群組樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的文字類別
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}

valid_gender_values = {
    "all",
    "female",
    "male"
}

valid_conversion_values = {
    "PolyHighResolution",
    "NoMinorHom",
    "MonoHighResolution",
    "CallRateBelowThreshold",
    "OffTargetVariant"
}


# 要進行型態摘要的欄位
summary_columns = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet"
]


# 每個欄位的資料型態統計
column_type_counts = {
    column: Counter()
    for column in summary_columns
}


# 每個欄位的常見值統計
column_value_counts = {
    column: Counter()
    for column in summary_columns
}


# 統計候選結構組合
candidate_group_counts = Counter()

# 統計候選 genotype count 合計
candidate_total_counts = Counter()

# 統計候選結構是否通過
structure_pass_count = 0
count_pass_count = 0
all_pass_count = 0

# HomRO 是否能取得候選 n_AA
homro_extract_success = 0

# shift3總列數
total_shift3_rows = 0

# 保存前20筆未通過範例
failed_examples = []


# 開啟加入left_shift分類的後段資料
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 逐列分析
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只分析left_shift = 3
        if row["left_shift"] != "3":
            continue

        total_shift3_rows += 1


        # =========================================
        # 1. 統計各欄位資料型態與常見值
        # =========================================

        for column in summary_columns:

            value = row[column]

            value_type = classify_value(value)

            column_type_counts[column][
                value_type
            ] += 1

            column_value_counts[column][
                value
            ] += 1


        # =========================================
        # 2. 測試整體提前3格的候選對應
        # =========================================

        # 若真正欄位整體提前3格：
        #
        # 真正n_AA       ← 目前HomRO
        # 真正n_AB       ← 目前nMinorAllele
        # 真正n_BB       ← 目前Nclus
        # 真正n_NC       ← 目前n_AA
        # 真正hemizygous ← 目前n_AB
        # 真正chr        ← 目前n_BB
        # 真正gender     ← 目前n_NC
        # 真正conversion ← 目前hemizygous

        candidate_n_AA_text = extract_numeric_tail(
            row["HomRO"]
        )

        candidate_n_AB_text = extract_numeric_tail(
            row["nMinorAllele"]
        )

        candidate_n_BB_text = extract_numeric_tail(
            row["Nclus"]
        )

        candidate_n_NC_text = extract_numeric_tail(
            row["n_AA"]
        )

        candidate_hemizygous = (
            row["n_AB"].strip()
        )

        candidate_chr = (
            row["n_BB"].strip()
        )

        candidate_gender = (
            row["n_NC"].strip()
        )

        candidate_conversion = (
            row["hemizygous"].strip()
        )

        candidate_best_probeset = (
            row["specialSNP_chr"].strip()
        )

        candidate_best_recommended = (
            row["gender_metrics"].strip()
        )

        candidate_homhet = (
            row["ConversionType"].strip()
        )


        # HomRO成功取得候選n_AA
        if candidate_n_AA_text is not None:
            homro_extract_success += 1


        # =========================================
        # 3. 檢查文字欄位結構
        # =========================================

        structure_ok = (
            candidate_hemizygous in {"0", "1"}
            and candidate_chr in valid_chr_values
            and candidate_gender in valid_gender_values
            and candidate_conversion
            in valid_conversion_values
            and candidate_best_probeset in {"0", "1"}
            and candidate_best_recommended in {"0", "1"}
            and candidate_homhet in {"0", "1"}
        )

        if structure_ok:
            structure_pass_count += 1


        # 統計候選群組
        candidate_group_counts[
            (
                candidate_chr,
                candidate_gender,
                candidate_conversion,
                candidate_hemizygous
            )
        ] += 1


        # =========================================
        # 4. 檢查候選genotype count
        # =========================================

        try:
            candidate_n_AA = int(
                candidate_n_AA_text
            )

            candidate_n_AB = int(
                candidate_n_AB_text
            )

            candidate_n_BB = int(
                candidate_n_BB_text
            )

            candidate_n_NC = int(
                candidate_n_NC_text
            )


            # 計算四個genotype count合計
            genotype_total = (
                candidate_n_AA
                + candidate_n_AB
                + candidate_n_BB
                + candidate_n_NC
            )


            # 根據候選gender取得預期樣本數
            expected_total = (
                expected_total_by_gender.get(
                    candidate_gender
                )
            )


            # 實際總數符合群組預期值
            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (
            ValueError,
            TypeError
        ):

            genotype_total = None

            expected_total = (
                expected_total_by_gender.get(
                    candidate_gender
                )
            )

            count_ok = False


        # 統計候選count合計
        candidate_total_counts[
            genotype_total
        ] += 1


        if count_ok:
            count_pass_count += 1


        # 結構與count同時通過
        row_all_ok = (
            structure_ok
            and count_ok
        )

        if row_all_ok:
            all_pass_count += 1

        else:

            # 最多保存前20筆問題範例
            if len(failed_examples) < 20:

                failed_examples.append(
                    {
                        "source_row_number":
                            source_row_number,

                        "FLD":
                            row["FLD"],

                        "HomFLD":
                            row["HomFLD"],

                        "HetSO":
                            row["HetSO"],

                        "HomRO":
                            row["HomRO"],

                        "candidate_n_AA":
                            candidate_n_AA_text,

                        "candidate_n_AB":
                            candidate_n_AB_text,

                        "candidate_n_BB":
                            candidate_n_BB_text,

                        "candidate_n_NC":
                            candidate_n_NC_text,

                        "genotype_total":
                            genotype_total,

                        "expected_total":
                            expected_total,

                        "candidate_hemizygous":
                            candidate_hemizygous,

                        "candidate_chr":
                            candidate_chr,

                        "candidate_gender":
                            candidate_gender,

                        "candidate_conversion":
                            candidate_conversion,

                        "candidate_BestProbeset":
                            candidate_best_probeset,

                        "candidate_BestandRecommended":
                            candidate_best_recommended,

                        "candidate_HomHet":
                            candidate_homhet
                    }
                )


# =========================================
# 5. 將完整分析結果寫入報告
# =========================================

with shift3_structure_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "left_shift = 3 structure analysis\n"
    )

    report.write(
        f"Total rows: {total_shift3_rows}\n\n"
    )


    # 寫入整體候選驗證結果
    report.write("=" * 80 + "\n")
    report.write("Candidate shift-by-3 validation\n")
    report.write("=" * 80 + "\n")

    report.write(
        f"HomRO numeric extraction success: "
        f"{homro_extract_success}\n"
    )

    report.write(
        f"Structure pass: "
        f"{structure_pass_count}\n"
    )

    report.write(
        f"Genotype count pass: "
        f"{count_pass_count}\n"
    )

    report.write(
        f"All conditions pass: "
        f"{all_pass_count}\n\n"
    )


    # 寫入每個欄位的型態與常見值
    report.write("=" * 80 + "\n")
    report.write("Column summaries\n")
    report.write("=" * 80 + "\n\n")

    for column in summary_columns:

        report.write(
            f"Column: {column}\n"
        )

        report.write(
            f"Type counts: "
            f"{dict(column_type_counts[column])}\n"
        )

        report.write(
            "Top values:\n"
        )

        for value, count in (
            column_value_counts[column]
            .most_common(15)
        ):
            report.write(
                f"  {repr(value)}: {count}\n"
            )

        report.write("\n")


    # 寫入候選群組
    report.write("=" * 80 + "\n")
    report.write("Candidate structural groups\n")
    report.write("=" * 80 + "\n\n")

    for group, count in (
        candidate_group_counts.most_common()
    ):

        report.write(
            f"chr={group[0]} | "
            f"gender={group[1]} | "
            f"ConversionType={group[2]} | "
            f"hemizygous={group[3]} | "
            f"count={count}\n"
        )


    # 寫入候選count總和分布
    report.write("\n" + "=" * 80 + "\n")
    report.write("Candidate genotype totals\n")
    report.write("=" * 80 + "\n\n")

    for genotype_total, count in (
        candidate_total_counts.most_common()
    ):
        report.write(
            f"total={genotype_total} | "
            f"count={count}\n"
        )


    # 寫入失敗範例
    report.write("\n" + "=" * 80 + "\n")
    report.write("Failed examples\n")
    report.write("=" * 80 + "\n\n")

    for example in failed_examples:
        report.write(
            repr(example) + "\n"
        )


# =========================================
# 6. 顯示重要結果
# =========================================

print("left_shift = 3 結構分析完成")
print()

print(
    "分析報告：",
    shift3_structure_report_path
)

print()
print(
    "shift3總列數：",
    total_shift3_rows
)

print(
    "HomRO可取得候選n_AA：",
    homro_extract_success
)

print(
    "候選文字結構合理：",
    structure_pass_count
)

print(
    "候選genotype count合理：",
    count_pass_count
)

print(
    "所有條件同時通過：",
    all_pass_count
)


print()
print("候選genotype count合計分布：")

for genotype_total, count in (
    candidate_total_counts.most_common(10)
):
    print(
        "合計 =",
        genotype_total,
        "| 列數 =",
        count
    )


print()
print("主要候選結構組合：")

for group, count in (
    candidate_group_counts.most_common(10)
):
    print(
        "chr =",
        group[0],
        "| gender =",
        group[1],
        "| ConversionType =",
        group[2],
        "| hemizygous =",
        group[3],
        "| 列數 =",
        count
    )


print()
print("前5筆未通過範例：")

for example in failed_examples[:5]:
    print(example)

left_shift = 3 結構分析完成

分析報告： /Users/sheena/Desktop/summerintern/split_data/shift3_structure_analysis.txt

shift3總列數： 155685
HomRO可取得候選n_AA： 155685
候選文字結構合理： 155685
候選genotype count合理： 155685
所有條件同時通過： 155685

候選genotype count合計分布：
合計 = 240 | 列數 = 132184
合計 = 77 | 列數 = 13418
合計 = 163 | 列數 = 10083

主要候選結構組合：
chr = autosomal | gender = all | ConversionType = MonoHighResolution | hemizygous = 0 | 列數 = 131659
chr = Y | gender = male | ConversionType = MonoHighResolution | hemizygous = 1 | 列數 = 12292
chr = X | gender = female | ConversionType = MonoHighResolution | hemizygous = 0 | 列數 = 9776
chr = Y | gender = male | ConversionType = PolyHighResolution | hemizygous = 1 | 列數 = 1126
chr = MT | gender = all | ConversionType = MonoHighResolution | hemizygous = 1 | 列數 = 525
chr = X | gender = female | ConversionType = PolyHighResolution | hemizygous = 0 | 列數 = 307

前5筆未通過範例：


發現從HomRO開使位移
但還是要先確認是否有其他欄位有不同位移

In [37]:
# 載入需要的套件
import csv
import re
from collections import Counter


# 設定分析報告的輸出位置
shift3_metric_report_path = (
    output_folder
    / "shift3_metric_block_analysis.txt"
)


# 要分析的前段QC欄位
metric_columns = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO"
]


# 定義完整純數字格式
# 可辨認：
# 1
# 0.032
# -2.35
# 1.2E-05
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義任意位置的數字格式
# 用來找出亂碼字串中所有可能的數值
number_token_pattern = re.compile(
    r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?"
)


# 判斷欄位值的基本型態
def classify_metric_value(value):

    # 處理None
    if value is None:
        return "missing"

    # 去除前後空白
    value = value.strip()

    # 空白
    if value == "":
        return "empty"

    # NA
    if value.upper() == "NA":
        return "NA"

    # 完整純數字
    if full_number_pattern.fullmatch(value):
        return "numeric"

    # 含亂碼符號
    if "�" in value or "?" in value:
        return "garbled"

    # 其他文字
    return "text"


# 取得字串中的全部數字
def extract_all_numeric_tokens(value):

    # 處理None
    if value is None:
        return []

    # 找出所有數字並回傳
    return [
        match.group(0)
        for match in number_token_pattern.finditer(
            value.strip()
        )
    ]


# 統計每個欄位的資料型態
column_type_counts = {
    column: Counter()
    for column in metric_columns
}


# 統計每個欄位內含幾個數字
column_token_count_distribution = {
    column: Counter()
    for column in metric_columns
}


# 統計四個欄位合併後的型態模式
type_pattern_counts = Counter()


# 統計四個欄位內數字數量的模式
token_pattern_counts = Counter()


# 保存每種模式前5筆範例
pattern_examples = {}


# 記錄總列數
total_shift3_rows = 0


# 開啟原始分類後的後段資料
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 逐列分析
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只處理left_shift = 3
        if row["left_shift"] != "3":
            continue

        total_shift3_rows += 1


        # 儲存這一列四個欄位的型態
        row_types = []


        # 儲存這一列四個欄位中的數字數量
        row_token_counts = []


        # 儲存這一列每欄擷取出的數字
        row_numeric_tokens = {}


        # 逐一分析四個QC欄位
        for column in metric_columns:

            # 取得原始值
            value = row[column]


            # 判斷資料型態
            value_type = classify_metric_value(
                value
            )


            # 找出欄位中的全部數字
            numeric_tokens = (
                extract_all_numeric_tokens(value)
            )


            # 此欄位含有幾個數字
            token_count = len(numeric_tokens)


            # 累計欄位型態
            column_type_counts[column][
                value_type
            ] += 1


            # 累計欄位中的數字數量
            column_token_count_distribution[
                column
            ][token_count] += 1


            # 放入本列型態模式
            row_types.append(value_type)


            # 放入本列數字數量模式
            row_token_counts.append(token_count)


            # 保存擷取出的實際數字
            row_numeric_tokens[column] = (
                numeric_tokens
            )


        # 建立型態模式
        type_signature = tuple(row_types)


        # 建立數字數量模式
        token_signature = tuple(
            row_token_counts
        )


        # 統計兩種模式
        type_pattern_counts[
            type_signature
        ] += 1

        token_pattern_counts[
            token_signature
        ] += 1


        # 將兩種模式合併成範例索引
        combined_pattern = (
            type_signature,
            token_signature
        )


        # 第一次出現此模式時建立空清單
        if combined_pattern not in pattern_examples:
            pattern_examples[
                combined_pattern
            ] = []


        # 每種模式最多保存前5筆
        if (
            len(
                pattern_examples[
                    combined_pattern
                ]
            ) < 5
        ):

            pattern_examples[
                combined_pattern
            ].append(
                {
                    "source_row_number":
                        source_row_number,

                    "raw_values": {
                        column: row[column]
                        for column in metric_columns
                    },

                    "numeric_tokens":
                        row_numeric_tokens,

                    # 目前HomRO尾端數值是候選n_AA
                    "candidate_n_AA":
                        (
                            row_numeric_tokens["HomRO"][-1]
                            if row_numeric_tokens["HomRO"]
                            else None
                        )
                }
            )


# 將完整結果寫入報告檔
with shift3_metric_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "left_shift = 3 metric block analysis\n"
    )

    report.write(
        f"Total rows: {total_shift3_rows}\n\n"
    )


    # 每個欄位的型態分布
    report.write("=" * 80 + "\n")
    report.write("Column type distributions\n")
    report.write("=" * 80 + "\n\n")

    for column in metric_columns:

        report.write(
            f"Column: {column}\n"
        )

        report.write(
            f"Type counts: "
            f"{dict(column_type_counts[column])}\n"
        )

        report.write(
            f"Numeric-token counts: "
            f"{dict(column_token_count_distribution[column])}\n\n"
        )


    # 四欄合併型態模式
    report.write("=" * 80 + "\n")
    report.write("Combined type patterns\n")
    report.write("=" * 80 + "\n\n")

    for pattern, count in (
        type_pattern_counts.most_common()
    ):

        report.write(
            f"pattern={pattern} | count={count}\n"
        )


    # 四欄數字數量模式
    report.write("\n" + "=" * 80 + "\n")
    report.write("Numeric-token patterns\n")
    report.write("=" * 80 + "\n\n")

    for pattern, count in (
        token_pattern_counts.most_common()
    ):

        report.write(
            f"pattern={pattern} | count={count}\n"
        )


    # 實際範例
    report.write("\n" + "=" * 80 + "\n")
    report.write("Pattern examples\n")
    report.write("=" * 80 + "\n\n")

    for combined_pattern, examples in (
        pattern_examples.items()
    ):

        report.write(
            f"Type pattern: "
            f"{combined_pattern[0]}\n"
        )

        report.write(
            f"Token pattern: "
            f"{combined_pattern[1]}\n"
        )

        for example in examples:

            report.write(
                f"source_row_number: "
                f"{example['source_row_number']}\n"
            )

            for column in metric_columns:

                report.write(
                    f"  {column}: "
                    f"{repr(example['raw_values'][column])}\n"
                )

                report.write(
                    f"  {column} numeric tokens: "
                    f"{example['numeric_tokens'][column]}\n"
                )

            report.write(
                f"  candidate_n_AA: "
                f"{example['candidate_n_AA']}\n"
            )

            report.write("-" * 60 + "\n")

        report.write("\n")


# 顯示重要結果
print("left_shift = 3 前段QC欄位分析完成")
print()

print(
    "分析報告：",
    shift3_metric_report_path
)

print(
    "shift3總列數：",
    total_shift3_rows
)


# 顯示每個欄位的型態與數字數量
for column in metric_columns:

    print()
    print("欄位：", column)

    print(
        "資料型態分布：",
        dict(column_type_counts[column])
    )

    print(
        "欄位內數字數量分布：",
        dict(
            column_token_count_distribution[
                column
            ]
        )
    )


# 顯示主要四欄型態模式
print()
print("主要型態模式：")

for pattern, count in (
    type_pattern_counts.most_common(10)
):

    print(
        pattern,
        "：",
        count
    )


# 顯示主要數字數量模式
print()
print("主要數字數量模式：")

for pattern, count in (
    token_pattern_counts.most_common(10)
):

    print(
        pattern,
        "：",
        count
    )

left_shift = 3 前段QC欄位分析完成

分析報告： /Users/sheena/Desktop/summerintern/split_data/shift3_metric_block_analysis.txt
shift3總列數： 155685

欄位： FLD
資料型態分布： {'garbled': 155685}
欄位內數字數量分布： {1: 155685}

欄位： HomFLD
資料型態分布： {'numeric': 155685}
欄位內數字數量分布： {1: 155685}

欄位： HetSO
資料型態分布： {'numeric': 155685}
欄位內數字數量分布： {1: 155685}

欄位： HomRO
資料型態分布： {'numeric': 155685}
欄位內數字數量分布： {1: 155685}

主要型態模式：
('garbled', 'numeric', 'numeric', 'numeric') ： 155685

主要數字數量模式：
(1, 1, 1, 1) ： 155685


In [38]:
# 載入處理Tab分隔檔案的工具
import csv

# 載入正規表示式工具
# 用來從含亂碼的FLD尾端擷取數值
import re


# 設定shift3第1階段輸出檔案
shift3_stage1_path = (
    output_folder
    / "genotype_back_shift3_stage1_clean_FLD.txt"
)


# 設定FLD清理前後的稽核檔案
shift3_stage1_audit_path = (
    output_folder
    / "genotype_back_shift3_stage1_FLD_audit.txt"
)


# 定義完整純數字格式
# 可接受：
# 0
# 0.532
# -0.125
# 1.2E-5
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端的數字格式
# 例如：
# '?�數??0.548' → 擷取0.548
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_shift3_rows = 0

# FLD成功擷取數值的列數
fld_extract_success = 0

# FLD無法擷取的列數
fld_extract_failed = 0

# HomFLD、HetSO、HomRO仍為純數字的列數
metric_block_pass = 0

# 保存前20筆問題範例
problem_examples = []


# 開啟：
# 1. 含left_shift分類的後段資料
# 2. shift3 stage1修正版
# 3. FLD修復稽核檔
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift3_stage1_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift3_stage1_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄名讀取Tab分隔資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 修正版保留原始欄位
    # 再新增stage1的QC狀態
    output_fields = (
        reader.fieldnames
        + ["stage1_qc_status"]
    )


    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 設定稽核檔欄位
    audit_fields = [
        "source_row_number",
        "raw_FLD",
        "fixed_FLD",
        "HomFLD",
        "HetSO",
        "HomRO",
        "stage1_qc_status"
    ]


    # 建立稽核檔寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # 逐列讀取
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只處理left_shift = 3
        if row["left_shift"] != "3":
            continue

        total_shift3_rows += 1


        # 完整保留原始資料
        raw = row.copy()

        # 建立修正版
        fixed = row.copy()


        # =========================================
        # 1. 清理FLD
        # =========================================

        # 從原始FLD字串尾端擷取數字
        fld_match = ending_number_pattern.search(
            raw["FLD"].strip()
        )

        if fld_match:

            # 將尾端數字放回FLD
            fixed["FLD"] = fld_match.group(1)

            fld_extract_success += 1

            fld_ok = True

        else:

            # 無法擷取時保留原始內容
            fixed["FLD"] = raw["FLD"]

            fld_extract_failed += 1

            fld_ok = False


        # =========================================
        # 2. 驗證HomFLD、HetSO、HomRO
        # =========================================

        homfld_ok = bool(
            full_number_pattern.fullmatch(
                raw["HomFLD"].strip()
            )
        )

        hetso_ok = bool(
            full_number_pattern.fullmatch(
                raw["HetSO"].strip()
            )
        )

        homro_ok = bool(
            full_number_pattern.fullmatch(
                raw["HomRO"].strip()
            )
        )


        # 三欄全部為純數字
        metric_ok = (
            homfld_ok
            and hetso_ok
            and homro_ok
        )

        if metric_ok:
            metric_block_pass += 1


        # =========================================
        # 3. 設定stage1 QC結果
        # =========================================

        stage1_ok = (
            fld_ok
            and metric_ok
        )

        if stage1_ok:
            stage1_qc_status = "pass"
        else:
            stage1_qc_status = "fail"


        # 保存前20筆問題範例
        if (
            not stage1_ok
            and len(problem_examples) < 20
        ):
            problem_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "raw_FLD":
                        raw["FLD"],

                    "fixed_FLD":
                        fixed["FLD"],

                    "HomFLD":
                        raw["HomFLD"],

                    "HetSO":
                        raw["HetSO"],

                    "HomRO":
                        raw["HomRO"]
                }
            )


        # 新增QC結果
        fixed["stage1_qc_status"] = (
            stage1_qc_status
        )


        # 寫入stage1修正版
        writer.writerow(fixed)


        # 寫入修復前後對照
        audit_writer.writerow(
            {
                "source_row_number":
                    source_row_number,

                "raw_FLD":
                    raw["FLD"],

                "fixed_FLD":
                    fixed["FLD"],

                "HomFLD":
                    raw["HomFLD"],

                "HetSO":
                    raw["HetSO"],

                "HomRO":
                    raw["HomRO"],

                "stage1_qc_status":
                    stage1_qc_status
            }
        )


# 顯示處理結果
print("left_shift = 3，第1階段FLD清理完成")
print()

print("stage1修正版：", shift3_stage1_path)
print("FLD稽核檔：", shift3_stage1_audit_path)
print()

print("shift3總列數：", total_shift3_rows)
print("FLD成功擷取：", fld_extract_success)
print("FLD無法擷取：", fld_extract_failed)
print(
    "HomFLD、HetSO、HomRO皆為純數字：",
    metric_block_pass
)

print()
print("問題範例：")

for example in problem_examples:
    print(example)

left_shift = 3，第1階段FLD清理完成

stage1修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_stage1_clean_FLD.txt
FLD稽核檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_stage1_FLD_audit.txt

shift3總列數： 155685
FLD成功擷取： 155685
FLD無法擷取： 0
HomFLD、HetSO、HomRO皆為純數字： 155685

問題範例：


In [39]:
# 載入套件
import csv
import re


# 重新設定輸出檔案
# 會覆蓋上一版未保存source_row_number的stage1檔案
shift3_stage1_path = (
    output_folder
    / "genotype_back_shift3_stage1_clean_FLD.txt"
)

shift3_stage1_audit_path = (
    output_folder
    / "genotype_back_shift3_stage1_FLD_audit.txt"
)


# 完整純數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 擷取字串尾端數字
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 建立計數器
total_shift3_rows = 0
fld_extract_success = 0
fld_extract_failed = 0
stage1_qc_pass = 0
stage1_qc_fail = 0

# 保存問題範例
problem_examples = []


# 從包含全部680,865列的classified_back_path重新讀取
with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift3_stage1_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift3_stage1_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄名讀取Tab分隔資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 保存原始欄位名稱
    original_fields = reader.fieldnames.copy()


    # 修正版欄位順序：
    # source_row_number放在最前面
    # 後面保留原始欄位
    # 最後新增stage1_qc_status
    output_fields = (
        ["source_row_number"]
        + original_fields
        + ["stage1_qc_status"]
    )


    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 設定稽核檔欄位
    audit_fields = [
        "source_row_number",
        "raw_FLD",
        "fixed_FLD",
        "HomFLD",
        "HetSO",
        "HomRO",
        "stage1_qc_status"
    ]


    # 建立稽核檔寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # source_row_number是資料在完整classified_back檔案中的原始列序
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只保留left_shift = 3
        if row["left_shift"] != "3":
            continue

        total_shift3_rows += 1


        # 保存原始資料
        raw = row.copy()

        # 建立修正版
        fixed = row.copy()


        # -----------------------------------------
        # 1. 清理FLD
        # -----------------------------------------

        fld_match = ending_number_pattern.search(
            raw["FLD"].strip()
        )

        if fld_match:

            # 擷取亂碼尾端的純數值
            fixed["FLD"] = fld_match.group(1)

            fld_extract_success += 1
            fld_ok = True

        else:

            # 無法擷取時保留原始內容
            fixed["FLD"] = raw["FLD"]

            fld_extract_failed += 1
            fld_ok = False


        # -----------------------------------------
        # 2. 驗證HomFLD、HetSO與HomRO
        # -----------------------------------------

        homfld_ok = bool(
            full_number_pattern.fullmatch(
                raw["HomFLD"].strip()
            )
        )

        hetso_ok = bool(
            full_number_pattern.fullmatch(
                raw["HetSO"].strip()
            )
        )

        homro_ok = bool(
            full_number_pattern.fullmatch(
                raw["HomRO"].strip()
            )
        )


        # 四個前段欄位都必須合理
        stage1_ok = (
            fld_ok
            and homfld_ok
            and hetso_ok
            and homro_ok
        )


        if stage1_ok:

            stage1_qc_status = "pass"
            stage1_qc_pass += 1

        else:

            stage1_qc_status = "fail"
            stage1_qc_fail += 1

            # 保存前20筆失敗範例
            if len(problem_examples) < 20:
                problem_examples.append(
                    {
                        "source_row_number":
                            source_row_number,

                        "raw_FLD":
                            raw["FLD"],

                        "fixed_FLD":
                            fixed["FLD"],

                        "HomFLD":
                            raw["HomFLD"],

                        "HetSO":
                            raw["HetSO"],

                        "HomRO":
                            raw["HomRO"]
                    }
                )


        # -----------------------------------------
        # 3. 加入原始列序與QC狀態
        # -----------------------------------------

        fixed["source_row_number"] = (
            source_row_number
        )

        fixed["stage1_qc_status"] = (
            stage1_qc_status
        )


        # 寫入修正版
        writer.writerow(fixed)


        # 寫入稽核資料
        audit_writer.writerow(
            {
                "source_row_number":
                    source_row_number,

                "raw_FLD":
                    raw["FLD"],

                "fixed_FLD":
                    fixed["FLD"],

                "HomFLD":
                    raw["HomFLD"],

                "HetSO":
                    raw["HetSO"],

                "HomRO":
                    raw["HomRO"],

                "stage1_qc_status":
                    stage1_qc_status
            }
        )


# 顯示結果
print("left_shift = 3，已重新建立含source_row_number的stage1")
print()

print("stage1修正版：", shift3_stage1_path)
print("stage1稽核檔：", shift3_stage1_audit_path)
print()

print("shift3總列數：", total_shift3_rows)
print("FLD成功擷取：", fld_extract_success)
print("FLD無法擷取：", fld_extract_failed)
print("stage1 QC通過：", stage1_qc_pass)
print("stage1 QC未通過：", stage1_qc_fail)

print()
print("問題範例：")

for example in problem_examples:
    print(example)

left_shift = 3，已重新建立含source_row_number的stage1

stage1修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_stage1_clean_FLD.txt
stage1稽核檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_stage1_FLD_audit.txt

shift3總列數： 155685
FLD成功擷取： 155685
FLD無法擷取： 0
stage1 QC通過： 155685
stage1 QC未通過： 0

問題範例：


In [40]:
# 載入套件
import csv
import re
from collections import Counter


# 設定分析報告輸出位置
shift3_tail_validation_report_path = (
    output_folder
    / "shift3_tail_mapping_validation.txt"
)


# 定義完整數字格式
# 可辨認整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 判斷是否為純數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# 判斷欄位資料型態
def classify_value(value):

    if value is None:
        return "missing"

    value = value.strip()

    if value == "":
        return "empty"

    if value.upper() == "NA":
        return "NA"

    if value.upper() in {"TRUE", "FALSE"}:
        return "boolean"

    if full_number_pattern.fullmatch(value):
        return "numeric"

    if "�" in value or "?" in value:
        return "garbled"

    return "text"


# 候選尾端欄位的名稱
candidate_tail_fields = [
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 建立計數器
total_shift3_rows = 0

stage1_qc_pass_count = 0

nc_mean_pass_count = 0
mmd_pass_count = 0
maf_pass_count = 0
hwp_pass_count = 0
hwchi_pass_count = 0
call_modified_pass_count = 0

all_conditions_pass_count = 0
all_conditions_fail_count = 0


# 統計尾端資料型態組合
tail_type_pattern_counts = Counter()

# 統計染色體與資料型態的組合
group_pattern_counts = Counter()

# 統計MAF、p-value與Call Modified的實際類型
candidate_field_type_counts = {
    field: Counter()
    for field in candidate_tail_fields
}


# 保存每種型態模式前3筆範例
pattern_examples = {}

# 保存前20筆未通過範例
failed_examples = []


# 從shift3 stage1檔案讀取
# 此檔案已清理FLD並保留source_row_number
with shift3_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 逐列檢查
    for row in reader:

        total_shift3_rows += 1


        # 檢查上一階段QC
        stage1_ok = (
            row["stage1_qc_status"] == "pass"
        )

        if stage1_ok:
            stage1_qc_pass_count += 1


        # =========================================
        # 1. 建立向右移3欄後的候選尾端值
        # =========================================

        candidate_nc_x = (
            row["AB.meanY"].strip()
        )

        candidate_nc_y = (
            row["BB.meanX"].strip()
        )

        candidate_mmd = (
            row["BB.meanY"].strip()
        )

        candidate_maf = (
            row["NC.meanX"].strip()
        )

        candidate_hwp = (
            row["NC.meanY"].strip()
        )

        candidate_hwchi = (
            row["MMD"].strip()
        )

        candidate_call_modified = (
            row["MinorAlleleFrequency"]
            .strip()
            .upper()
        )


        # 建立候選尾端欄位字典
        candidate_values = {
            "NC.meanX": candidate_nc_x,
            "NC.meanY": candidate_nc_y,
            "MMD": candidate_mmd,
            "MinorAlleleFrequency":
                candidate_maf,
            "H.W.p-Value":
                candidate_hwp,
            "H.W.chisquared.statistic":
                candidate_hwchi,
            "Call Modified":
                candidate_call_modified
        }


        # =========================================
        # 2. 統計候選欄位資料型態
        # =========================================

        tail_type_pattern = tuple(
            classify_value(
                candidate_values[field]
            )
            for field in candidate_tail_fields
        )


        # 累計整列尾端型態模式
        tail_type_pattern_counts[
            tail_type_pattern
        ] += 1


        # 分別累計每個候選欄位的型態
        for field in candidate_tail_fields:

            value_type = classify_value(
                candidate_values[field]
            )

            candidate_field_type_counts[
                field
            ][value_type] += 1


        # =========================================
        # 3. 取得候選群組
        # =========================================

        # shift3向右移3格後：
        # 真正specialSNP_chr ← 目前n_BB
        # 真正gender_metrics ← 目前n_NC
        # 真正ConversionType ← 目前hemizygous
        candidate_chr = row["n_BB"].strip()

        candidate_gender = row["n_NC"].strip()

        candidate_conversion = (
            row["hemizygous"].strip()
        )


        group_pattern_counts[
            (
                candidate_chr,
                candidate_gender,
                candidate_conversion,
                tail_type_pattern
            )
        ] += 1


        # =========================================
        # 4. 驗證NC.meanX與NC.meanY
        # =========================================

        nc_mean_ok = (
            is_numeric_or_na(candidate_nc_x)
            and is_numeric_or_na(candidate_nc_y)
        )

        if nc_mean_ok:
            nc_mean_pass_count += 1


        # =========================================
        # 5. 驗證MMD
        # =========================================

        # MMD可以是NA或數值
        mmd_ok = is_numeric_or_na(
            candidate_mmd
        )

        if mmd_ok:
            mmd_pass_count += 1


        # =========================================
        # 6. 驗證MinorAlleleFrequency
        # =========================================

        # MAF可為NA
        if is_na(candidate_maf):

            maf_ok = True
            maf_value = None

        else:

            try:
                maf_value = float(candidate_maf)

                # Minor allele frequency應介於0與0.5
                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:
                maf_value = None
                maf_ok = False


        if maf_ok:
            maf_pass_count += 1


        # =========================================
        # 7. 驗證H.W.p-Value
        # =========================================

        # HWE p-value可為NA
        if is_na(candidate_hwp):

            hwp_ok = True
            hwp_value = None

        else:

            try:
                hwp_value = float(candidate_hwp)

                # p-value應介於0與1
                hwp_ok = (
                    0 <= hwp_value <= 1
                )

            except ValueError:
                hwp_value = None
                hwp_ok = False


        if hwp_ok:
            hwp_pass_count += 1


        # =========================================
        # 8. 驗證H.W.chisquared.statistic
        # =========================================

        # 卡方統計量可為NA
        if is_na(candidate_hwchi):

            hwchi_ok = True
            hwchi_value = None

        else:

            try:
                hwchi_value = float(
                    candidate_hwchi
                )

                # 卡方值應為非負數
                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:
                hwchi_value = None
                hwchi_ok = False


        if hwchi_ok:
            hwchi_pass_count += 1


        # =========================================
        # 9. 驗證Call Modified
        # =========================================

        call_modified_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )

        if call_modified_ok:
            call_modified_pass_count += 1


        # =========================================
        # 10. 綜合判斷
        # =========================================

        all_ok = (
            stage1_ok
            and nc_mean_ok
            and mmd_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
        )


        if all_ok:

            all_conditions_pass_count += 1

        else:

            all_conditions_fail_count += 1

            # 保存前20筆未通過範例
            if len(failed_examples) < 20:

                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_chr":
                            candidate_chr,

                        "candidate_gender":
                            candidate_gender,

                        "candidate_conversion":
                            candidate_conversion,

                        "candidate_NC.meanX":
                            candidate_nc_x,

                        "candidate_NC.meanY":
                            candidate_nc_y,

                        "candidate_MMD":
                            candidate_mmd,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified
                    }
                )


        # =========================================
        # 11. 保存每種型態模式的範例
        # =========================================

        if tail_type_pattern not in pattern_examples:
            pattern_examples[
                tail_type_pattern
            ] = []


        if (
            len(
                pattern_examples[
                    tail_type_pattern
                ]
            ) < 3
        ):

            pattern_examples[
                tail_type_pattern
            ].append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "candidate_chr":
                        candidate_chr,

                    "candidate_gender":
                        candidate_gender,

                    "candidate_conversion":
                        candidate_conversion,

                    "candidate_values":
                        candidate_values
                }
            )


# =========================================
# 12. 寫入完整分析報告
# =========================================

with shift3_tail_validation_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "left_shift = 3 tail mapping validation\n"
    )

    report.write(
        f"Total rows: {total_shift3_rows}\n\n"
    )


    report.write("=" * 80 + "\n")
    report.write("Validation summary\n")
    report.write("=" * 80 + "\n")

    report.write(
        f"Stage1 QC pass: "
        f"{stage1_qc_pass_count}\n"
    )

    report.write(
        f"NC means pass: "
        f"{nc_mean_pass_count}\n"
    )

    report.write(
        f"MMD pass: "
        f"{mmd_pass_count}\n"
    )

    report.write(
        f"MAF pass: "
        f"{maf_pass_count}\n"
    )

    report.write(
        f"H.W.p-Value pass: "
        f"{hwp_pass_count}\n"
    )

    report.write(
        f"H.W.chisquared pass: "
        f"{hwchi_pass_count}\n"
    )

    report.write(
        f"Call Modified pass: "
        f"{call_modified_pass_count}\n"
    )

    report.write(
        f"All conditions pass: "
        f"{all_conditions_pass_count}\n"
    )

    report.write(
        f"All conditions fail: "
        f"{all_conditions_fail_count}\n\n"
    )


    # 每個候選欄位的型態分布
    report.write("=" * 80 + "\n")
    report.write("Candidate field type counts\n")
    report.write("=" * 80 + "\n\n")

    for field in candidate_tail_fields:

        report.write(
            f"{field}: "
            f"{dict(candidate_field_type_counts[field])}\n"
        )


    # 尾端型態組合
    report.write("\n" + "=" * 80 + "\n")
    report.write("Tail type patterns\n")
    report.write("=" * 80 + "\n\n")

    for pattern, count in (
        tail_type_pattern_counts.most_common()
    ):

        report.write(
            f"pattern={pattern} | count={count}\n"
        )


    # 每種模式範例
    report.write("\n" + "=" * 80 + "\n")
    report.write("Pattern examples\n")
    report.write("=" * 80 + "\n\n")

    for pattern, examples in (
        pattern_examples.items()
    ):

        report.write(
            f"Pattern: {pattern}\n"
        )

        for example in examples:

            report.write(
                f"source_row_number: "
                f"{example['source_row_number']}\n"
            )

            report.write(
                f"chr: "
                f"{example['candidate_chr']}\n"
            )

            report.write(
                f"gender: "
                f"{example['candidate_gender']}\n"
            )

            report.write(
                f"ConversionType: "
                f"{example['candidate_conversion']}\n"
            )

            for field in candidate_tail_fields:

                report.write(
                    f"  {field}: "
                    f"{repr(example['candidate_values'][field])}\n"
                )

            report.write("-" * 60 + "\n")

        report.write("\n")


    # 未通過範例
    report.write("\n" + "=" * 80 + "\n")
    report.write("Failed examples\n")
    report.write("=" * 80 + "\n\n")

    for example in failed_examples:

        report.write(
            repr(example) + "\n"
        )


# =========================================
# 13. 顯示重要結果
# =========================================

print("left_shift = 3 尾端候選對應驗證完成")
print()

print(
    "分析報告：",
    shift3_tail_validation_report_path
)

print()
print("shift3總列數：", total_shift3_rows)
print("Stage1 QC通過：", stage1_qc_pass_count)
print("候選NC mean合理：", nc_mean_pass_count)
print("候選MMD合理：", mmd_pass_count)
print("候選MAF合理：", maf_pass_count)
print("候選H.W.p-Value合理：", hwp_pass_count)
print(
    "候選H.W.chisquared合理：",
    hwchi_pass_count
)
print(
    "候選Call Modified合理：",
    call_modified_pass_count
)
print(
    "所有條件同時通過：",
    all_conditions_pass_count
)
print(
    "未通過列數：",
    all_conditions_fail_count
)


print()
print("主要尾端型態模式：")

for pattern, count in (
    tail_type_pattern_counts.most_common(10)
):

    print(
        pattern,
        "：",
        count
    )


print()
print("前5筆未通過範例：")

for example in failed_examples[:5]:
    print(example)

left_shift = 3 尾端候選對應驗證完成

分析報告： /Users/sheena/Desktop/summerintern/split_data/shift3_tail_mapping_validation.txt

shift3總列數： 155685
Stage1 QC通過： 155685
候選NC mean合理： 155685
候選MMD合理： 0
候選MAF合理： 1126
候選H.W.p-Value合理： 155685
候選H.W.chisquared合理： 0
候選Call Modified合理： 0
所有條件同時通過： 0
未通過列數： 155685

主要尾端型態模式：
('NA', 'NA', 'garbled', 'numeric', 'NA', 'boolean', 'empty') ： 155685

前5筆未通過範例：
{'source_row_number': '92', 'candidate_chr': 'autosomal', 'candidate_gender': 'all', 'candidate_conversion': 'MonoHighResolution', 'candidate_NC.meanX': 'NA', 'candidate_NC.meanY': 'NA', 'candidate_MMD': '?�數??0', 'candidate_MAF': '1', 'candidate_H.W.p': 'NA', 'candidate_H.W.chi': 'FALSE', 'candidate_CallModified': ''}
{'source_row_number': '97', 'candidate_chr': 'autosomal', 'candidate_gender': 'all', 'candidate_conversion': 'MonoHighResolution', 'candidate_NC.meanX': 'NA', 'candidate_NC.meanY': 'NA', 'candidate_MMD': '?�數??0', 'candidate_MAF': '1', 'candidate_H.W.p': 'NA', 'candidate_H.W.chi': 'FALSE', 'cand

還是有缺失值 表示後面欄位有不同的位移
發現NC.meanX之後有不同位移

In [41]:
# 載入需要的套件
import csv
import re
from collections import Counter


# 設定驗證報告輸出位置
shift3_tail_v2_report_path = (
    output_folder
    / "shift3_tail_mapping_v2_validation.txt"
)


# 定義完整純數字格式
# 可接受整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 定義字串尾端數字格式
# 例如：
# '?�數??0.0123' → 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 判斷是否為純數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# 建立計數器
total_rows = 0

nc_mean_pass = 0
maf_extract_pass = 0
maf_range_pass = 0
hwp_pass = 0
hwchi_pass = 0
call_modified_pass = 0
trailing_empty_pass = 0

all_conditions_pass = 0
all_conditions_fail = 0


# 統計不同染色體與ConversionType
group_counts = Counter()


# 保存前20筆未通過範例
failed_examples = []


# 保存前5筆候選修正範例
candidate_examples = []


# 從shift3 stage1讀取
# 此檔已清理FLD，但其他欄位尚未移動
with shift3_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 逐列驗證
    for row in reader:

        total_rows += 1


        # =========================================
        # 1. 候選NC.meanX與NC.meanY
        # =========================================

        # 固定向右移3格後：
        # 真正NC.meanX ← 原始AB.meanY
        candidate_nc_x = row["AB.meanY"].strip()

        # 真正NC.meanY ← 原始BB.meanX
        candidate_nc_y = row["BB.meanX"].strip()


        # NC mean可為數字或NA
        nc_ok = (
            is_numeric_or_na(candidate_nc_x)
            and is_numeric_or_na(candidate_nc_y)
        )

        if nc_ok:
            nc_mean_pass += 1


        # =========================================
        # 2. 候選MinorAlleleFrequency
        # =========================================

        # 原始BB.meanY是亂碼加數字
        # 從尾端擷取真正的MAF
        maf_match = ending_number_pattern.search(
            row["BB.meanY"].strip()
        )


        if maf_match:

            candidate_maf = maf_match.group(1)

            maf_extract_pass += 1

            try:
                maf_value = float(candidate_maf)

                # MAF應介於0至0.5
                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:
                maf_value = None
                maf_ok = False

        else:

            candidate_maf = ""
            maf_value = None
            maf_ok = False


        if maf_ok:
            maf_range_pass += 1


        # =========================================
        # 3. 候選H.W.p-Value
        # =========================================

        # 原始NC.meanX候選對應真正HWE p-value
        candidate_hwp = row["NC.meanX"].strip()


        try:
            hwp_value = float(candidate_hwp)

            # p-value應介於0至1
            hwp_ok = (
                0 <= hwp_value <= 1
            )

        except ValueError:
            hwp_value = None
            hwp_ok = False


        if hwp_ok:
            hwp_pass += 1


        # =========================================
        # 4. 候選H.W.chisquared.statistic
        # =========================================

        # 原始NC.meanY候選對應真正卡方值
        candidate_hwchi = row["NC.meanY"].strip()


        # NA可接受
        if is_na(candidate_hwchi):

            hwchi_ok = True

        else:

            try:
                hwchi_value = float(
                    candidate_hwchi
                )

                # 卡方統計量應為非負數
                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:
                hwchi_ok = False


        if hwchi_ok:
            hwchi_pass += 1


        # =========================================
        # 5. 候選Call Modified
        # =========================================

        # 原始MMD欄位中的布林值
        # 候選對應真正Call Modified
        candidate_call_modified = (
            row["MMD"]
            .strip()
            .upper()
        )


        call_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )


        if call_ok:
            call_modified_pass += 1


        # =========================================
        # 6. 檢查尾端剩餘欄位是否為空白
        # =========================================

        # 若前面的值都提前，最後幾欄應為空白
        trailing_columns = [
            "MinorAlleleFrequency",
            "H.W.p-Value",
            "H.W.chisquared.statistic",
            "Call Modified"
        ]


        trailing_empty_ok = all(
            row[column].strip() == ""
            for column in trailing_columns
        )


        if trailing_empty_ok:
            trailing_empty_pass += 1


        # =========================================
        # 7. 取得候選結構群組
        # =========================================

        # shift3修正後的結構欄位
        candidate_chr = row["n_BB"].strip()

        candidate_gender = row["n_NC"].strip()

        candidate_conversion = (
            row["hemizygous"].strip()
        )


        group_counts[
            (
                candidate_chr,
                candidate_gender,
                candidate_conversion
            )
        ] += 1


        # =========================================
        # 8. 綜合判斷
        # =========================================

        all_ok = (
            row["stage1_qc_status"] == "pass"
            and nc_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_ok
            and trailing_empty_ok
        )


        if all_ok:

            all_conditions_pass += 1

        else:

            all_conditions_fail += 1


            # 最多保存前20筆未通過範例
            if len(failed_examples) < 20:

                failed_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "candidate_chr":
                            candidate_chr,

                        "candidate_gender":
                            candidate_gender,

                        "candidate_conversion":
                            candidate_conversion,

                        "raw_AB.meanY":
                            row["AB.meanY"],

                        "raw_BB.meanX":
                            row["BB.meanX"],

                        "raw_BB.meanY":
                            row["BB.meanY"],

                        "candidate_NC.meanX":
                            candidate_nc_x,

                        "candidate_NC.meanY":
                            candidate_nc_y,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p":
                            candidate_hwp,

                        "candidate_H.W.chi":
                            candidate_hwchi,

                        "candidate_CallModified":
                            candidate_call_modified,

                        "trailing_empty_ok":
                            trailing_empty_ok
                    }
                )


        # 保存前5筆候選修正範例
        if len(candidate_examples) < 5:

            candidate_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "fixed_NC.meanX":
                        candidate_nc_x,

                    "fixed_NC.meanY":
                        candidate_nc_y,

                    # 此模式沒有可靠的MMD
                    "fixed_MMD":
                        "",

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    "fixed_H.W.chisquared.statistic":
                        candidate_hwchi,

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# =========================================
# 9. 寫入驗證報告
# =========================================

with shift3_tail_v2_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "left_shift = 3 tail mapping v2 validation\n"
    )

    report.write(
        f"Total rows: {total_rows}\n\n"
    )


    report.write(
        f"NC mean pass: {nc_mean_pass}\n"
    )

    report.write(
        f"MAF extraction pass: "
        f"{maf_extract_pass}\n"
    )

    report.write(
        f"MAF range pass: "
        f"{maf_range_pass}\n"
    )

    report.write(
        f"H.W.p-Value pass: "
        f"{hwp_pass}\n"
    )

    report.write(
        f"H.W.chisquared pass: "
        f"{hwchi_pass}\n"
    )

    report.write(
        f"Call Modified pass: "
        f"{call_modified_pass}\n"
    )

    report.write(
        f"Trailing empty pass: "
        f"{trailing_empty_pass}\n"
    )

    report.write(
        f"All conditions pass: "
        f"{all_conditions_pass}\n"
    )

    report.write(
        f"All conditions fail: "
        f"{all_conditions_fail}\n\n"
    )


    report.write("=" * 80 + "\n")
    report.write("Group summary\n")
    report.write("=" * 80 + "\n")


    for group, count in group_counts.most_common():

        report.write(
            f"chr={group[0]} | "
            f"gender={group[1]} | "
            f"ConversionType={group[2]} | "
            f"count={count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Failed examples\n")
    report.write("=" * 80 + "\n")


    for example in failed_examples:

        report.write(
            repr(example) + "\n"
        )


# =========================================
# 10. 顯示驗證結果
# =========================================

print("left_shift = 3 尾端候選對應v2驗證完成")
print()

print(
    "驗證報告：",
    shift3_tail_v2_report_path
)

print()
print("shift3總列數：", total_rows)
print("候選NC mean合理：", nc_mean_pass)
print("MAF尾端擷取成功：", maf_extract_pass)
print("候選MAF值域合理：", maf_range_pass)
print("候選H.W.p-Value合理：", hwp_pass)
print(
    "候選H.W.chisquared合理：",
    hwchi_pass
)
print(
    "候選Call Modified合理：",
    call_modified_pass
)
print(
    "尾端剩餘欄位皆空白：",
    trailing_empty_pass
)
print(
    "所有條件同時通過：",
    all_conditions_pass
)
print(
    "未通過列數：",
    all_conditions_fail
)


print()
print("前5筆候選修正範例：")

for example in candidate_examples:
    print(example)


print()
print("前5筆未通過範例：")

for example in failed_examples[:5]:
    print(example)

left_shift = 3 尾端候選對應v2驗證完成

驗證報告： /Users/sheena/Desktop/summerintern/split_data/shift3_tail_mapping_v2_validation.txt

shift3總列數： 155685
候選NC mean合理： 155685
MAF尾端擷取成功： 155685
候選MAF值域合理： 155685
候選H.W.p-Value合理： 155685
候選H.W.chisquared合理： 155685
候選Call Modified合理： 155685
尾端剩餘欄位皆空白： 155685
所有條件同時通過： 155685
未通過列數： 0

前5筆候選修正範例：
{'source_row_number': '92', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '97', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': '106', 'fixed_NC.meanX': 'NA', 'fixed_NC.meanY': 'NA', 'fixed_MMD': '', 'fixed_MinorAlleleFrequency': '0', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared.statistic': 'NA', 'fixed_Call Modified': 'FA

In [42]:
# 載入需要的套件
import csv
import re
from collections import Counter


# =========================================
# 1. 設定輸出檔案
# =========================================

# left_shift = 3 正式完整修正版
shift3_all_fixed_path = (
    output_folder
    / "genotype_back_shift3_all_fixed.txt"
)


# 修復前後對照紀錄
shift3_all_audit_path = (
    output_folder
    / "genotype_back_shift3_all_fixed_audit.txt"
)


# =========================================
# 2. 定義數字格式
# =========================================

# 完整純數字格式
# 可辨認整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 從亂碼字串尾端擷取數字
# 例如：
# '?�數??0.0123' → 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# =========================================
# 3. 建立驗證函數
# =========================================

# 判斷是否為完整純數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# =========================================
# 4. 中段欄位修復對應
# =========================================

# key：修正後的真正欄位
# value：stage1中目前存放該值的欄位
shift3_mapping = {

    # genotype count
    "n_AA": "HomRO",
    "n_AB": "nMinorAllele",
    "n_BB": "Nclus",
    "n_NC": "n_AA",

    # SNP分類資訊
    "hemizygous": "n_AB",
    "specialSNP_chr": "n_BB",
    "gender_metrics": "n_NC",
    "ConversionType": "hemizygous",

    # probeset分類
    "BestProbeset": "specialSNP_chr",
    "BestandRecommended": "gender_metrics",
    "HomHet": "ConversionType",

    # cluster mean
    "AA.meanX": "BestProbeset",
    "AA.meanY": "BestandRecommended",
    "AB.meanX": "HomHet",
    "AB.meanY": "AA.meanX",
    "BB.meanX": "AA.meanY",
    "BB.meanY": "AB.meanX",
    "NC.meanX": "AB.meanY",
    "NC.meanY": "BB.meanX"
}


# 不同性別群組對應的樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體類別
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# 合理的ConversionType
valid_conversion_values = {
    "MonoHighResolution",
    "PolyHighResolution"
}


# 修復後應為數字或NA的cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# =========================================
# 5. 建立計數器
# =========================================

total_rows = 0

maf_extract_success = 0
maf_extract_failed = 0

final_qc_pass = 0
final_qc_fail = 0


# 統計各群組
group_counts = Counter()


# 統計ConversionType
conversion_counts = Counter()


# 保存前20筆問題範例
problem_examples = []


# =========================================
# 6. 正式讀取並修復資料
# =========================================

with shift3_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift3_all_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift3_all_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取stage1資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 正式修正版保留stage1所有欄位
    # 再新增修復模式與最終QC狀態
    output_fields = (
        reader.fieldnames
        + [
            "shift3_subpattern",
            "final_qc_status"
        ]
    )


    # 建立正式修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 設定修復稽核檔欄位
    audit_fields = [
        "source_row_number",
        "shift3_subpattern",

        "raw_HomRO",
        "raw_nMinorAllele",
        "raw_Nclus",

        "raw_BB.meanY",
        "raw_NC.meanX",
        "raw_NC.meanY",
        "raw_MMD",

        "fixed_HomRO",
        "fixed_nMinorAllele",
        "fixed_Nclus",

        "fixed_n_AA",
        "fixed_n_AB",
        "fixed_n_BB",
        "fixed_n_NC",

        "fixed_specialSNP_chr",
        "fixed_gender_metrics",
        "fixed_ConversionType",

        "fixed_MMD",
        "fixed_MinorAlleleFrequency",
        "fixed_H.W.p-Value",
        "fixed_H.W.chisquared.statistic",
        "fixed_Call Modified",

        "genotype_total",
        "expected_total",
        "final_qc_status"
    ]


    # 建立修復稽核寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # 逐列修復
    for row in reader:

        total_rows += 1


        # 完整保存修復前資料
        # 後續全部從raw取值，避免欄位互相覆蓋
        raw = row.copy()


        # 建立修正版資料
        fixed = row.copy()


        # =====================================
        # 6-1. 設定修復模式
        # =====================================

        shift3_subpattern = "shift3-fixed"


        # =====================================
        # 6-2. 處理缺失的前段欄位
        # =====================================

        # 真正HomRO沒有輸出
        fixed["HomRO"] = ""

        # 真正nMinorAllele沒有輸出
        fixed["nMinorAllele"] = ""

        # 真正Nclus沒有輸出
        fixed["Nclus"] = ""


        # =====================================
        # 6-3. 修復中段與cluster mean欄位
        # =====================================

        for target_field, source_field in (
            shift3_mapping.items()
        ):

            fixed[target_field] = (
                raw[source_field]
            )


        # =====================================
        # 6-4. 修復尾端欄位
        # =====================================

        # 真正MMD沒有可靠輸出
        fixed["MMD"] = ""


        # 從原始BB.meanY的亂碼尾端擷取MAF
        maf_match = ending_number_pattern.search(
            raw["BB.meanY"].strip()
        )


        if maf_match:

            fixed["MinorAlleleFrequency"] = (
                maf_match.group(1)
            )

            maf_extract_success += 1

        else:

            fixed["MinorAlleleFrequency"] = ""

            maf_extract_failed += 1


        # 原始NC.meanX對應真正H.W.p-Value
        fixed["H.W.p-Value"] = (
            raw["NC.meanX"]
        )


        # 原始NC.meanY對應真正卡方統計量
        fixed["H.W.chisquared.statistic"] = (
            raw["NC.meanY"]
        )


        # 原始MMD中的FALSE對應真正Call Modified
        fixed["Call Modified"] = (
            raw["MMD"]
            .strip()
            .upper()
        )


        # =====================================
        # 6-5. 修復後QC
        # =====================================

        problem_reasons = []


        # 檢查stage1 QC
        if raw["stage1_qc_status"] != "pass":

            problem_reasons.append(
                "stage1_qc_status不是pass"
            )


        # 檢查left_shift
        if raw["left_shift"] != "3":

            problem_reasons.append(
                "left_shift不是3"
            )


        # -------------------------------------
        # genotype count QC
        # -------------------------------------

        try:

            genotype_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            expected_total = (
                expected_total_by_gender.get(
                    fixed["gender_metrics"]
                )
            )


            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        if not count_ok:

            problem_reasons.append(
                "genotype count合計錯誤"
            )


        # -------------------------------------
        # 文字結構QC
        # -------------------------------------

        structure_ok = (
            fixed["hemizygous"] in {"0", "1"}
            and fixed["specialSNP_chr"]
            in valid_chr_values
            and fixed["gender_metrics"]
            in expected_total_by_gender
            and fixed["ConversionType"]
            in valid_conversion_values
            and fixed["BestProbeset"]
            in {"0", "1"}
            and fixed["BestandRecommended"]
            in {"0", "1"}
            and fixed["HomHet"]
            in {"0", "1"}
        )


        if not structure_ok:

            problem_reasons.append(
                "文字分類欄位結構錯誤"
            )


        # -------------------------------------
        # cluster mean QC
        # -------------------------------------

        cluster_mean_ok = all(
            is_numeric_or_na(
                fixed[column]
            )
            for column in cluster_mean_columns
        )


        if not cluster_mean_ok:

            problem_reasons.append(
                "cluster mean不是數字或NA"
            )


        # -------------------------------------
        # MAF QC
        # -------------------------------------

        try:

            maf_value = float(
                fixed["MinorAlleleFrequency"]
            )

            maf_ok = (
                0 <= maf_value <= 0.5
            )

        except (ValueError, TypeError):

            maf_value = None
            maf_ok = False


        if not maf_ok:

            problem_reasons.append(
                "MAF超出0至0.5"
            )


        # -------------------------------------
        # H.W.p-Value QC
        # -------------------------------------

        try:

            hwp_value = float(
                fixed["H.W.p-Value"]
            )

            hwp_ok = (
                0 <= hwp_value <= 1
            )

        except (ValueError, TypeError):

            hwp_value = None
            hwp_ok = False


        if not hwp_ok:

            problem_reasons.append(
                "H.W.p-Value超出0至1"
            )


        # -------------------------------------
        # H.W.chisquared QC
        # -------------------------------------

        hwchi_text = (
            fixed["H.W.chisquared.statistic"]
            .strip()
        )


        if hwchi_text.upper() == "NA":

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(
                    hwchi_text
                )

                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:

                hwchi_ok = False


        if not hwchi_ok:

            problem_reasons.append(
                "H.W.chisquared不合理"
            )


        # -------------------------------------
        # Call Modified QC
        # -------------------------------------

        call_modified_ok = (
            fixed["Call Modified"]
            in {"TRUE", "FALSE"}
        )


        if not call_modified_ok:

            problem_reasons.append(
                "Call Modified不是布林值"
            )


        # -------------------------------------
        # 缺失欄位QC
        # -------------------------------------

        missing_fields_ok = (
            fixed["HomRO"] == ""
            and fixed["nMinorAllele"] == ""
            and fixed["Nclus"] == ""
            and fixed["MMD"] == ""
        )


        if not missing_fields_ok:

            problem_reasons.append(
                "應空白的缺失欄位未設為空白"
            )


        # =====================================
        # 6-6. 設定最終QC狀態
        # =====================================

        if len(problem_reasons) == 0:

            final_qc_status = "pass"

            final_qc_pass += 1

        else:

            final_qc_status = "fail"

            final_qc_fail += 1


            # 最多保存前20筆問題範例
            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            raw["source_row_number"],

                        "specialSNP_chr":
                            fixed["specialSNP_chr"],

                        "gender_metrics":
                            fixed["gender_metrics"],

                        "ConversionType":
                            fixed["ConversionType"],

                        "genotype_total":
                            genotype_total,

                        "expected_total":
                            expected_total,

                        "MAF":
                            fixed[
                                "MinorAlleleFrequency"
                            ],

                        "H.W.p-Value":
                            fixed["H.W.p-Value"],

                        "H.W.chisquared":
                            fixed[
                                "H.W.chisquared.statistic"
                            ],

                        "Call Modified":
                            fixed["Call Modified"],

                        "problem_reasons":
                            problem_reasons
                    }
                )


        # 加入修復模式與QC狀態
        fixed["shift3_subpattern"] = (
            shift3_subpattern
        )

        fixed["final_qc_status"] = (
            final_qc_status
        )


        # 統計修正後群組
        group_counts[
            (
                fixed["specialSNP_chr"],
                fixed["gender_metrics"]
            )
        ] += 1


        # 統計ConversionType
        conversion_counts[
            fixed["ConversionType"]
        ] += 1


        # 寫入正式修正版
        writer.writerow(fixed)


        # =====================================
        # 6-7. 寫入修復稽核檔
        # =====================================

        audit_writer.writerow(
            {
                "source_row_number":
                    raw["source_row_number"],

                "shift3_subpattern":
                    shift3_subpattern,

                "raw_HomRO":
                    raw["HomRO"],

                "raw_nMinorAllele":
                    raw["nMinorAllele"],

                "raw_Nclus":
                    raw["Nclus"],

                "raw_BB.meanY":
                    raw["BB.meanY"],

                "raw_NC.meanX":
                    raw["NC.meanX"],

                "raw_NC.meanY":
                    raw["NC.meanY"],

                "raw_MMD":
                    raw["MMD"],

                "fixed_HomRO":
                    fixed["HomRO"],

                "fixed_nMinorAllele":
                    fixed["nMinorAllele"],

                "fixed_Nclus":
                    fixed["Nclus"],

                "fixed_n_AA":
                    fixed["n_AA"],

                "fixed_n_AB":
                    fixed["n_AB"],

                "fixed_n_BB":
                    fixed["n_BB"],

                "fixed_n_NC":
                    fixed["n_NC"],

                "fixed_specialSNP_chr":
                    fixed["specialSNP_chr"],

                "fixed_gender_metrics":
                    fixed["gender_metrics"],

                "fixed_ConversionType":
                    fixed["ConversionType"],

                "fixed_MMD":
                    fixed["MMD"],

                "fixed_MinorAlleleFrequency":
                    fixed[
                        "MinorAlleleFrequency"
                    ],

                "fixed_H.W.p-Value":
                    fixed["H.W.p-Value"],

                "fixed_H.W.chisquared.statistic":
                    fixed[
                        "H.W.chisquared.statistic"
                    ],

                "fixed_Call Modified":
                    fixed["Call Modified"],

                "genotype_total":
                    genotype_total,

                "expected_total":
                    expected_total,

                "final_qc_status":
                    final_qc_status
            }
        )


# =========================================
# 7. 顯示正式修復結果
# =========================================

print("left_shift = 3 正式修復完成")
print()

print(
    "正式修正版：",
    shift3_all_fixed_path
)

print(
    "修復稽核檔：",
    shift3_all_audit_path
)

print()

print("shift3總列數：", total_rows)
print("MAF擷取成功：", maf_extract_success)
print("MAF擷取失敗：", maf_extract_failed)
print("最終QC通過：", final_qc_pass)
print("最終QC未通過：", final_qc_fail)


print()
print("染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for conversion, count in (
    conversion_counts.most_common()
):

    print(
        conversion,
        "：",
        count
    )


print()
print("問題範例：")

for example in problem_examples:

    print(example)

left_shift = 3 正式修復完成

正式修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_all_fixed.txt
修復稽核檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_all_fixed_audit.txt

shift3總列數： 155685
MAF擷取成功： 155685
MAF擷取失敗： 0
最終QC通過： 155685
最終QC未通過： 0

染色體與性別群組：
specialSNP_chr = autosomal | gender_metrics = all | 列數 = 131659
specialSNP_chr = Y | gender_metrics = male | 列數 = 13418
specialSNP_chr = X | gender_metrics = female | 列數 = 10083
specialSNP_chr = MT | gender_metrics = all | 列數 = 525

ConversionType分布：
MonoHighResolution ： 154252
PolyHighResolution ： 1433

問題範例：


In [43]:
# 載入需要的套件
import csv
import re
from collections import Counter


# =========================================
# 1. 設定輸出檔案
# =========================================

# left_shift = 3 正式完整修正版
shift3_all_fixed_path = (
    output_folder
    / "genotype_back_shift3_all_fixed.txt"
)


# 修復前後對照紀錄
shift3_all_audit_path = (
    output_folder
    / "genotype_back_shift3_all_fixed_audit.txt"
)


# =========================================
# 2. 定義數字格式
# =========================================

# 完整純數字格式
# 可辨認整數、小數、負數與科學記號
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 從亂碼字串尾端擷取數字
# 例如：
# '?�數??0.0123' → 0.0123
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# =========================================
# 3. 建立驗證函數
# =========================================

# 判斷是否為完整純數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# =========================================
# 4. 中段欄位修復對應
# =========================================

# key：修正後的真正欄位
# value：stage1中目前存放該值的欄位
shift3_mapping = {

    # genotype count
    "n_AA": "HomRO",
    "n_AB": "nMinorAllele",
    "n_BB": "Nclus",
    "n_NC": "n_AA",

    # SNP分類資訊
    "hemizygous": "n_AB",
    "specialSNP_chr": "n_BB",
    "gender_metrics": "n_NC",
    "ConversionType": "hemizygous",

    # probeset分類
    "BestProbeset": "specialSNP_chr",
    "BestandRecommended": "gender_metrics",
    "HomHet": "ConversionType",

    # cluster mean
    "AA.meanX": "BestProbeset",
    "AA.meanY": "BestandRecommended",
    "AB.meanX": "HomHet",
    "AB.meanY": "AA.meanX",
    "BB.meanX": "AA.meanY",
    "BB.meanY": "AB.meanX",
    "NC.meanX": "AB.meanY",
    "NC.meanY": "BB.meanX"
}


# 不同性別群組對應的樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體類別
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# 合理的ConversionType
valid_conversion_values = {
    "MonoHighResolution",
    "PolyHighResolution"
}


# 修復後應為數字或NA的cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# =========================================
# 5. 建立計數器
# =========================================

total_rows = 0

maf_extract_success = 0
maf_extract_failed = 0

final_qc_pass = 0
final_qc_fail = 0


# 統計各群組
group_counts = Counter()


# 統計ConversionType
conversion_counts = Counter()


# 保存前20筆問題範例
problem_examples = []


# =========================================
# 6. 正式讀取並修復資料
# =========================================

with shift3_stage1_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift3_all_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift3_all_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取stage1資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 正式修正版保留stage1所有欄位
    # 再新增修復模式與最終QC狀態
    output_fields = (
        reader.fieldnames
        + [
            "shift3_subpattern",
            "final_qc_status"
        ]
    )


    # 建立正式修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 設定修復稽核檔欄位
    audit_fields = [
        "source_row_number",
        "shift3_subpattern",

        "raw_HomRO",
        "raw_nMinorAllele",
        "raw_Nclus",

        "raw_BB.meanY",
        "raw_NC.meanX",
        "raw_NC.meanY",
        "raw_MMD",

        "fixed_HomRO",
        "fixed_nMinorAllele",
        "fixed_Nclus",

        "fixed_n_AA",
        "fixed_n_AB",
        "fixed_n_BB",
        "fixed_n_NC",

        "fixed_specialSNP_chr",
        "fixed_gender_metrics",
        "fixed_ConversionType",

        "fixed_MMD",
        "fixed_MinorAlleleFrequency",
        "fixed_H.W.p-Value",
        "fixed_H.W.chisquared.statistic",
        "fixed_Call Modified",

        "genotype_total",
        "expected_total",
        "final_qc_status"
    ]


    # 建立修復稽核寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # 逐列修復
    for row in reader:

        total_rows += 1


        # 完整保存修復前資料
        # 後續全部從raw取值，避免欄位互相覆蓋
        raw = row.copy()


        # 建立修正版資料
        fixed = row.copy()


        # =====================================
        # 6-1. 設定修復模式
        # =====================================

        shift3_subpattern = "shift3-fixed"


        # =====================================
        # 6-2. 處理缺失的前段欄位
        # =====================================

        # 真正HomRO沒有輸出
        fixed["HomRO"] = ""

        # 真正nMinorAllele沒有輸出
        fixed["nMinorAllele"] = ""

        # 真正Nclus沒有輸出
        fixed["Nclus"] = ""


        # =====================================
        # 6-3. 修復中段與cluster mean欄位
        # =====================================

        for target_field, source_field in (
            shift3_mapping.items()
        ):

            fixed[target_field] = (
                raw[source_field]
            )


        # =====================================
        # 6-4. 修復尾端欄位
        # =====================================

        # 真正MMD沒有可靠輸出
        fixed["MMD"] = ""


        # 從原始BB.meanY的亂碼尾端擷取MAF
        maf_match = ending_number_pattern.search(
            raw["BB.meanY"].strip()
        )


        if maf_match:

            fixed["MinorAlleleFrequency"] = (
                maf_match.group(1)
            )

            maf_extract_success += 1

        else:

            fixed["MinorAlleleFrequency"] = ""

            maf_extract_failed += 1


        # 原始NC.meanX對應真正H.W.p-Value
        fixed["H.W.p-Value"] = (
            raw["NC.meanX"]
        )


        # 原始NC.meanY對應真正卡方統計量
        fixed["H.W.chisquared.statistic"] = (
            raw["NC.meanY"]
        )


        # 原始MMD中的FALSE對應真正Call Modified
        fixed["Call Modified"] = (
            raw["MMD"]
            .strip()
            .upper()
        )


        # =====================================
        # 6-5. 修復後QC
        # =====================================

        problem_reasons = []


        # 檢查stage1 QC
        if raw["stage1_qc_status"] != "pass":

            problem_reasons.append(
                "stage1_qc_status不是pass"
            )


        # 檢查left_shift
        if raw["left_shift"] != "3":

            problem_reasons.append(
                "left_shift不是3"
            )


        # -------------------------------------
        # genotype count QC
        # -------------------------------------

        try:

            genotype_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            expected_total = (
                expected_total_by_gender.get(
                    fixed["gender_metrics"]
                )
            )


            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        if not count_ok:

            problem_reasons.append(
                "genotype count合計錯誤"
            )


        # -------------------------------------
        # 文字結構QC
        # -------------------------------------

        structure_ok = (
            fixed["hemizygous"] in {"0", "1"}
            and fixed["specialSNP_chr"]
            in valid_chr_values
            and fixed["gender_metrics"]
            in expected_total_by_gender
            and fixed["ConversionType"]
            in valid_conversion_values
            and fixed["BestProbeset"]
            in {"0", "1"}
            and fixed["BestandRecommended"]
            in {"0", "1"}
            and fixed["HomHet"]
            in {"0", "1"}
        )


        if not structure_ok:

            problem_reasons.append(
                "文字分類欄位結構錯誤"
            )


        # -------------------------------------
        # cluster mean QC
        # -------------------------------------

        cluster_mean_ok = all(
            is_numeric_or_na(
                fixed[column]
            )
            for column in cluster_mean_columns
        )


        if not cluster_mean_ok:

            problem_reasons.append(
                "cluster mean不是數字或NA"
            )


        # -------------------------------------
        # MAF QC
        # -------------------------------------

        try:

            maf_value = float(
                fixed["MinorAlleleFrequency"]
            )

            maf_ok = (
                0 <= maf_value <= 0.5
            )

        except (ValueError, TypeError):

            maf_value = None
            maf_ok = False


        if not maf_ok:

            problem_reasons.append(
                "MAF超出0至0.5"
            )


        # -------------------------------------
        # H.W.p-Value QC
        # -------------------------------------

        try:

            hwp_value = float(
                fixed["H.W.p-Value"]
            )

            hwp_ok = (
                0 <= hwp_value <= 1
            )

        except (ValueError, TypeError):

            hwp_value = None
            hwp_ok = False


        if not hwp_ok:

            problem_reasons.append(
                "H.W.p-Value超出0至1"
            )


        # -------------------------------------
        # H.W.chisquared QC
        # -------------------------------------

        hwchi_text = (
            fixed["H.W.chisquared.statistic"]
            .strip()
        )


        if hwchi_text.upper() == "NA":

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(
                    hwchi_text
                )

                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:

                hwchi_ok = False


        if not hwchi_ok:

            problem_reasons.append(
                "H.W.chisquared不合理"
            )


        # -------------------------------------
        # Call Modified QC
        # -------------------------------------

        call_modified_ok = (
            fixed["Call Modified"]
            in {"TRUE", "FALSE"}
        )


        if not call_modified_ok:

            problem_reasons.append(
                "Call Modified不是布林值"
            )


        # -------------------------------------
        # 缺失欄位QC
        # -------------------------------------

        missing_fields_ok = (
            fixed["HomRO"] == ""
            and fixed["nMinorAllele"] == ""
            and fixed["Nclus"] == ""
            and fixed["MMD"] == ""
        )


        if not missing_fields_ok:

            problem_reasons.append(
                "應空白的缺失欄位未設為空白"
            )


        # =====================================
        # 6-6. 設定最終QC狀態
        # =====================================

        if len(problem_reasons) == 0:

            final_qc_status = "pass"

            final_qc_pass += 1

        else:

            final_qc_status = "fail"

            final_qc_fail += 1


            # 最多保存前20筆問題範例
            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            raw["source_row_number"],

                        "specialSNP_chr":
                            fixed["specialSNP_chr"],

                        "gender_metrics":
                            fixed["gender_metrics"],

                        "ConversionType":
                            fixed["ConversionType"],

                        "genotype_total":
                            genotype_total,

                        "expected_total":
                            expected_total,

                        "MAF":
                            fixed[
                                "MinorAlleleFrequency"
                            ],

                        "H.W.p-Value":
                            fixed["H.W.p-Value"],

                        "H.W.chisquared":
                            fixed[
                                "H.W.chisquared.statistic"
                            ],

                        "Call Modified":
                            fixed["Call Modified"],

                        "problem_reasons":
                            problem_reasons
                    }
                )


        # 加入修復模式與QC狀態
        fixed["shift3_subpattern"] = (
            shift3_subpattern
        )

        fixed["final_qc_status"] = (
            final_qc_status
        )


        # 統計修正後群組
        group_counts[
            (
                fixed["specialSNP_chr"],
                fixed["gender_metrics"]
            )
        ] += 1


        # 統計ConversionType
        conversion_counts[
            fixed["ConversionType"]
        ] += 1


        # 寫入正式修正版
        writer.writerow(fixed)


        # =====================================
        # 6-7. 寫入修復稽核檔
        # =====================================

        audit_writer.writerow(
            {
                "source_row_number":
                    raw["source_row_number"],

                "shift3_subpattern":
                    shift3_subpattern,

                "raw_HomRO":
                    raw["HomRO"],

                "raw_nMinorAllele":
                    raw["nMinorAllele"],

                "raw_Nclus":
                    raw["Nclus"],

                "raw_BB.meanY":
                    raw["BB.meanY"],

                "raw_NC.meanX":
                    raw["NC.meanX"],

                "raw_NC.meanY":
                    raw["NC.meanY"],

                "raw_MMD":
                    raw["MMD"],

                "fixed_HomRO":
                    fixed["HomRO"],

                "fixed_nMinorAllele":
                    fixed["nMinorAllele"],

                "fixed_Nclus":
                    fixed["Nclus"],

                "fixed_n_AA":
                    fixed["n_AA"],

                "fixed_n_AB":
                    fixed["n_AB"],

                "fixed_n_BB":
                    fixed["n_BB"],

                "fixed_n_NC":
                    fixed["n_NC"],

                "fixed_specialSNP_chr":
                    fixed["specialSNP_chr"],

                "fixed_gender_metrics":
                    fixed["gender_metrics"],

                "fixed_ConversionType":
                    fixed["ConversionType"],

                "fixed_MMD":
                    fixed["MMD"],

                "fixed_MinorAlleleFrequency":
                    fixed[
                        "MinorAlleleFrequency"
                    ],

                "fixed_H.W.p-Value":
                    fixed["H.W.p-Value"],

                "fixed_H.W.chisquared.statistic":
                    fixed[
                        "H.W.chisquared.statistic"
                    ],

                "fixed_Call Modified":
                    fixed["Call Modified"],

                "genotype_total":
                    genotype_total,

                "expected_total":
                    expected_total,

                "final_qc_status":
                    final_qc_status
            }
        )


# =========================================
# 7. 顯示正式修復結果
# =========================================

print("left_shift = 3 正式修復完成")
print()

print(
    "正式修正版：",
    shift3_all_fixed_path
)

print(
    "修復稽核檔：",
    shift3_all_audit_path
)

print()

print("shift3總列數：", total_rows)
print("MAF擷取成功：", maf_extract_success)
print("MAF擷取失敗：", maf_extract_failed)
print("最終QC通過：", final_qc_pass)
print("最終QC未通過：", final_qc_fail)


print()
print("染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for conversion, count in (
    conversion_counts.most_common()
):

    print(
        conversion,
        "：",
        count
    )


print()
print("問題範例：")

for example in problem_examples:

    print(example)

left_shift = 3 正式修復完成

正式修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_all_fixed.txt
修復稽核檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift3_all_fixed_audit.txt

shift3總列數： 155685
MAF擷取成功： 155685
MAF擷取失敗： 0
最終QC通過： 155685
最終QC未通過： 0

染色體與性別群組：
specialSNP_chr = autosomal | gender_metrics = all | 列數 = 131659
specialSNP_chr = Y | gender_metrics = male | 列數 = 13418
specialSNP_chr = X | gender_metrics = female | 列數 = 10083
specialSNP_chr = MT | gender_metrics = all | 列數 = 525

ConversionType分布：
MonoHighResolution ： 154252
PolyHighResolution ： 1433

問題範例：


In [44]:
# 載入需要的套件
import csv
import re
from collections import Counter


# ============================================================
# 1. 設定驗證報告輸出位置
# ============================================================

shift0_validation_report_path = (
    output_folder
    / "shift0_original_structure_validation.txt"
)


# ============================================================
# 2. 定義數字格式與檢查函數
# ============================================================

# 完整數字格式：
# 0、1.25、-0.52、1.2E-5
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 判斷是否為數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# ============================================================
# 3. 設定合理類別
# ============================================================

# gender_metrics對應的樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體分類
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# 前段QC指標欄位
metric_columns = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO"
]


# Cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# ============================================================
# 4. 建立計數器
# ============================================================

total_shift0_rows = 0

metric_pass_count = 0
genotype_count_pass = 0
structure_pass_count = 0
cluster_mean_pass = 0
maf_pass_count = 0
hwp_pass_count = 0
hwchi_pass_count = 0
call_modified_pass = 0

core_all_pass = 0
full_all_pass = 0


# 統計染色體與性別群組
group_counts = Counter()

# 統計ConversionType
conversion_counts = Counter()

# 統計genotype count合計
genotype_total_counts = Counter()

# 統計Call Modified類型
call_modified_counts = Counter()


# 保存前20筆核心結構未通過的範例
failed_examples = []


# ============================================================
# 5. 讀取classified_back_path並驗證shift0
# ============================================================

with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    # 使用欄位名稱讀取Tab分隔資料
    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # source_row_number代表完整680,865列中的原始順序
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只分析left_shift = 0
        if row["left_shift"] != "0":
            continue

        total_shift0_rows += 1


        # ====================================================
        # 5-1. 驗證FLD、HomFLD、HetSO、HomRO
        # ====================================================

        metric_ok = all(
            is_numeric_or_na(
                row[column]
            )
            for column in metric_columns
        )

        if metric_ok:
            metric_pass_count += 1


        # ====================================================
        # 5-2. 驗證genotype count
        # ====================================================

        try:

            n_AA = int(row["n_AA"])
            n_AB = int(row["n_AB"])
            n_BB = int(row["n_BB"])
            n_NC = int(row["n_NC"])

            genotype_total = (
                n_AA
                + n_AB
                + n_BB
                + n_NC
            )

            expected_total = (
                expected_total_by_gender.get(
                    row["gender_metrics"]
                )
            )

            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        genotype_total_counts[
            genotype_total
        ] += 1


        if count_ok:
            genotype_count_pass += 1


        # ====================================================
        # 5-3. 驗證文字欄位結構
        # ====================================================

        structure_ok = (
            row["hemizygous"] in {"0", "1"}
            and row["specialSNP_chr"]
            in valid_chr_values
            and row["gender_metrics"]
            in expected_total_by_gender
            and row["ConversionType"].strip() != ""
            and row["BestProbeset"]
            in {"0", "1"}
            and row["BestandRecommended"]
            in {"0", "1"}
            and row["HomHet"]
            in {"0", "1"}
        )


        if structure_ok:
            structure_pass_count += 1


        # 統計群組
        group_counts[
            (
                row["specialSNP_chr"],
                row["gender_metrics"],
                row["hemizygous"]
            )
        ] += 1


        # 統計ConversionType
        conversion_counts[
            row["ConversionType"]
        ] += 1


        # ====================================================
        # 5-4. 驗證cluster mean
        # ====================================================

        cluster_ok = all(
            is_numeric_or_na(
                row[column]
            )
            for column in cluster_mean_columns
        )


        if cluster_ok:
            cluster_mean_pass += 1


        # ====================================================
        # 5-5. 驗證MinorAlleleFrequency
        # ====================================================

        maf_text = (
            row["MinorAlleleFrequency"]
            .strip()
        )


        if is_na(maf_text):

            maf_ok = True

        else:

            try:
                maf_value = float(maf_text)

                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:
                maf_ok = False


        if maf_ok:
            maf_pass_count += 1


        # ====================================================
        # 5-6. 驗證H.W.p-Value
        # ====================================================

        hwp_text = (
            row["H.W.p-Value"]
            .strip()
        )


        if is_na(hwp_text):

            hwp_ok = True

        else:

            try:
                hwp_value = float(hwp_text)

                hwp_ok = (
                    0 <= hwp_value <= 1
                )

            except ValueError:
                hwp_ok = False


        if hwp_ok:
            hwp_pass_count += 1


        # ====================================================
        # 5-7. 驗證H.W.chisquared.statistic
        # ====================================================

        hwchi_text = (
            row["H.W.chisquared.statistic"]
            .strip()
        )


        if is_na(hwchi_text):

            hwchi_ok = True

        else:

            try:
                hwchi_value = float(hwchi_text)

                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:
                hwchi_ok = False


        if hwchi_ok:
            hwchi_pass_count += 1


        # ====================================================
        # 5-8. 驗證Call Modified
        # ====================================================

        call_modified_value = (
            row["Call Modified"]
            .strip()
            .upper()
        )


        call_modified_counts[
            call_modified_value
        ] += 1


        call_ok = (
            call_modified_value
            in {"TRUE", "FALSE"}
        )


        if call_ok:
            call_modified_pass += 1


        # ====================================================
        # 5-9. 核心與完整QC
        # ====================================================

        # 核心QC主要判斷欄位是否對齊
        core_ok = (
            metric_ok
            and count_ok
            and structure_ok
            and cluster_ok
        )


        if core_ok:
            core_all_pass += 1


        # 完整QC再加入MAF、HWE與Call Modified
        full_ok = (
            core_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_ok
        )


        if full_ok:
            full_all_pass += 1


        # 保存前20筆核心未通過範例
        if (
            not core_ok
            and len(failed_examples) < 20
        ):

            failed_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "FLD":
                        row["FLD"],

                    "HomFLD":
                        row["HomFLD"],

                    "HetSO":
                        row["HetSO"],

                    "HomRO":
                        row["HomRO"],

                    "n_AA":
                        row["n_AA"],

                    "n_AB":
                        row["n_AB"],

                    "n_BB":
                        row["n_BB"],

                    "n_NC":
                        row["n_NC"],

                    "genotype_total":
                        genotype_total,

                    "expected_total":
                        expected_total,

                    "specialSNP_chr":
                        row["specialSNP_chr"],

                    "gender_metrics":
                        row["gender_metrics"],

                    "ConversionType":
                        row["ConversionType"],

                    "metric_ok":
                        metric_ok,

                    "count_ok":
                        count_ok,

                    "structure_ok":
                        structure_ok,

                    "cluster_ok":
                        cluster_ok
                }
            )


# ============================================================
# 6. 寫入驗證報告
# ============================================================

with shift0_validation_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "left_shift = 0 original structure validation\n"
    )

    report.write(
        f"Total rows: {total_shift0_rows}\n\n"
    )


    report.write(
        f"Metric block pass: "
        f"{metric_pass_count}\n"
    )

    report.write(
        f"Genotype count pass: "
        f"{genotype_count_pass}\n"
    )

    report.write(
        f"Structure pass: "
        f"{structure_pass_count}\n"
    )

    report.write(
        f"Cluster mean pass: "
        f"{cluster_mean_pass}\n"
    )

    report.write(
        f"MAF pass: "
        f"{maf_pass_count}\n"
    )

    report.write(
        f"H.W.p-Value pass: "
        f"{hwp_pass_count}\n"
    )

    report.write(
        f"H.W.chisquared pass: "
        f"{hwchi_pass_count}\n"
    )

    report.write(
        f"Call Modified pass: "
        f"{call_modified_pass}\n"
    )

    report.write(
        f"Core all pass: "
        f"{core_all_pass}\n"
    )

    report.write(
        f"Full all pass: "
        f"{full_all_pass}\n\n"
    )


    report.write("=" * 80 + "\n")
    report.write("Genotype totals\n")
    report.write("=" * 80 + "\n")

    for total, count in (
        genotype_total_counts.most_common()
    ):

        report.write(
            f"total={total} | count={count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Chromosome and gender groups\n")
    report.write("=" * 80 + "\n")

    for group, count in group_counts.most_common():

        report.write(
            f"chr={group[0]} | "
            f"gender={group[1]} | "
            f"hemizygous={group[2]} | "
            f"count={count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("ConversionType distribution\n")
    report.write("=" * 80 + "\n")

    for conversion, count in (
        conversion_counts.most_common()
    ):

        report.write(
            f"{conversion}: {count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Call Modified distribution\n")
    report.write("=" * 80 + "\n")

    for value, count in (
        call_modified_counts.most_common()
    ):

        report.write(
            f"{repr(value)}: {count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Failed core examples\n")
    report.write("=" * 80 + "\n")

    for example in failed_examples:

        report.write(
            repr(example) + "\n"
        )


# ============================================================
# 7. 顯示重要結果
# ============================================================

print("left_shift = 0 原始結構驗證完成")
print()

print(
    "驗證報告：",
    shift0_validation_report_path
)

print()
print("shift0總列數：", total_shift0_rows)
print("前段QC指標合理：", metric_pass_count)
print("genotype count合理：", genotype_count_pass)
print("文字欄位結構合理：", structure_pass_count)
print("cluster mean合理：", cluster_mean_pass)
print("MAF合理：", maf_pass_count)
print("H.W.p-Value合理：", hwp_pass_count)
print("H.W.chisquared合理：", hwchi_pass_count)
print("Call Modified合理：", call_modified_pass)
print("核心條件全部通過：", core_all_pass)
print("完整條件全部通過：", full_all_pass)


print()
print("genotype count合計分布：")

for total, count in (
    genotype_total_counts.most_common()
):

    print(
        "合計 =",
        total,
        "| 列數 =",
        count
    )


print()
print("主要染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| hemizygous =",
        group[2],
        "| 列數 =",
        count
    )


print()
print("Call Modified分布：")

for value, count in (
    call_modified_counts.most_common()
):

    print(
        repr(value),
        "：",
        count
    )


print()
print("前5筆核心未通過範例：")

for example in failed_examples[:5]:

    print(example)

left_shift = 0 原始結構驗證完成

驗證報告： /Users/sheena/Desktop/summerintern/split_data/shift0_original_structure_validation.txt

shift0總列數： 297913
前段QC指標合理： 297913
genotype count合理： 297913
文字欄位結構合理： 297913
cluster mean合理： 297913
MAF合理： 297913
H.W.p-Value合理： 297913
H.W.chisquared合理： 297913
Call Modified合理： 297913
核心條件全部通過： 297913
完整條件全部通過： 297913

genotype count合計分布：
合計 = 240 | 列數 = 292114
合計 = 163 | 列數 = 5799

主要染色體與性別群組：
specialSNP_chr = autosomal | gender_metrics = all | hemizygous = 0 | 列數 = 292114
specialSNP_chr = X | gender_metrics = female | hemizygous = 0 | 列數 = 5799

Call Modified分布：
'FALSE' ： 297913

前5筆核心未通過範例：


In [45]:
# 載入需要的套件
import csv
import re
from collections import Counter


# ============================================================
# 1. 設定輸出檔案
# ============================================================

# shift0正式輸出檔
shift0_all_fixed_path = (
    output_folder
    / "genotype_back_shift0_all_fixed.txt"
)


# shift0稽核摘要檔
shift0_audit_path = (
    output_folder
    / "genotype_back_shift0_all_fixed_audit.txt"
)


# ============================================================
# 2. 定義數字格式與檢查函數
# ============================================================

full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 判斷是否為數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# 不同性別群組的預期樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體分類
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# 前段QC欄位
metric_columns = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO"
]


# Cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# ============================================================
# 3. 建立計數器
# ============================================================

total_rows = 0
final_qc_pass = 0
final_qc_fail = 0

group_counts = Counter()
conversion_counts = Counter()

problem_examples = []


# ============================================================
# 4. 建立shift0正式輸出檔
# ============================================================

with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift0_all_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift0_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取完整後段資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 正式輸出保留原始欄位
    # 並增加原始列號、修復模式與QC狀態
    output_fields = (
        ["source_row_number"]
        + reader.fieldnames
        + [
            "repair_group",
            "final_qc_status"
        ]
    )


    # 建立正式輸出寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 稽核檔只保存重要欄位
    audit_fields = [
        "source_row_number",
        "repair_group",
        "specialSNP_chr",
        "gender_metrics",
        "ConversionType",
        "n_AA",
        "n_AB",
        "n_BB",
        "n_NC",
        "genotype_total",
        "expected_total",
        "final_qc_status"
    ]


    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # source_row_number代表680,865列中的原始順序
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只處理left_shift = 0
        if row["left_shift"] != "0":
            continue

        total_rows += 1


        # shift0不修改任何原始欄位
        fixed = row.copy()


        # ====================================================
        # 4-1. 前段QC指標
        # ====================================================

        metric_ok = all(
            is_numeric_or_na(
                fixed[column]
            )
            for column in metric_columns
        )


        # ====================================================
        # 4-2. Genotype count QC
        # ====================================================

        try:

            genotype_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            expected_total = (
                expected_total_by_gender.get(
                    fixed["gender_metrics"]
                )
            )


            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        # ====================================================
        # 4-3. 文字欄位結構QC
        # ====================================================

        structure_ok = (
            fixed["hemizygous"] in {"0", "1"}
            and fixed["specialSNP_chr"]
            in valid_chr_values
            and fixed["gender_metrics"]
            in expected_total_by_gender
            and fixed["ConversionType"].strip() != ""
            and fixed["BestProbeset"] in {"0", "1"}
            and fixed["BestandRecommended"] in {"0", "1"}
            and fixed["HomHet"] in {"0", "1"}
        )


        # ====================================================
        # 4-4. Cluster mean QC
        # ====================================================

        cluster_ok = all(
            is_numeric_or_na(
                fixed[column]
            )
            for column in cluster_mean_columns
        )


        # ====================================================
        # 4-5. MAF QC
        # ====================================================

        maf_text = (
            fixed["MinorAlleleFrequency"]
            .strip()
        )


        if is_na(maf_text):

            maf_ok = True

        else:

            try:

                maf_value = float(maf_text)

                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:

                maf_ok = False


        # ====================================================
        # 4-6. HWE p-value QC
        # ====================================================

        hwp_text = (
            fixed["H.W.p-Value"]
            .strip()
        )


        if is_na(hwp_text):

            hwp_ok = True

        else:

            try:

                hwp_value = float(hwp_text)

                hwp_ok = (
                    0 <= hwp_value <= 1
                )

            except ValueError:

                hwp_ok = False


        # ====================================================
        # 4-7. HWE chi-square QC
        # ====================================================

        hwchi_text = (
            fixed["H.W.chisquared.statistic"]
            .strip()
        )


        if is_na(hwchi_text):

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(hwchi_text)

                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:

                hwchi_ok = False


        # ====================================================
        # 4-8. Call Modified QC
        # ====================================================

        call_modified_ok = (
            fixed["Call Modified"]
            .strip()
            .upper()
            in {"TRUE", "FALSE"}
        )


        # ====================================================
        # 4-9. 最終QC
        # ====================================================

        final_ok = (
            metric_ok
            and count_ok
            and structure_ok
            and cluster_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
        )


        if final_ok:

            final_qc_status = "pass"
            final_qc_pass += 1

        else:

            final_qc_status = "fail"
            final_qc_fail += 1


            # 最多保存前20筆異常範例
            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            source_row_number,

                        "specialSNP_chr":
                            fixed["specialSNP_chr"],

                        "gender_metrics":
                            fixed["gender_metrics"],

                        "genotype_total":
                            genotype_total,

                        "expected_total":
                            expected_total,

                        "metric_ok":
                            metric_ok,

                        "count_ok":
                            count_ok,

                        "structure_ok":
                            structure_ok,

                        "cluster_ok":
                            cluster_ok,

                        "maf_ok":
                            maf_ok,

                        "hwp_ok":
                            hwp_ok,

                        "hwchi_ok":
                            hwchi_ok,

                        "call_modified_ok":
                            call_modified_ok
                    }
                )


        # ====================================================
        # 4-10. 加入管理欄位
        # ====================================================

        fixed["source_row_number"] = (
            source_row_number
        )

        fixed["repair_group"] = (
            "shift0-original"
        )

        fixed["final_qc_status"] = (
            final_qc_status
        )


        # 統計群組
        group_counts[
            (
                fixed["specialSNP_chr"],
                fixed["gender_metrics"]
            )
        ] += 1


        conversion_counts[
            fixed["ConversionType"]
        ] += 1


        # 寫入正式檔案
        writer.writerow(fixed)


        # 寫入稽核檔案
        audit_writer.writerow(
            {
                "source_row_number":
                    source_row_number,

                "repair_group":
                    "shift0-original",

                "specialSNP_chr":
                    fixed["specialSNP_chr"],

                "gender_metrics":
                    fixed["gender_metrics"],

                "ConversionType":
                    fixed["ConversionType"],

                "n_AA":
                    fixed["n_AA"],

                "n_AB":
                    fixed["n_AB"],

                "n_BB":
                    fixed["n_BB"],

                "n_NC":
                    fixed["n_NC"],

                "genotype_total":
                    genotype_total,

                "expected_total":
                    expected_total,

                "final_qc_status":
                    final_qc_status
            }
        )


# ============================================================
# 5. 顯示結果
# ============================================================

print("left_shift = 0 正式檔案建立完成")
print()

print(
    "正式檔案：",
    shift0_all_fixed_path
)

print(
    "稽核檔案：",
    shift0_audit_path
)

print()

print("shift0總列數：", total_rows)
print("最終QC通過：", final_qc_pass)
print("最終QC未通過：", final_qc_fail)


print()
print("染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for conversion, count in (
    conversion_counts.most_common()
):

    print(
        conversion,
        "：",
        count
    )


print()
print("問題範例：")

for example in problem_examples:

    print(example)

left_shift = 0 正式檔案建立完成

正式檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift0_all_fixed.txt
稽核檔案： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift0_all_fixed_audit.txt

shift0總列數： 297913
最終QC通過： 297913
最終QC未通過： 0

染色體與性別群組：
specialSNP_chr = autosomal | gender_metrics = all | 列數 = 292114
specialSNP_chr = X | gender_metrics = female | 列數 = 5799

ConversionType分布：
PolyHighResolution ： 297913

問題範例：


In [46]:
# 載入需要的套件
import csv
import re
from collections import Counter


# ============================================================
# 1. 設定輸出檔案
# ============================================================

# shift1正式完整修正版
shift1_all_fixed_path = (
    output_folder
    / "genotype_back_shift1_all_fixed.txt"
)


# shift1修復稽核檔
shift1_all_audit_path = (
    output_folder
    / "genotype_back_shift1_all_fixed_audit.txt"
)


# ============================================================
# 2. 定義數字格式與檢查函數
# ============================================================

# 完整純數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 從字串尾端擷取數字
# 例如：
# '?�數??0.538' → 0.538
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 判斷是否為完整數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# ============================================================
# 3. 設定QC使用的合理值
# ============================================================

# 不同性別群組的預期樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體類別
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# Cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# ============================================================
# 4. 設定向右移一欄的對應
# ============================================================

# 修正後的目標欄位
shift1_target_fields = [
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 上述目標欄位的資料來源
# 每個值都來自原本左邊一欄
shift1_source_fields = [
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic"
]


# ============================================================
# 5. 建立計數器
# ============================================================

total_rows = 0

homfld_extract_success = 0
homfld_extract_failed = 0

trailing_blank_pass = 0

final_qc_pass = 0
final_qc_fail = 0


# 統計修正後群組
group_counts = Counter()

# 統計ConversionType
conversion_counts = Counter()

# 統計genotype count總和
genotype_total_counts = Counter()


# 保存前20筆問題範例
problem_examples = []


# ============================================================
# 6. 正式修復shift1
# ============================================================

with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift1_all_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift1_all_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 使用欄位名稱讀取完整後段資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 正式輸出欄位
    output_fields = (
        ["source_row_number"]
        + reader.fieldnames
        + [
            "repair_group",
            "shift1_subpattern",
            "final_qc_status"
        ]
    )


    # 建立修正版寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 稽核檔欄位
    audit_fields = [
        "source_row_number",
        "repair_group",
        "shift1_subpattern",

        "raw_FLD",
        "raw_HomFLD",
        "raw_HetSO",
        "raw_HomRO",
        "raw_Call Modified",

        "fixed_FLD",
        "fixed_HomFLD",
        "fixed_HetSO",
        "fixed_HomRO",

        "fixed_n_AA",
        "fixed_n_AB",
        "fixed_n_BB",
        "fixed_n_NC",

        "fixed_specialSNP_chr",
        "fixed_gender_metrics",
        "fixed_ConversionType",

        "fixed_MinorAlleleFrequency",
        "fixed_H.W.p-Value",
        "fixed_H.W.chisquared.statistic",
        "fixed_Call Modified",

        "genotype_total",
        "expected_total",
        "raw_trailing_blank",
        "final_qc_status"
    ]


    # 建立稽核檔寫入器
    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # source_row_number代表680,865列中的原始順序
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只處理left_shift = 1
        if row["left_shift"] != "1":
            continue

        total_rows += 1


        # 完整保存原始資料
        raw = row.copy()

        # 建立修正版
        fixed = row.copy()


        # ====================================================
        # 6-1. FLD維持原值
        # ====================================================

        fixed["FLD"] = raw["FLD"].strip()

        fld_ok = is_numeric(
            fixed["FLD"]
        )


        # ====================================================
        # 6-2. 清理HomFLD
        # ====================================================

        homfld_match = ending_number_pattern.search(
            raw["HomFLD"].strip()
        )


        if homfld_match:

            fixed["HomFLD"] = (
                homfld_match.group(1)
            )

            homfld_extract_success += 1
            homfld_ok = True

        else:

            fixed["HomFLD"] = raw["HomFLD"]

            homfld_extract_failed += 1
            homfld_ok = False


        # ====================================================
        # 6-3. HetSO原始值缺失
        # ====================================================

        fixed["HetSO"] = ""


        # ====================================================
        # 6-4. 從HomRO開始向右移一欄
        # ====================================================

        # 必須全部從raw讀取
        # 避免前面修改後覆蓋後續仍需使用的值
        for target_field, source_field in zip(
            shift1_target_fields,
            shift1_source_fields
        ):

            fixed[target_field] = (
                raw[source_field]
            )


        # ====================================================
        # 6-5. 檢查原始最後一欄是否為空白
        # ====================================================

        raw_trailing_blank = (
            raw["Call Modified"].strip() == ""
        )


        if raw_trailing_blank:
            trailing_blank_pass += 1


        # ====================================================
        # 6-6. 判定shift1子模式
        # ====================================================

        if (
            fixed["specialSNP_chr"] == "X"
            and fixed["gender_metrics"] == "female"
        ):

            shift1_subpattern = "shift1-X-female"

        elif fixed["gender_metrics"] == "all":

            shift1_subpattern = "shift1-all"

        elif fixed["gender_metrics"] == "male":

            shift1_subpattern = "shift1-male"

        else:

            shift1_subpattern = "shift1-other"


        # ====================================================
        # 6-7. Genotype count QC
        # ====================================================

        try:

            genotype_total = (
                int(fixed["n_AA"])
                + int(fixed["n_AB"])
                + int(fixed["n_BB"])
                + int(fixed["n_NC"])
            )


            expected_total = (
                expected_total_by_gender.get(
                    fixed["gender_metrics"]
                )
            )


            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        genotype_total_counts[
            genotype_total
        ] += 1


        # ====================================================
        # 6-8. 文字結構QC
        # ====================================================

        structure_ok = (
            fixed["hemizygous"] in {"0", "1"}
            and fixed["specialSNP_chr"]
            in valid_chr_values
            and fixed["gender_metrics"]
            in expected_total_by_gender
            and fixed["ConversionType"].strip() != ""
            and fixed["BestProbeset"]
            in {"0", "1"}
            and fixed["BestandRecommended"]
            in {"0", "1"}
            and fixed["HomHet"]
            in {"0", "1"}
        )


        # ====================================================
        # 6-9. Cluster mean QC
        # ====================================================

        cluster_ok = all(
            is_numeric_or_na(
                fixed[column]
            )
            for column in cluster_mean_columns
        )


        # ====================================================
        # 6-10. MAF QC
        # ====================================================

        maf_text = (
            fixed["MinorAlleleFrequency"]
            .strip()
        )


        if is_na(maf_text):

            maf_ok = True

        else:

            try:

                maf_value = float(maf_text)

                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:

                maf_ok = False


        # ====================================================
        # 6-11. H.W.p-Value QC
        # ====================================================

        hwp_text = (
            fixed["H.W.p-Value"]
            .strip()
        )


        if is_na(hwp_text):

            hwp_ok = True

        else:

            try:

                hwp_value = float(hwp_text)

                hwp_ok = (
                    0 <= hwp_value <= 1
                )

            except ValueError:

                hwp_ok = False


        # ====================================================
        # 6-12. H.W.chisquared QC
        # ====================================================

        hwchi_text = (
            fixed["H.W.chisquared.statistic"]
            .strip()
        )


        if is_na(hwchi_text):

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(hwchi_text)

                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:

                hwchi_ok = False


        # ====================================================
        # 6-13. Call Modified QC
        # ====================================================

        fixed["Call Modified"] = (
            fixed["Call Modified"]
            .strip()
            .upper()
        )


        call_modified_ok = (
            fixed["Call Modified"]
            in {"TRUE", "FALSE"}
        )


        # ====================================================
        # 6-14. 最終QC
        # ====================================================

        final_ok = (
            fld_ok
            and homfld_ok
            and fixed["HetSO"] == ""
            and is_numeric_or_na(
                fixed["HomRO"]
            )
            and count_ok
            and structure_ok
            and cluster_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
            and raw_trailing_blank
        )


        if final_ok:

            final_qc_status = "pass"
            final_qc_pass += 1

        else:

            final_qc_status = "fail"
            final_qc_fail += 1


            # 保存前20筆問題資料
            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            source_row_number,

                        "FLD":
                            fixed["FLD"],

                        "HomFLD":
                            fixed["HomFLD"],

                        "HomRO":
                            fixed["HomRO"],

                        "genotype_total":
                            genotype_total,

                        "expected_total":
                            expected_total,

                        "specialSNP_chr":
                            fixed["specialSNP_chr"],

                        "gender_metrics":
                            fixed["gender_metrics"],

                        "ConversionType":
                            fixed["ConversionType"],

                        "Call Modified":
                            fixed["Call Modified"],

                        "raw_trailing_blank":
                            raw_trailing_blank,

                        "fld_ok":
                            fld_ok,

                        "homfld_ok":
                            homfld_ok,

                        "count_ok":
                            count_ok,

                        "structure_ok":
                            structure_ok,

                        "cluster_ok":
                            cluster_ok,

                        "maf_ok":
                            maf_ok,

                        "hwp_ok":
                            hwp_ok,

                        "hwchi_ok":
                            hwchi_ok,

                        "call_modified_ok":
                            call_modified_ok
                    }
                )


        # ====================================================
        # 6-15. 加入管理欄位
        # ====================================================

        fixed["source_row_number"] = (
            source_row_number
        )

        fixed["repair_group"] = (
            "shift1-repaired"
        )

        fixed["shift1_subpattern"] = (
            shift1_subpattern
        )

        fixed["final_qc_status"] = (
            final_qc_status
        )


        # 統計群組
        group_counts[
            (
                fixed["specialSNP_chr"],
                fixed["gender_metrics"],
                shift1_subpattern
            )
        ] += 1


        conversion_counts[
            fixed["ConversionType"]
        ] += 1


        # 寫入正式修正版
        writer.writerow(fixed)


        # ====================================================
        # 6-16. 寫入稽核檔
        # ====================================================

        audit_writer.writerow(
            {
                "source_row_number":
                    source_row_number,

                "repair_group":
                    "shift1-repaired",

                "shift1_subpattern":
                    shift1_subpattern,

                "raw_FLD":
                    raw["FLD"],

                "raw_HomFLD":
                    raw["HomFLD"],

                "raw_HetSO":
                    raw["HetSO"],

                "raw_HomRO":
                    raw["HomRO"],

                "raw_Call Modified":
                    raw["Call Modified"],

                "fixed_FLD":
                    fixed["FLD"],

                "fixed_HomFLD":
                    fixed["HomFLD"],

                "fixed_HetSO":
                    fixed["HetSO"],

                "fixed_HomRO":
                    fixed["HomRO"],

                "fixed_n_AA":
                    fixed["n_AA"],

                "fixed_n_AB":
                    fixed["n_AB"],

                "fixed_n_BB":
                    fixed["n_BB"],

                "fixed_n_NC":
                    fixed["n_NC"],

                "fixed_specialSNP_chr":
                    fixed["specialSNP_chr"],

                "fixed_gender_metrics":
                    fixed["gender_metrics"],

                "fixed_ConversionType":
                    fixed["ConversionType"],

                "fixed_MinorAlleleFrequency":
                    fixed[
                        "MinorAlleleFrequency"
                    ],

                "fixed_H.W.p-Value":
                    fixed["H.W.p-Value"],

                "fixed_H.W.chisquared.statistic":
                    fixed[
                        "H.W.chisquared.statistic"
                    ],

                "fixed_Call Modified":
                    fixed["Call Modified"],

                "genotype_total":
                    genotype_total,

                "expected_total":
                    expected_total,

                "raw_trailing_blank":
                    raw_trailing_blank,

                "final_qc_status":
                    final_qc_status
            }
        )


# ============================================================
# 7. 顯示正式修復結果
# ============================================================

print("left_shift = 1 正式修復完成")
print()

print(
    "正式修正版：",
    shift1_all_fixed_path
)

print(
    "修復稽核檔：",
    shift1_all_audit_path
)

print()

print("shift1總列數：", total_rows)
print(
    "HomFLD數值擷取成功：",
    homfld_extract_success
)
print(
    "HomFLD數值擷取失敗：",
    homfld_extract_failed
)
print(
    "原始最後一欄為空白：",
    trailing_blank_pass
)
print("最終QC通過：", final_qc_pass)
print("最終QC未通過：", final_qc_fail)


print()
print("genotype count合計分布：")

for total, count in (
    genotype_total_counts.most_common()
):

    print(
        "合計 =",
        total,
        "| 列數 =",
        count
    )


print()
print("染色體、性別與子模式分布：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| shift1_subpattern =",
        group[2],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for conversion, count in (
    conversion_counts.most_common()
):

    print(
        conversion,
        "：",
        count
    )


print()
print("問題範例：")

for example in problem_examples:

    print(example)

left_shift = 1 正式修復完成

正式修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_all_fixed.txt
修復稽核檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_all_fixed_audit.txt

shift1總列數： 223971
HomFLD數值擷取成功： 223971
HomFLD數值擷取失敗： 0
原始最後一欄為空白： 223971
最終QC通過： 0
最終QC未通過： 223971

genotype count合計分布：
合計 = 240 | 列數 = 220896
合計 = 163 | 列數 = 3075

染色體、性別與子模式分布：
specialSNP_chr = autosomal | gender_metrics = all | shift1_subpattern = shift1-all | 列數 = 220896
specialSNP_chr = X | gender_metrics = female | shift1_subpattern = shift1-X-female | 列數 = 3075

ConversionType分布：
NoMinorHom ： 221275
PolyHighResolution ： 2696

問題範例：
{'source_row_number': 89, 'FLD': '4.462', 'HomFLD': '-0.0327', 'HomRO': '1.69', 'genotype_total': 240, 'expected_total': 240, 'specialSNP_chr': 'autosomal', 'gender_metrics': 'all', 'ConversionType': 'NoMinorHom', 'Call Modified': '', 'raw_trailing_blank': True, 'fld_ok': True, 'homfld_ok': True, 'count_ok': True, 'structure_ok': True, 'cluster_ok':

In [47]:
# 載入需要的套件
import csv
import re
from collections import Counter


# ============================================================
# 1. 定義數字格式與檢查函數
# ============================================================

# 完整數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 判斷是否為純數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# 根據gender_metrics決定預期樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體類別
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# Cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# ============================================================
# 2. 建立各QC條件計數器
# ============================================================

total_rows = 0


# 每一個QC條件的通過列數
qc_pass_counts = Counter()


# 統計每列失敗條件的組合
failure_signature_counts = Counter()


# 統計各cluster mean欄位失敗列數
cluster_invalid_counts = Counter()


# 統計重要尾端欄位常見值
important_value_counts = {
    "HetSO": Counter(),
    "HomRO": Counter(),
    "MMD": Counter(),
    "MinorAlleleFrequency": Counter(),
    "H.W.p-Value": Counter(),
    "H.W.chisquared.statistic": Counter(),
    "Call Modified": Counter()
}


# 保存前20筆失敗範例
failed_examples = []


# ============================================================
# 3. 讀取shift1正式修正版
# ============================================================

with shift1_all_fixed_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 逐列檢查
    for row in reader:

        total_rows += 1


        # ----------------------------------------------------
        # 3-1. 前段數值欄位
        # ----------------------------------------------------

        fld_ok = is_numeric(
            row["FLD"]
        )

        homfld_ok = is_numeric(
            row["HomFLD"]
        )

        hetso_blank_ok = (
            row["HetSO"].strip() == ""
        )

        homro_ok = is_numeric_or_na(
            row["HomRO"]
        )


        # ----------------------------------------------------
        # 3-2. Genotype count
        # ----------------------------------------------------

        try:

            genotype_total = (
                int(row["n_AA"])
                + int(row["n_AB"])
                + int(row["n_BB"])
                + int(row["n_NC"])
            )

            expected_total = (
                expected_total_by_gender.get(
                    row["gender_metrics"]
                )
            )

            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        # ----------------------------------------------------
        # 3-3. 文字結構欄位
        # ----------------------------------------------------

        structure_ok = (
            row["hemizygous"] in {"0", "1"}
            and row["specialSNP_chr"]
            in valid_chr_values
            and row["gender_metrics"]
            in expected_total_by_gender
            and row["ConversionType"].strip() != ""
            and row["BestProbeset"]
            in {"0", "1"}
            and row["BestandRecommended"]
            in {"0", "1"}
            and row["HomHet"]
            in {"0", "1"}
        )


        # ----------------------------------------------------
        # 3-4. Cluster mean欄位
        # ----------------------------------------------------

        cluster_column_results = {
            column: is_numeric_or_na(
                row[column]
            )
            for column in cluster_mean_columns
        }


        cluster_ok = all(
            cluster_column_results.values()
        )


        # 分別統計哪個cluster欄位不合理
        for column, column_ok in (
            cluster_column_results.items()
        ):

            if not column_ok:
                cluster_invalid_counts[column] += 1


        # ----------------------------------------------------
        # 3-5. MinorAlleleFrequency
        # ----------------------------------------------------

        maf_text = (
            row["MinorAlleleFrequency"]
            .strip()
        )


        if is_na(maf_text):

            maf_ok = True

        else:

            try:

                maf_value = float(maf_text)

                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:

                maf_value = None
                maf_ok = False


        # ----------------------------------------------------
        # 3-6. H.W.p-Value
        # ----------------------------------------------------

        hwp_text = (
            row["H.W.p-Value"]
            .strip()
        )


        if is_na(hwp_text):

            hwp_ok = True

        else:

            try:

                hwp_value = float(hwp_text)

                hwp_ok = (
                    0 <= hwp_value <= 1
                )

            except ValueError:

                hwp_value = None
                hwp_ok = False


        # ----------------------------------------------------
        # 3-7. H.W.chisquared.statistic
        # ----------------------------------------------------

        hwchi_text = (
            row["H.W.chisquared.statistic"]
            .strip()
        )


        if is_na(hwchi_text):

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(hwchi_text)

                hwchi_ok = (
                    hwchi_value >= 0
                )

            except ValueError:

                hwchi_value = None
                hwchi_ok = False


        # ----------------------------------------------------
        # 3-8. Call Modified
        # ----------------------------------------------------

        call_modified_text = (
            row["Call Modified"]
            .strip()
            .upper()
        )


        call_modified_ok = (
            call_modified_text
            in {"TRUE", "FALSE"}
        )


        # ----------------------------------------------------
        # 3-9. 建立此列所有QC結果
        # ----------------------------------------------------

        row_qc_results = {
            "FLD_numeric":
                fld_ok,

            "HomFLD_numeric":
                homfld_ok,

            "HetSO_blank":
                hetso_blank_ok,

            "HomRO_numeric_or_NA":
                homro_ok,

            "genotype_count":
                count_ok,

            "structure":
                structure_ok,

            "cluster_mean":
                cluster_ok,

            "MAF":
                maf_ok,

            "H.W.p-Value":
                hwp_ok,

            "H.W.chisquared":
                hwchi_ok,

            "Call_Modified":
                call_modified_ok
        }


        # 累計各條件通過數
        for condition, condition_ok in (
            row_qc_results.items()
        ):

            if condition_ok:
                qc_pass_counts[condition] += 1


        # 找出本列失敗的條件
        failed_conditions = tuple(
            condition
            for condition, condition_ok
            in row_qc_results.items()
            if not condition_ok
        )


        # 統計失敗條件組合
        failure_signature_counts[
            failed_conditions
        ] += 1


        # ----------------------------------------------------
        # 3-10. 統計尾端欄位常見值
        # ----------------------------------------------------

        for column in important_value_counts:

            important_value_counts[column][
                row[column]
            ] += 1


        # ----------------------------------------------------
        # 3-11. 保存少量失敗範例
        # ----------------------------------------------------

        if (
            failed_conditions
            and len(failed_examples) < 20
        ):

            failed_examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "failed_conditions":
                        failed_conditions,

                    "HetSO":
                        row["HetSO"],

                    "HomRO":
                        row["HomRO"],

                    "NC.meanX":
                        row["NC.meanX"],

                    "NC.meanY":
                        row["NC.meanY"],

                    "MMD":
                        row["MMD"],

                    "MinorAlleleFrequency":
                        row["MinorAlleleFrequency"],

                    "H.W.p-Value":
                        row["H.W.p-Value"],

                    "H.W.chisquared.statistic":
                        row[
                            "H.W.chisquared.statistic"
                        ],

                    "Call Modified":
                        row["Call Modified"]
                }
            )


# ============================================================
# 4. 顯示診斷結果
# ============================================================

print("left_shift = 1 最終QC失敗原因診斷")
print()

print("總列數：", total_rows)


print()
print("各QC條件通過列數：")

for condition in [
    "FLD_numeric",
    "HomFLD_numeric",
    "HetSO_blank",
    "HomRO_numeric_or_NA",
    "genotype_count",
    "structure",
    "cluster_mean",
    "MAF",
    "H.W.p-Value",
    "H.W.chisquared",
    "Call_Modified"
]:

    print(
        condition,
        "：",
        qc_pass_counts[condition]
    )


print()
print("主要失敗條件組合：")

for signature, count in (
    failure_signature_counts.most_common(10)
):

    print(
        signature,
        "：",
        count
    )


print()
print("各cluster mean欄位不合理列數：")

for column in cluster_mean_columns:

    print(
        column,
        "：",
        cluster_invalid_counts[column]
    )


print()
print("重要尾端欄位常見值：")

for column, value_counter in (
    important_value_counts.items()
):

    print()
    print("欄位：", column)

    for value, count in (
        value_counter.most_common(10)
    ):

        print(
            repr(value),
            "：",
            count
        )


print()
print("前5筆失敗範例：")

for example in failed_examples[:5]:

    print(example)

left_shift = 1 最終QC失敗原因診斷

總列數： 223971

各QC條件通過列數：
FLD_numeric ： 223971
HomFLD_numeric ： 223971
HetSO_blank ： 223971
HomRO_numeric_or_NA ： 223971
genotype_count ： 223971
structure ： 223971
cluster_mean ： 223971
MAF ： 8928
H.W.p-Value ： 223971
H.W.chisquared ： 0
Call_Modified ： 0

主要失敗條件組合：
('MAF', 'H.W.chisquared', 'Call_Modified') ： 215043
('H.W.chisquared', 'Call_Modified') ： 8928

各cluster mean欄位不合理列數：
AA.meanX ： 0
AA.meanY ： 0
AB.meanX ： 0
AB.meanY ： 0
BB.meanX ： 0
BB.meanY ： 0
NC.meanX ： 0
NC.meanY ： 0

重要尾端欄位常見值：

欄位： HetSO
'' ： 223971

欄位： HomRO
'1.913' ： 170
'2.007' ： 167
'1.839' ： 166
'2.026' ： 166
'2.105' ： 163
'2.01' ： 163
'1.872' ： 162
'1.885' ： 162
'1.888' ： 162
'1.857' ： 161

欄位： MMD
'?�數??0.0146' ： 12869
'?�數??0.0125' ： 11776
'?�數??0.0167' ： 11738
'?�數??0.0104' ： 11522
'?�數??0.0063' ： 11016
'?�數??0.0083' ： 10912
'?�數??0.0187' ： 9581
'?�數??0.0042' ： 9411
'?�數??0.0021' ： 9290
'?�數??0.0208' ： 8663

欄位： MinorAlleleFrequency
'1' ： 207551
'0.608' ： 2672
'0.375' ： 1541
'0.607' 

發現位移一個的尾端有位移

In [48]:
# 載入需要的套件
import csv
import re
from collections import Counter


# ============================================================
# 1. 設定驗證報告位置
# ============================================================

shift1_tail_v2_report_path = (
    output_folder
    / "shift1_tail_mapping_v2_validation.txt"
)


# ============================================================
# 2. 定義數字格式
# ============================================================

# 完整純數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 從亂碼字串尾端擷取數字
# 例如：
# '?�數??0.025' → 0.025
ending_number_pattern = re.compile(
    r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?)\s*$"
)


# 判斷是否為完整數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# 判斷是否為NA
def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


# 判斷是否為數字或NA
def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# ============================================================
# 3. 建立計數器
# ============================================================

total_rows = 0

candidate_mmd_pass = 0

maf_direct_numeric = 0
maf_extracted_from_garbled = 0
maf_extraction_failed = 0
candidate_maf_pass = 0

candidate_hwp_pass = 0

raw_hwp_boolean_pass = 0
raw_hwchi_blank_pass = 0
raw_call_modified_blank_pass = 0

candidate_call_modified_pass = 0

all_conditions_pass = 0
all_conditions_fail = 0


# 統計原始MMD資料型態
raw_mmd_type_counts = Counter()

# 統計原始H.W.p-Value內容
raw_hwp_value_counts = Counter()

# 保存前20筆失敗範例
failed_examples = []

# 保存前5筆候選對應範例
candidate_examples = []


# ============================================================
# 4. 從原始classified_back_path驗證shift1
# ============================================================

with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # source_row_number是完整680,865列中的原始順序
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只檢查left_shift = 1
        if row["left_shift"] != "1":
            continue

        total_rows += 1


        # ====================================================
        # 4-1. 候選MMD
        # ====================================================

        # 向右移一欄後：
        # 真正MMD應來自原始NC.meanY
        candidate_mmd = (
            row["NC.meanY"].strip()
        )


        # MMD可為數字或NA
        mmd_ok = is_numeric_or_na(
            candidate_mmd
        )


        if mmd_ok:
            candidate_mmd_pass += 1


        # ====================================================
        # 4-2. 候選MinorAlleleFrequency
        # ====================================================

        # 真正MAF藏在原始MMD欄位
        raw_mmd = row["MMD"].strip()


        # 原始MMD本身就是純數字
        if is_numeric(raw_mmd):

            candidate_maf = raw_mmd

            maf_direct_numeric += 1

            raw_mmd_type_counts[
                "direct_numeric"
            ] += 1


        # 原始MMD為NA
        elif is_na(raw_mmd):

            candidate_maf = "NA"

            raw_mmd_type_counts[
                "NA"
            ] += 1


        # 原始MMD含亂碼，嘗試擷取尾端數字
        else:

            maf_match = ending_number_pattern.search(
                raw_mmd
            )


            if maf_match:

                candidate_maf = (
                    maf_match.group(1)
                )

                maf_extracted_from_garbled += 1

                raw_mmd_type_counts[
                    "garbled_with_numeric_tail"
                ] += 1

            else:

                candidate_maf = ""

                maf_extraction_failed += 1

                raw_mmd_type_counts[
                    "unrecognized"
                ] += 1


        # 驗證MAF值域
        if is_na(candidate_maf):

            maf_ok = True

        else:

            try:

                maf_value = float(
                    candidate_maf
                )

                maf_ok = (
                    0 <= maf_value <= 0.5
                )

            except ValueError:

                maf_value = None
                maf_ok = False


        if maf_ok:
            candidate_maf_pass += 1


        # ====================================================
        # 4-3. 候選H.W.p-Value
        # ====================================================

        # 原始MinorAlleleFrequency
        # 實際對應真正的H.W.p-Value
        candidate_hwp = (
            row["MinorAlleleFrequency"]
            .strip()
        )


        if is_na(candidate_hwp):

            hwp_ok = True

        else:

            try:

                hwp_value = float(
                    candidate_hwp
                )

                hwp_ok = (
                    0 <= hwp_value <= 1
                )

            except ValueError:

                hwp_value = None
                hwp_ok = False


        if hwp_ok:
            candidate_hwp_pass += 1


        # ====================================================
        # 4-4. 驗證原始H.W.p-Value是Call Modified
        # ====================================================

        raw_hwp = (
            row["H.W.p-Value"]
            .strip()
            .upper()
        )


        raw_hwp_value_counts[
            raw_hwp
        ] += 1


        raw_hwp_is_boolean = (
            raw_hwp in {"TRUE", "FALSE"}
        )


        if raw_hwp_is_boolean:
            raw_hwp_boolean_pass += 1


        # 候選Call Modified
        candidate_call_modified = raw_hwp


        call_modified_ok = (
            candidate_call_modified
            in {"TRUE", "FALSE"}
        )


        if call_modified_ok:
            candidate_call_modified_pass += 1


        # ====================================================
        # 4-5. 驗證最後兩個原始欄位為空白
        # ====================================================

        raw_hwchi_blank = (
            row["H.W.chisquared.statistic"]
            .strip() == ""
        )


        if raw_hwchi_blank:
            raw_hwchi_blank_pass += 1


        raw_call_blank = (
            row["Call Modified"]
            .strip() == ""
        )


        if raw_call_blank:
            raw_call_modified_blank_pass += 1


        # ====================================================
        # 4-6. 綜合判斷
        # ====================================================

        all_ok = (
            mmd_ok
            and maf_ok
            and hwp_ok
            and raw_hwp_is_boolean
            and raw_hwchi_blank
            and raw_call_blank
            and call_modified_ok
        )


        if all_ok:

            all_conditions_pass += 1

        else:

            all_conditions_fail += 1


            # 保存前20筆失敗範例
            if len(failed_examples) < 20:

                failed_examples.append(
                    {
                        "source_row_number":
                            source_row_number,

                        "candidate_MMD":
                            candidate_mmd,

                        "raw_MMD":
                            raw_mmd,

                        "candidate_MAF":
                            candidate_maf,

                        "candidate_H.W.p-Value":
                            candidate_hwp,

                        "raw_H.W.p-Value":
                            raw_hwp,

                        "candidate_Call Modified":
                            candidate_call_modified,

                        "raw_H.W.chisquared":
                            row[
                                "H.W.chisquared.statistic"
                            ],

                        "raw_Call Modified":
                            row["Call Modified"],

                        "mmd_ok":
                            mmd_ok,

                        "maf_ok":
                            maf_ok,

                        "hwp_ok":
                            hwp_ok,

                        "raw_hwp_is_boolean":
                            raw_hwp_is_boolean,

                        "raw_hwchi_blank":
                            raw_hwchi_blank,

                        "raw_call_blank":
                            raw_call_blank
                    }
                )


        # 保存前5筆候選修正範例
        if len(candidate_examples) < 5:

            candidate_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "fixed_MMD":
                        candidate_mmd,

                    "fixed_MinorAlleleFrequency":
                        candidate_maf,

                    "fixed_H.W.p-Value":
                        candidate_hwp,

                    # 此模式沒有可靠的HWE卡方值
                    "fixed_H.W.chisquared.statistic":
                        "",

                    "fixed_Call Modified":
                        candidate_call_modified
                }
            )


# ============================================================
# 5. 寫入驗證報告
# ============================================================

with shift1_tail_v2_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "left_shift = 1 tail mapping v2 validation\n"
    )

    report.write(
        f"Total rows: {total_rows}\n\n"
    )

    report.write(
        f"Candidate MMD pass: "
        f"{candidate_mmd_pass}\n"
    )

    report.write(
        f"MAF direct numeric: "
        f"{maf_direct_numeric}\n"
    )

    report.write(
        f"MAF extracted from garbled: "
        f"{maf_extracted_from_garbled}\n"
    )

    report.write(
        f"MAF extraction failed: "
        f"{maf_extraction_failed}\n"
    )

    report.write(
        f"Candidate MAF pass: "
        f"{candidate_maf_pass}\n"
    )

    report.write(
        f"Candidate H.W.p-Value pass: "
        f"{candidate_hwp_pass}\n"
    )

    report.write(
        f"Raw H.W.p-Value boolean pass: "
        f"{raw_hwp_boolean_pass}\n"
    )

    report.write(
        f"Raw H.W.chisquared blank pass: "
        f"{raw_hwchi_blank_pass}\n"
    )

    report.write(
        f"Raw Call Modified blank pass: "
        f"{raw_call_modified_blank_pass}\n"
    )

    report.write(
        f"Candidate Call Modified pass: "
        f"{candidate_call_modified_pass}\n"
    )

    report.write(
        f"All conditions pass: "
        f"{all_conditions_pass}\n"
    )

    report.write(
        f"All conditions fail: "
        f"{all_conditions_fail}\n\n"
    )


    report.write("=" * 80 + "\n")
    report.write("Raw MMD type distribution\n")
    report.write("=" * 80 + "\n")

    for value_type, count in (
        raw_mmd_type_counts.most_common()
    ):

        report.write(
            f"{value_type}: {count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Raw H.W.p-Value distribution\n")
    report.write("=" * 80 + "\n")

    for value, count in (
        raw_hwp_value_counts.most_common()
    ):

        report.write(
            f"{repr(value)}: {count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Failed examples\n")
    report.write("=" * 80 + "\n")

    for example in failed_examples:

        report.write(
            repr(example) + "\n"
        )


# ============================================================
# 6. 顯示結果
# ============================================================

print("left_shift = 1 尾端候選對應v2驗證完成")
print()

print(
    "驗證報告：",
    shift1_tail_v2_report_path
)

print()

print("shift1總列數：", total_rows)
print("候選MMD合理：", candidate_mmd_pass)

print(
    "MAF原本為純數字：",
    maf_direct_numeric
)

print(
    "MAF從亂碼尾端擷取：",
    maf_extracted_from_garbled
)

print(
    "MAF擷取失敗：",
    maf_extraction_failed
)

print("候選MAF合理：", candidate_maf_pass)
print("候選H.W.p-Value合理：", candidate_hwp_pass)

print(
    "原始H.W.p-Value為布林值：",
    raw_hwp_boolean_pass
)

print(
    "原始H.W.chisquared為空白：",
    raw_hwchi_blank_pass
)

print(
    "原始Call Modified為空白：",
    raw_call_modified_blank_pass
)

print(
    "候選Call Modified合理：",
    candidate_call_modified_pass
)

print(
    "所有條件同時通過：",
    all_conditions_pass
)

print(
    "未通過列數：",
    all_conditions_fail
)


print()
print("原始MMD型態分布：")

for value_type, count in (
    raw_mmd_type_counts.most_common()
):

    print(
        value_type,
        "：",
        count
    )


print()
print("原始H.W.p-Value內容分布：")

for value, count in (
    raw_hwp_value_counts.most_common()
):

    print(
        repr(value),
        "：",
        count
    )


print()
print("前5筆候選修正範例：")

for example in candidate_examples:

    print(example)


print()
print("前5筆未通過範例：")

for example in failed_examples[:5]:

    print(example)

left_shift = 1 尾端候選對應v2驗證完成

驗證報告： /Users/sheena/Desktop/summerintern/split_data/shift1_tail_mapping_v2_validation.txt

shift1總列數： 223971
候選MMD合理： 0
MAF原本為純數字： 223971
MAF從亂碼尾端擷取： 0
MAF擷取失敗： 0
候選MAF合理： 8928
候選H.W.p-Value合理： 223971
原始H.W.p-Value為布林值： 223971
原始H.W.chisquared為空白： 223971
原始Call Modified為空白： 223971
候選Call Modified合理： 223971
所有條件同時通過： 0
未通過列數： 223971

原始MMD型態分布：
direct_numeric ： 223971

原始H.W.p-Value內容分布：
'FALSE' ： 223971

前5筆候選修正範例：
{'source_row_number': 89, 'fixed_MMD': '?�數??0.0272', 'fixed_MinorAlleleFrequency': '1', 'fixed_H.W.p-Value': 'NA', 'fixed_H.W.chisquared.statistic': '', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': 91, 'fixed_MMD': '?�數??0.0063', 'fixed_MinorAlleleFrequency': '1', 'fixed_H.W.p-Value': 'NA', 'fixed_H.W.chisquared.statistic': '', 'fixed_Call Modified': 'FALSE'}
{'source_row_number': 93, 'fixed_MMD': '?�數??0.0167', 'fixed_MinorAlleleFrequency': '1', 'fixed_H.W.p-Value': 'NA', 'fixed_H.W.chisquared.statistic': '', 'fixed_Call Modified': 'FA

從其他位移格式的MMD缺失 去回推位移一個是否也是在MMD缺失
把位移一個的資料中 MMD之後的5個欄位分割出來

In [52]:
# 載入套件
import csv
import re
import math


# ============================================================
# 1. 設定輸出檔案
# ============================================================

# 原始尾端切割檔
shift1_tail_raw_path = (
    output_folder
    / "genotype_back_shift1_tail_raw.txt"
)


# 修正後尾端檔
shift1_tail_fixed_path = (
    output_folder
    / "genotype_back_shift1_tail_fixed.txt"
)


# ============================================================
# 2. 定義數字格式
# ============================================================

# 完整數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 找出字串中的數字
# 原始NC.meanY是「亂碼文字＋MAF數值」
number_token_pattern = re.compile(
    r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?"
)


# 判斷是否為完整數字
def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


# ============================================================
# 3. 建立計數器
# ============================================================

total_rows = 0

maf_extract_success = 0
maf_extract_failed = 0

maf_qc_pass = 0
hwp_qc_pass = 0
hwchi_qc_pass = 0
call_modified_qc_pass = 0
trailing_blank_pass = 0

all_qc_pass = 0
all_qc_fail = 0


# 保存前20筆問題範例
problem_examples = []


# ============================================================
# 4. 讀取原始classified back檔案
# ============================================================

with classified_back_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift1_tail_raw_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as raw_output_file, \
shift1_tail_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as fixed_output_file:

    # 讀取完整後段資料
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # ========================================================
    # 4-1. 設定原始尾端切割檔欄位
    # ========================================================

    raw_tail_fields = [
        "source_row_number",
        "left_shift",

        "raw_NC.meanY",
        "raw_MMD",
        "raw_MinorAlleleFrequency",
        "raw_H.W.p-Value",
        "raw_H.W.chisquared.statistic",
        "raw_Call Modified"
    ]


    raw_writer = csv.DictWriter(
        raw_output_file,
        fieldnames=raw_tail_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    raw_writer.writeheader()


    # ========================================================
    # 4-2. 設定修正後尾端檔欄位
    # ========================================================

    fixed_tail_fields = [
        "source_row_number",

        "MMD",
        "MinorAlleleFrequency",
        "H.W.p-Value",
        "H.W.chisquared.statistic",
        "Call Modified",

        "MAF_source",
        "tail_repair_status"
    ]


    fixed_writer = csv.DictWriter(
        fixed_output_file,
        fieldnames=fixed_tail_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    fixed_writer.writeheader()


    # ========================================================
    # 4-3. 逐列切割並修正
    # ========================================================

    # source_row_number代表完整680,865列中的原始順序
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        # 只處理left_shift = 1
        if row["left_shift"] != "1":
            continue

        total_rows += 1


        # ----------------------------------------------------
        # 保存原始尾端資料
        # ----------------------------------------------------

        raw_ncmeany = (
            row["NC.meanY"].strip()
        )

        raw_mmd = (
            row["MMD"].strip()
        )

        raw_minor_allele_frequency = (
            row["MinorAlleleFrequency"]
            .strip()
        )

        raw_hwp = (
            row["H.W.p-Value"].strip()
        )

        raw_hwchi = (
            row["H.W.chisquared.statistic"]
            .strip()
        )

        raw_call_modified = (
            row["Call Modified"].strip()
        )


        # 寫入原始尾端切割檔
        raw_writer.writerow(
            {
                "source_row_number":
                    source_row_number,

                "left_shift":
                    row["left_shift"],

                "raw_NC.meanY":
                    raw_ncmeany,

                "raw_MMD":
                    raw_mmd,

                "raw_MinorAlleleFrequency":
                    raw_minor_allele_frequency,

                "raw_H.W.p-Value":
                    raw_hwp,

                "raw_H.W.chisquared.statistic":
                    raw_hwchi,

                "raw_Call Modified":
                    raw_call_modified
            }
        )


        # ====================================================
        # 5. 修正尾端欄位
        # ====================================================

        # ----------------------------------------------------
        # 5-1. MMD缺失
        # ----------------------------------------------------

        fixed_mmd = ""


        # ----------------------------------------------------
        # 5-2. 從原始NC.meanY擷取MAF
        # ----------------------------------------------------

        numeric_tokens = (
            number_token_pattern.findall(
                raw_ncmeany
            )
        )


        # 先前已確認每列只有一個數字
        if len(numeric_tokens) == 1:

            fixed_maf = numeric_tokens[0]

            maf_extract_success += 1

        else:

            fixed_maf = ""

            maf_extract_failed += 1


        # 驗證MAF介於0與0.5
        try:

            maf_value = float(
                fixed_maf
            )

            maf_ok = (
                math.isfinite(maf_value)
                and 0 <= maf_value <= 0.5
            )

        except ValueError:

            maf_value = None
            maf_ok = False


        if maf_ok:
            maf_qc_pass += 1


        # ----------------------------------------------------
        # 5-3. 修正H.W.p-Value
        # ----------------------------------------------------

        # 真正H.W.p-Value位於原始MMD
        fixed_hwp = raw_mmd


        try:

            hwp_value = float(
                fixed_hwp
            )

            hwp_ok = (
                math.isfinite(hwp_value)
                and 0 <= hwp_value <= 1
            )

        except ValueError:

            hwp_value = None
            hwp_ok = False


        if hwp_ok:
            hwp_qc_pass += 1


        # ----------------------------------------------------
        # 5-4. 修正H.W.chisquared.statistic
        # ----------------------------------------------------

        # 真正HWE卡方值位於原始MinorAlleleFrequency
        fixed_hwchi = (
            raw_minor_allele_frequency
        )


        try:

            hwchi_value = float(
                fixed_hwchi
            )

            # 卡方值只需大於或等於0
            hwchi_ok = (
                math.isfinite(hwchi_value)
                and hwchi_value >= 0
            )

        except ValueError:

            hwchi_value = None
            hwchi_ok = False


        if hwchi_ok:
            hwchi_qc_pass += 1


        # ----------------------------------------------------
        # 5-5. 修正Call Modified
        # ----------------------------------------------------

        # 原始H.W.p-Value欄位實際放的是FALSE
        fixed_call_modified = (
            raw_hwp.upper()
        )


        call_modified_ok = (
            fixed_call_modified
            in {"TRUE", "FALSE"}
        )


        if call_modified_ok:
            call_modified_qc_pass += 1


        # ----------------------------------------------------
        # 5-6. 檢查原始最後兩欄
        # ----------------------------------------------------

        # 因為資料已經提前，
        # 原始最後兩欄應全部為空白
        trailing_blank_ok = (
            raw_hwchi == ""
            and raw_call_modified == ""
        )


        if trailing_blank_ok:
            trailing_blank_pass += 1


        # ====================================================
        # 6. 尾端修復QC
        # ====================================================

        tail_ok = (
            fixed_mmd == ""
            and len(numeric_tokens) == 1
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
            and trailing_blank_ok
        )


        if tail_ok:

            tail_repair_status = "pass"

            all_qc_pass += 1

        else:

            tail_repair_status = "fail"

            all_qc_fail += 1


            # 保存前20筆問題資料
            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            source_row_number,

                        "raw_NC.meanY":
                            raw_ncmeany,

                        "numeric_tokens":
                            numeric_tokens,

                        "fixed_MAF":
                            fixed_maf,

                        "fixed_H.W.p-Value":
                            fixed_hwp,

                        "fixed_H.W.chisquared":
                            fixed_hwchi,

                        "fixed_Call Modified":
                            fixed_call_modified,

                        "raw_H.W.chisquared":
                            raw_hwchi,

                        "raw_Call Modified":
                            raw_call_modified,

                        "maf_ok":
                            maf_ok,

                        "hwp_ok":
                            hwp_ok,

                        "hwchi_ok":
                            hwchi_ok,

                        "call_modified_ok":
                            call_modified_ok,

                        "trailing_blank_ok":
                            trailing_blank_ok
                    }
                )


        # ====================================================
        # 7. 寫入修正後尾端檔
        # ====================================================

        fixed_writer.writerow(
            {
                "source_row_number":
                    source_row_number,

                "MMD":
                    fixed_mmd,

                "MinorAlleleFrequency":
                    fixed_maf,

                "H.W.p-Value":
                    fixed_hwp,

                "H.W.chisquared.statistic":
                    fixed_hwchi,

                "Call Modified":
                    fixed_call_modified,

                "MAF_source":
                    "raw_NC.meanY_numeric_token",

                "tail_repair_status":
                    tail_repair_status
            }
        )


# ============================================================
# 8. 顯示結果
# ============================================================

print("left_shift = 1 尾端切割與修正完成")
print()

print(
    "原始尾端切割檔：",
    shift1_tail_raw_path
)

print(
    "修正後尾端檔：",
    shift1_tail_fixed_path
)

print()

print("shift1尾端總列數：", total_rows)

print(
    "MAF擷取成功：",
    maf_extract_success
)

print(
    "MAF擷取失敗：",
    maf_extract_failed
)

print(
    "MAF QC通過：",
    maf_qc_pass
)

print(
    "H.W.p-Value QC通過：",
    hwp_qc_pass
)

print(
    "H.W.chisquared QC通過：",
    hwchi_qc_pass
)

print(
    "Call Modified QC通過：",
    call_modified_qc_pass
)

print(
    "原始最後兩欄為空白：",
    trailing_blank_pass
)

print(
    "尾端修復全部通過：",
    all_qc_pass
)

print(
    "尾端修復未通過：",
    all_qc_fail
)


print()
print("問題範例：")

for example in problem_examples[:5]:

    print(example)

left_shift = 1 尾端切割與修正完成

原始尾端切割檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_tail_raw.txt
修正後尾端檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_tail_fixed.txt

shift1尾端總列數： 223971
MAF擷取成功： 223971
MAF擷取失敗： 0
MAF QC通過： 223971
H.W.p-Value QC通過： 223971
H.W.chisquared QC通過： 0
Call Modified QC通過： 223971
原始最後兩欄為空白： 223971
尾端修復全部通過： 0
尾端修復未通過： 223971

問題範例：
{'source_row_number': 89, 'raw_NC.meanY': '?�數??0.0272', 'numeric_tokens': ['0.0272'], 'fixed_MAF': '0.0272', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared': 'NA', 'fixed_Call Modified': 'FALSE', 'raw_H.W.chisquared': '', 'raw_Call Modified': '', 'maf_ok': True, 'hwp_ok': True, 'hwchi_ok': False, 'call_modified_ok': True, 'trailing_blank_ok': True}
{'source_row_number': 91, 'raw_NC.meanY': '?�數??0.0063', 'numeric_tokens': ['0.0063'], 'fixed_MAF': '0.0063', 'fixed_H.W.p-Value': '1', 'fixed_H.W.chisquared': 'NA', 'fixed_Call Modified': 'FALSE', 'raw_H.W.chisquared': '', 'raw_Call Modified': '', 

In [53]:
# 載入套件
import csv
import re
from collections import Counter


# 建立內容分布計數器
raw_minor_value_counts = Counter()


# 建立資料型態計數器
raw_minor_type_counts = Counter()


# 完整數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


# 保存前10筆資料
examples = []


# 讀取剛才切割出的shift1原始尾端檔
with shift1_tail_raw_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    for row in reader:

        # 取得原始MinorAlleleFrequency欄位
        value = (
            row["raw_MinorAlleleFrequency"]
            .strip()
        )


        # 統計實際內容
        raw_minor_value_counts[value] += 1


        # 判斷資料型態
        if value == "":

            value_type = "empty"

        elif value.upper() == "NA":

            value_type = "NA"

        elif full_number_pattern.fullmatch(value):

            value_type = "numeric"

        else:

            value_type = "text"


        raw_minor_type_counts[value_type] += 1


        # 保存前10筆範例
        if len(examples) < 10:

            examples.append(
                {
                    "source_row_number":
                        row["source_row_number"],

                    "raw_MinorAlleleFrequency":
                        value,

                    "value_type":
                        value_type
                }
            )


# 顯示結果
print(
    "原始MinorAlleleFrequency資料型態分布："
)

for value_type, count in (
    raw_minor_type_counts.most_common()
):

    print(
        value_type,
        "：",
        count
    )


print()
print("最常見的實際內容：")

for value, count in (
    raw_minor_value_counts.most_common(20)
):

    print(
        repr(value),
        "：",
        count
    )


print()
print("前10筆範例：")

for example in examples:

    print(example)

原始MinorAlleleFrequency資料型態分布：
NA ： 223971

最常見的實際內容：
'NA' ： 223971

前10筆範例：
{'source_row_number': '89', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '91', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '93', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '98', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '101', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '103', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '105', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '107', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '108', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}
{'source_row_number': '109', 'raw_MinorAlleleFrequency': 'NA', 'value_type': 'NA'}


In [54]:
# 載入套件
import csv
import re
import math


# ============================================================
# 1. 定義數字格式
# ============================================================

# 找出原始NC.meanY中的數字
number_token_pattern = re.compile(
    r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?"
)


# ============================================================
# 2. 建立計數器
# ============================================================

total_rows = 0

maf_extract_success = 0
maf_extract_failed = 0

maf_qc_pass = 0
hwp_qc_pass = 0
hwchi_qc_pass = 0
call_modified_qc_pass = 0
trailing_blank_pass = 0

all_qc_pass = 0
all_qc_fail = 0


# 保存前20筆問題資料
problem_examples = []


# ============================================================
# 3. 從原始尾端切割檔重新建立修正版
# ============================================================

with shift1_tail_raw_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift1_tail_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file:

    # 讀取原始尾端切割檔
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 修正後尾端欄位
    fixed_tail_fields = [
        "source_row_number",

        "MMD",
        "MinorAlleleFrequency",
        "H.W.p-Value",
        "H.W.chisquared.statistic",
        "Call Modified",

        "MAF_source",
        "tail_repair_status"
    ]


    # 建立輸出寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=fixed_tail_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    writer.writeheader()


    # 逐列修復
    for row in reader:

        total_rows += 1


        # 取得原始尾端內容
        raw_ncmeany = (
            row["raw_NC.meanY"].strip()
        )

        raw_mmd = (
            row["raw_MMD"].strip()
        )

        raw_minor = (
            row["raw_MinorAlleleFrequency"]
            .strip()
        )

        raw_hwp = (
            row["raw_H.W.p-Value"].strip()
        )

        raw_hwchi = (
            row["raw_H.W.chisquared.statistic"]
            .strip()
        )

        raw_call = (
            row["raw_Call Modified"].strip()
        )


        # ====================================================
        # 4. 修正MMD
        # ====================================================

        # 真正MMD沒有輸出
        fixed_mmd = ""


        # ====================================================
        # 5. 修正MinorAlleleFrequency
        # ====================================================

        # 從原始NC.meanY擷取唯一數字
        numeric_tokens = (
            number_token_pattern.findall(
                raw_ncmeany
            )
        )


        if len(numeric_tokens) == 1:

            fixed_maf = numeric_tokens[0]

            maf_extract_success += 1

        else:

            fixed_maf = ""

            maf_extract_failed += 1


        # 驗證MAF介於0與0.5
        try:

            maf_value = float(
                fixed_maf
            )

            maf_ok = (
                math.isfinite(maf_value)
                and 0 <= maf_value <= 0.5
            )

        except ValueError:

            maf_value = None
            maf_ok = False


        if maf_ok:
            maf_qc_pass += 1


        # ====================================================
        # 6. 修正H.W.p-Value
        # ====================================================

        # 原始MMD實際存放HWE p-value
        fixed_hwp = raw_mmd


        try:

            hwp_value = float(
                fixed_hwp
            )

            hwp_ok = (
                math.isfinite(hwp_value)
                and 0 <= hwp_value <= 1
            )

        except ValueError:

            hwp_value = None
            hwp_ok = False


        if hwp_ok:
            hwp_qc_pass += 1


        # ====================================================
        # 7. 修正H.W.chisquared.statistic
        # ====================================================

        # 原始MinorAlleleFrequency實際存放卡方欄位
        # 本批資料全部為NA
        fixed_hwchi = raw_minor


        # NA是合法缺失值
        if fixed_hwchi.upper() == "NA":

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(
                    fixed_hwchi
                )

                hwchi_ok = (
                    math.isfinite(hwchi_value)
                    and hwchi_value >= 0
                )

            except ValueError:

                hwchi_value = None
                hwchi_ok = False


        if hwchi_ok:
            hwchi_qc_pass += 1


        # ====================================================
        # 8. 修正Call Modified
        # ====================================================

        # 原始H.W.p-Value實際存放FALSE
        fixed_call_modified = (
            raw_hwp.upper()
        )


        call_modified_ok = (
            fixed_call_modified
            in {"TRUE", "FALSE"}
        )


        if call_modified_ok:
            call_modified_qc_pass += 1


        # ====================================================
        # 9. 檢查原始最後兩欄
        # ====================================================

        trailing_blank_ok = (
            raw_hwchi == ""
            and raw_call == ""
        )


        if trailing_blank_ok:
            trailing_blank_pass += 1


        # ====================================================
        # 10. 綜合QC
        # ====================================================

        tail_ok = (
            fixed_mmd == ""
            and len(numeric_tokens) == 1
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
            and trailing_blank_ok
        )


        if tail_ok:

            tail_repair_status = "pass"
            all_qc_pass += 1

        else:

            tail_repair_status = "fail"
            all_qc_fail += 1


            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            row["source_row_number"],

                        "raw_NC.meanY":
                            raw_ncmeany,

                        "fixed_MAF":
                            fixed_maf,

                        "fixed_H.W.p-Value":
                            fixed_hwp,

                        "fixed_H.W.chisquared":
                            fixed_hwchi,

                        "fixed_Call Modified":
                            fixed_call_modified,

                        "maf_ok":
                            maf_ok,

                        "hwp_ok":
                            hwp_ok,

                        "hwchi_ok":
                            hwchi_ok,

                        "call_modified_ok":
                            call_modified_ok,

                        "trailing_blank_ok":
                            trailing_blank_ok
                    }
                )


        # ====================================================
        # 11. 寫入修正後尾端檔
        # ====================================================

        writer.writerow(
            {
                "source_row_number":
                    row["source_row_number"],

                "MMD":
                    fixed_mmd,

                "MinorAlleleFrequency":
                    fixed_maf,

                "H.W.p-Value":
                    fixed_hwp,

                "H.W.chisquared.statistic":
                    fixed_hwchi,

                "Call Modified":
                    fixed_call_modified,

                "MAF_source":
                    "raw_NC.meanY_numeric_token",

                "tail_repair_status":
                    tail_repair_status
            }
        )


# ============================================================
# 12. 顯示結果
# ============================================================

print("left_shift = 1 尾端修正版重新建立完成")
print()

print(
    "修正後尾端檔：",
    shift1_tail_fixed_path
)

print()

print("shift1尾端總列數：", total_rows)
print("MAF擷取成功：", maf_extract_success)
print("MAF擷取失敗：", maf_extract_failed)
print("MAF QC通過：", maf_qc_pass)
print("H.W.p-Value QC通過：", hwp_qc_pass)
print(
    "H.W.chisquared QC通過：",
    hwchi_qc_pass
)
print(
    "Call Modified QC通過：",
    call_modified_qc_pass
)
print(
    "原始最後兩欄為空白：",
    trailing_blank_pass
)
print("尾端修復全部通過：", all_qc_pass)
print("尾端修復未通過：", all_qc_fail)


print()
print("問題範例：")

for example in problem_examples[:5]:

    print(example)

left_shift = 1 尾端修正版重新建立完成

修正後尾端檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_tail_fixed.txt

shift1尾端總列數： 223971
MAF擷取成功： 223971
MAF擷取失敗： 0
MAF QC通過： 223971
H.W.p-Value QC通過： 223971
H.W.chisquared QC通過： 223971
Call Modified QC通過： 223971
原始最後兩欄為空白： 223971
尾端修復全部通過： 223971
尾端修復未通過： 0

問題範例：


In [55]:
# 載入套件
import csv
import re
import math
from collections import Counter


# ============================================================
# 1. 設定輸出檔案
# ============================================================

# shift1最終完整修正版
shift1_final_fixed_path = (
    output_folder
    / "genotype_back_shift1_final_fixed.txt"
)


# 尾端合併稽核檔
shift1_tail_merge_audit_path = (
    output_folder
    / "genotype_back_shift1_tail_merge_audit.txt"
)


# ============================================================
# 2. 定義檢查函數
# ============================================================

# 完整數字格式
full_number_pattern = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?$"
)


def is_numeric(value):

    if value is None:
        return False

    return bool(
        full_number_pattern.fullmatch(
            value.strip()
        )
    )


def is_na(value):

    if value is None:
        return False

    return value.strip().upper() == "NA"


def is_numeric_or_na(value):

    return (
        is_numeric(value)
        or is_na(value)
    )


# 不同性別群組的預期樣本數
expected_total_by_gender = {
    "all": 240,
    "female": 163,
    "male": 77
}


# 合理的染色體類別
valid_chr_values = {
    "autosomal",
    "X",
    "Y",
    "MT"
}


# Cluster mean欄位
cluster_mean_columns = [
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY"
]


# ============================================================
# 3. 讀取修正後尾端檔
# ============================================================

# key：source_row_number
# value：該列修正後的尾端資料
tail_fixed_lookup = {}


tail_file_rows = 0
tail_duplicate_ids = 0
tail_pass_rows = 0
tail_fail_rows = 0


with shift1_tail_fixed_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as tail_file:

    tail_reader = csv.DictReader(
        tail_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    for row in tail_reader:

        tail_file_rows += 1

        source_id = (
            row["source_row_number"].strip()
        )


        # 檢查source_row_number是否重複
        if source_id in tail_fixed_lookup:

            tail_duplicate_ids += 1


        tail_fixed_lookup[source_id] = row


        if row["tail_repair_status"] == "pass":

            tail_pass_rows += 1

        else:

            tail_fail_rows += 1


# ============================================================
# 4. 建立合併計數器
# ============================================================

shift1_input_rows = 0

tail_matched_rows = 0
tail_unmatched_rows = 0

source_id_duplicate_in_main = 0

final_qc_pass = 0
final_qc_fail = 0


# 記錄主檔中已出現的source_row_number
seen_main_source_ids = set()


# 統計合併後的群組
group_counts = Counter()

# 統計ConversionType
conversion_counts = Counter()

# 保存前20筆問題範例
problem_examples = []


# ============================================================
# 5. 將尾端修正版合併回shift1完整資料
# ============================================================

with shift1_all_fixed_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift1_final_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file, \
shift1_tail_merge_audit_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as audit_file:

    # 讀取上一版shift1完整修正版
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 新增尾端合併狀態欄位
    output_fields = list(reader.fieldnames)

    if "tail_merge_status" not in output_fields:

        output_fields.append(
            "tail_merge_status"
        )


    # 建立最終完整檔寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 稽核檔欄位
    audit_fields = [
        "source_row_number",

        "old_MMD",
        "old_MinorAlleleFrequency",
        "old_H.W.p-Value",
        "old_H.W.chisquared.statistic",
        "old_Call Modified",

        "new_MMD",
        "new_MinorAlleleFrequency",
        "new_H.W.p-Value",
        "new_H.W.chisquared.statistic",
        "new_Call Modified",

        "tail_repair_status",
        "tail_merge_status",
        "final_qc_status"
    ]


    audit_writer = csv.DictWriter(
        audit_file,
        fieldnames=audit_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )

    audit_writer.writeheader()


    # 逐列合併
    for row in reader:

        shift1_input_rows += 1

        source_id = (
            row["source_row_number"].strip()
        )


        # 檢查主檔source_row_number是否重複
        if source_id in seen_main_source_ids:

            source_id_duplicate_in_main += 1

        else:

            seen_main_source_ids.add(
                source_id
            )


        # 保存合併前資料
        old_values = {
            "MMD":
                row["MMD"],

            "MinorAlleleFrequency":
                row["MinorAlleleFrequency"],

            "H.W.p-Value":
                row["H.W.p-Value"],

            "H.W.chisquared.statistic":
                row["H.W.chisquared.statistic"],

            "Call Modified":
                row["Call Modified"]
        }


        # ====================================================
        # 5-1. 依source_row_number尋找尾端修正版
        # ====================================================

        tail_row = tail_fixed_lookup.get(
            source_id
        )


        if tail_row is not None:

            tail_matched_rows += 1


            # 只覆蓋5個尾端欄位
            row["MMD"] = (
                tail_row["MMD"]
            )

            row["MinorAlleleFrequency"] = (
                tail_row[
                    "MinorAlleleFrequency"
                ]
            )

            row["H.W.p-Value"] = (
                tail_row["H.W.p-Value"]
            )

            row["H.W.chisquared.statistic"] = (
                tail_row[
                    "H.W.chisquared.statistic"
                ]
            )

            row["Call Modified"] = (
                tail_row["Call Modified"]
                .strip()
                .upper()
            )


            tail_repair_status = (
                tail_row[
                    "tail_repair_status"
                ]
            )

            tail_merge_status = "matched"

        else:

            tail_unmatched_rows += 1

            tail_repair_status = "missing"

            tail_merge_status = "unmatched"


        # ====================================================
        # 5-2. 重新驗證前段與中段
        # ====================================================

        fld_ok = is_numeric(
            row["FLD"]
        )

        homfld_ok = is_numeric(
            row["HomFLD"]
        )

        # shift1真正的HetSO缺失
        hetso_ok = (
            row["HetSO"].strip() == ""
        )

        homro_ok = is_numeric_or_na(
            row["HomRO"]
        )


        # Genotype count QC
        try:

            genotype_total = (
                int(row["n_AA"])
                + int(row["n_AB"])
                + int(row["n_BB"])
                + int(row["n_NC"])
            )

            expected_total = (
                expected_total_by_gender.get(
                    row["gender_metrics"]
                )
            )

            count_ok = (
                expected_total is not None
                and genotype_total == expected_total
            )

        except (ValueError, TypeError):

            genotype_total = None
            expected_total = None
            count_ok = False


        # 文字結構QC
        structure_ok = (
            row["hemizygous"] in {"0", "1"}
            and row["specialSNP_chr"]
            in valid_chr_values
            and row["gender_metrics"]
            in expected_total_by_gender
            and row["ConversionType"].strip() != ""
            and row["BestProbeset"] in {"0", "1"}
            and row["BestandRecommended"] in {"0", "1"}
            and row["HomHet"] in {"0", "1"}
        )


        # Cluster mean QC
        cluster_ok = all(
            is_numeric_or_na(
                row[column]
            )
            for column in cluster_mean_columns
        )


        # ====================================================
        # 5-3. 驗證合併後尾端
        # ====================================================

        # shift1真正的MMD缺失
        mmd_ok = (
            row["MMD"].strip() == ""
        )


        # MAF應介於0至0.5
        try:

            maf_value = float(
                row["MinorAlleleFrequency"]
            )

            maf_ok = (
                math.isfinite(maf_value)
                and 0 <= maf_value <= 0.5
            )

        except ValueError:

            maf_ok = False


        # HWE p-value應介於0至1
        try:

            hwp_value = float(
                row["H.W.p-Value"]
            )

            hwp_ok = (
                math.isfinite(hwp_value)
                and 0 <= hwp_value <= 1
            )

        except ValueError:

            hwp_ok = False


        # HWE卡方值可為NA或非負數
        hwchi_text = (
            row["H.W.chisquared.statistic"]
            .strip()
        )


        if hwchi_text.upper() == "NA":

            hwchi_ok = True

        else:

            try:

                hwchi_value = float(
                    hwchi_text
                )

                hwchi_ok = (
                    math.isfinite(hwchi_value)
                    and hwchi_value >= 0
                )

            except ValueError:

                hwchi_ok = False


        # Call Modified應為布林值
        call_modified_ok = (
            row["Call Modified"]
            in {"TRUE", "FALSE"}
        )


        # ====================================================
        # 5-4. 最終QC
        # ====================================================

        final_ok = (
            tail_merge_status == "matched"
            and tail_repair_status == "pass"
            and fld_ok
            and homfld_ok
            and hetso_ok
            and homro_ok
            and count_ok
            and structure_ok
            and cluster_ok
            and mmd_ok
            and maf_ok
            and hwp_ok
            and hwchi_ok
            and call_modified_ok
        )


        if final_ok:

            final_status = "pass"
            final_qc_pass += 1

        else:

            final_status = "fail"
            final_qc_fail += 1


            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            source_id,

                        "tail_merge_status":
                            tail_merge_status,

                        "tail_repair_status":
                            tail_repair_status,

                        "fld_ok":
                            fld_ok,

                        "homfld_ok":
                            homfld_ok,

                        "hetso_ok":
                            hetso_ok,

                        "homro_ok":
                            homro_ok,

                        "count_ok":
                            count_ok,

                        "structure_ok":
                            structure_ok,

                        "cluster_ok":
                            cluster_ok,

                        "mmd_ok":
                            mmd_ok,

                        "maf_ok":
                            maf_ok,

                        "hwp_ok":
                            hwp_ok,

                        "hwchi_ok":
                            hwchi_ok,

                        "call_modified_ok":
                            call_modified_ok
                    }
                )


        # ====================================================
        # 5-5. 更新管理欄位
        # ====================================================

        row["repair_group"] = (
            "shift1-repaired-two-missing-fields"
        )

        row["tail_merge_status"] = (
            tail_merge_status
        )

        row["final_qc_status"] = (
            final_status
        )


        # 統計群組
        group_counts[
            (
                row["specialSNP_chr"],
                row["gender_metrics"]
            )
        ] += 1


        conversion_counts[
            row["ConversionType"]
        ] += 1


        # 寫入最終完整檔
        writer.writerow(row)


        # 寫入合併稽核檔
        audit_writer.writerow(
            {
                "source_row_number":
                    source_id,

                "old_MMD":
                    old_values["MMD"],

                "old_MinorAlleleFrequency":
                    old_values[
                        "MinorAlleleFrequency"
                    ],

                "old_H.W.p-Value":
                    old_values["H.W.p-Value"],

                "old_H.W.chisquared.statistic":
                    old_values[
                        "H.W.chisquared.statistic"
                    ],

                "old_Call Modified":
                    old_values["Call Modified"],

                "new_MMD":
                    row["MMD"],

                "new_MinorAlleleFrequency":
                    row[
                        "MinorAlleleFrequency"
                    ],

                "new_H.W.p-Value":
                    row["H.W.p-Value"],

                "new_H.W.chisquared.statistic":
                    row[
                        "H.W.chisquared.statistic"
                    ],

                "new_Call Modified":
                    row["Call Modified"],

                "tail_repair_status":
                    tail_repair_status,

                "tail_merge_status":
                    tail_merge_status,

                "final_qc_status":
                    final_status
            }
        )


# ============================================================
# 6. 檢查尾端檔是否有未被主檔使用的列
# ============================================================

unused_tail_ids = (
    set(tail_fixed_lookup.keys())
    - seen_main_source_ids
)


# ============================================================
# 7. 顯示結果
# ============================================================

print("left_shift = 1 尾端合併完成")
print()

print(
    "最終完整修正版：",
    shift1_final_fixed_path
)

print(
    "尾端合併稽核檔：",
    shift1_tail_merge_audit_path
)

print()

print("尾端修正版列數：", tail_file_rows)
print("尾端修復pass：", tail_pass_rows)
print("尾端修復fail：", tail_fail_rows)
print("尾端source ID重複：", tail_duplicate_ids)

print()
print("shift1主檔列數：", shift1_input_rows)
print("成功配對尾端：", tail_matched_rows)
print("未配對尾端：", tail_unmatched_rows)
print(
    "主檔source ID重複：",
    source_id_duplicate_in_main
)
print(
    "尾端檔未被使用的列數：",
    len(unused_tail_ids)
)

print()
print("最終QC通過：", final_qc_pass)
print("最終QC未通過：", final_qc_fail)


print()
print("染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for conversion, count in (
    conversion_counts.most_common()
):

    print(
        conversion,
        "：",
        count
    )


print()
print("問題範例：")

for example in problem_examples[:5]:

    print(example)

left_shift = 1 尾端合併完成

最終完整修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_final_fixed.txt
尾端合併稽核檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift1_tail_merge_audit.txt

尾端修正版列數： 223971
尾端修復pass： 223971
尾端修復fail： 0
尾端source ID重複： 0

shift1主檔列數： 223971
成功配對尾端： 223971
未配對尾端： 0
主檔source ID重複： 0
尾端檔未被使用的列數： 0

最終QC通過： 223971
最終QC未通過： 0

染色體與性別群組：
specialSNP_chr = autosomal | gender_metrics = all | 列數 = 220896
specialSNP_chr = X | gender_metrics = female | 列數 = 3075

ConversionType分布：
NoMinorHom ： 221275
PolyHighResolution ： 2696

問題範例：


In [56]:
# 載入套件
import csv
from collections import Counter


# ============================================================
# 1. 設定檢查報告
# ============================================================

shift2_premerge_report_path = (
    output_folder
    / "shift2_premerge_validation.txt"
)


# ============================================================
# 2. 建立計數器
# ============================================================

total_rows = 0

source_id_missing = 0
source_id_duplicate = 0

seen_source_ids = set()


# 統計重要欄位
left_shift_counts = Counter()
group_counts = Counter()
conversion_counts = Counter()


# 統計可能存在的QC狀態欄位
status_counts = {
    "repair_status": Counter(),
    "final_qc_status": Counter(),
    "stage1_qc_status": Counter(),
    "tail_repair_status": Counter()
}


# 統計重要欄位缺失
missing_key_field_counts = Counter()


# 保存前10筆資料範例
row_examples = []


# ============================================================
# 3. 讀取shift2完整修正版
# ============================================================

with shift2_all_fixed_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.DictReader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 保存實際欄位名稱
    shift2_fieldnames = (
        reader.fieldnames
        if reader.fieldnames is not None
        else []
    )


    # 逐列檢查
    for row in reader:

        total_rows += 1


        # ----------------------------------------------------
        # 3-1. 檢查source_row_number
        # ----------------------------------------------------

        source_id = (
            row.get(
                "source_row_number",
                ""
            ).strip()
        )


        if source_id == "":

            source_id_missing += 1

        elif source_id in seen_source_ids:

            source_id_duplicate += 1

        else:

            seen_source_ids.add(
                source_id
            )


        # ----------------------------------------------------
        # 3-2. 統計left_shift
        # ----------------------------------------------------

        left_shift_counts[
            row.get(
                "left_shift",
                ""
            ).strip()
        ] += 1


        # ----------------------------------------------------
        # 3-3. 統計染色體與性別群組
        # ----------------------------------------------------

        chromosome_group = (
            row.get(
                "specialSNP_chr",
                ""
            ).strip()
        )

        gender_group = (
            row.get(
                "gender_metrics",
                ""
            ).strip()
        )


        group_counts[
            (
                chromosome_group,
                gender_group
            )
        ] += 1


        # ----------------------------------------------------
        # 3-4. 統計ConversionType
        # ----------------------------------------------------

        conversion_counts[
            row.get(
                "ConversionType",
                ""
            ).strip()
        ] += 1


        # ----------------------------------------------------
        # 3-5. 統計已有的QC狀態欄位
        # ----------------------------------------------------

        for status_field in status_counts:

            if status_field in shift2_fieldnames:

                status_value = (
                    row.get(
                        status_field,
                        ""
                    ).strip()
                )

                status_counts[
                    status_field
                ][status_value] += 1


        # ----------------------------------------------------
        # 3-6. 檢查合併需要的重要欄位
        # ----------------------------------------------------

        key_fields = [
            "left_shift",
            "n_AA",
            "n_AB",
            "n_BB",
            "n_NC",
            "hemizygous",
            "specialSNP_chr",
            "gender_metrics",
            "ConversionType",
            "BestProbeset",
            "BestandRecommended",
            "HomHet",
            "Call Modified"
        ]


        for field in key_fields:

            # 只檢查欄位是否存在
            # 部分數值欄位合法地可能是空白
            if field not in shift2_fieldnames:

                missing_key_field_counts[
                    field
                ] += 1


        # 保存前10筆範例
        if len(row_examples) < 10:

            row_examples.append(
                {
                    "source_row_number":
                        source_id,

                    "left_shift":
                        row.get(
                            "left_shift",
                            ""
                        ),

                    "specialSNP_chr":
                        chromosome_group,

                    "gender_metrics":
                        gender_group,

                    "ConversionType":
                        row.get(
                            "ConversionType",
                            ""
                        ),

                    "repair_status":
                        row.get(
                            "repair_status",
                            ""
                        ),

                    "final_qc_status":
                        row.get(
                            "final_qc_status",
                            ""
                        )
                }
            )


# ============================================================
# 4. 寫入檢查報告
# ============================================================

with shift2_premerge_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "shift2 pre-merge validation\n\n"
    )


    report.write(
        f"Total rows: {total_rows}\n"
    )

    report.write(
        f"Source ID missing: "
        f"{source_id_missing}\n"
    )

    report.write(
        f"Source ID duplicate: "
        f"{source_id_duplicate}\n\n"
    )


    report.write("=" * 80 + "\n")
    report.write("Field names\n")
    report.write("=" * 80 + "\n")

    for index, field in enumerate(
        shift2_fieldnames,
        start=1
    ):

        report.write(
            f"{index}. {field}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Left shift distribution\n")
    report.write("=" * 80 + "\n")

    for value, count in (
        left_shift_counts.most_common()
    ):

        report.write(
            f"{repr(value)}: {count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Chromosome and gender groups\n")
    report.write("=" * 80 + "\n")

    for group, count in group_counts.most_common():

        report.write(
            f"specialSNP_chr={group[0]} | "
            f"gender_metrics={group[1]} | "
            f"count={count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("ConversionType distribution\n")
    report.write("=" * 80 + "\n")

    for value, count in (
        conversion_counts.most_common()
    ):

        report.write(
            f"{repr(value)}: {count}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Status fields\n")
    report.write("=" * 80 + "\n")

    for status_field, counter in (
        status_counts.items()
    ):

        if status_field in shift2_fieldnames:

            report.write(
                f"{status_field}: "
                f"{dict(counter)}\n"
            )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Missing required columns\n")
    report.write("=" * 80 + "\n")

    report.write(
        repr(
            dict(missing_key_field_counts)
        )
        + "\n"
    )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Examples\n")
    report.write("=" * 80 + "\n")

    for example in row_examples:

        report.write(
            repr(example) + "\n"
        )


# ============================================================
# 5. 顯示重要結果
# ============================================================

print("shift2合併前檢查完成")
print()

print(
    "檢查報告：",
    shift2_premerge_report_path
)

print()
print("shift2總列數：", total_rows)
print("source ID缺失：", source_id_missing)
print("source ID重複：", source_id_duplicate)


print()
print("left_shift分布：")

for value, count in (
    left_shift_counts.most_common()
):

    print(
        repr(value),
        "：",
        count
    )


print()
print("染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for value, count in (
    conversion_counts.most_common()
):

    print(
        repr(value),
        "：",
        count
    )


print()
print("目前存在的QC狀態欄位：")

for status_field, counter in (
    status_counts.items()
):

    if status_field in shift2_fieldnames:

        print(
            status_field,
            "：",
            dict(counter)
        )


print()
print("缺少的重要欄位：")

print(
    dict(
        missing_key_field_counts
    )
)


print()
print("前5筆資料範例：")

for example in row_examples[:5]:

    print(example)

shift2合併前檢查完成

檢查報告： /Users/sheena/Desktop/summerintern/split_data/shift2_premerge_validation.txt

shift2總列數： 3296
source ID缺失： 0
source ID重複： 0

left_shift分布：
'2' ： 3296

染色體與性別群組：
specialSNP_chr = X | gender_metrics = female | 列數 = 3058
specialSNP_chr = MT | gender_metrics = all | 列數 = 238

ConversionType分布：
'NoMinorHom' ： 1681
'PolyHighResolution' ： 1615

目前存在的QC狀態欄位：
repair_status ： {'success': 3296}
final_qc_status ： {'pass': 3296}

缺少的重要欄位：
{}

前5筆資料範例：
{'source_row_number': '88', 'left_shift': '2', 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution', 'repair_status': 'success', 'final_qc_status': 'pass'}
{'source_row_number': '141', 'left_shift': '2', 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'PolyHighResolution', 'repair_status': 'success', 'final_qc_status': 'pass'}
{'source_row_number': '459', 'left_shift': '2', 'specialSNP_chr': 'X', 'gender_metrics': 'female', 'ConversionType': 'NoMinorHom', 'repair_status': '

In [57]:
# 載入套件
import csv
from collections import Counter


# ============================================================
# 1. 設定輸出檔案
# ============================================================

shift2_final_fixed_path = (
    output_folder
    / "genotype_back_shift2_final_fixed.txt"
)


# ============================================================
# 2. 設定四組合併時共同使用的欄位
# ============================================================

# classified back原本的30個資料欄位
back_data_fields = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 標準輸出欄位
standard_output_fields = (
    [
        "source_row_number",
        "left_shift"
    ]
    + back_data_fields
    + [
        "repair_group",
        "final_qc_status"
    ]
)


# ============================================================
# 3. 建立計數器
# ============================================================

total_rows = 0

source_id_missing = 0
source_id_duplicate = 0

input_qc_pass = 0
input_qc_fail = 0

standard_qc_pass = 0
standard_qc_fail = 0


# 保存已出現的source ID
seen_source_ids = set()


# 統計群組
group_counts = Counter()

conversion_counts = Counter()


# 保存問題範例
problem_examples = []


# ============================================================
# 4. 建立shift2標準正式檔
# ============================================================

with shift2_all_fixed_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as input_file, \
shift2_final_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file:

    # 讀取先前修好的shift2檔案
    reader = csv.DictReader(
        input_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 建立標準輸出檔寫入器
    writer = csv.DictWriter(
        output_file,
        fieldnames=standard_output_fields,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n",
        extrasaction="ignore"
    )

    writer.writeheader()


    # 逐列整理
    for row in reader:

        total_rows += 1


        # ----------------------------------------------------
        # 4-1. 檢查source_row_number
        # ----------------------------------------------------

        source_id = (
            row.get(
                "source_row_number",
                ""
            ).strip()
        )


        if source_id == "":

            source_id_missing += 1

        elif source_id in seen_source_ids:

            source_id_duplicate += 1

        else:

            seen_source_ids.add(
                source_id
            )


        # ----------------------------------------------------
        # 4-2. 檢查原本QC狀態
        # ----------------------------------------------------

        original_final_status = (
            row.get(
                "final_qc_status",
                ""
            ).strip()
        )


        original_repair_status = (
            row.get(
                "repair_status",
                ""
            ).strip()
        )


        input_ok = (
            original_final_status == "pass"
            and original_repair_status == "success"
        )


        if input_ok:

            input_qc_pass += 1

        else:

            input_qc_fail += 1


        # ----------------------------------------------------
        # 4-3. 檢查基本結構
        # ----------------------------------------------------

        required_values_ok = (
            row.get("left_shift", "").strip() == "2"
            and source_id != ""
            and row.get(
                "specialSNP_chr",
                ""
            ).strip()
            in {"X", "MT"}
            and row.get(
                "gender_metrics",
                ""
            ).strip()
            in {"female", "all"}
        )


        # 檢查所有30個back欄位都存在
        all_columns_exist = all(
            field in reader.fieldnames
            for field in back_data_fields
        )


        standard_ok = (
            input_ok
            and required_values_ok
            and all_columns_exist
        )


        if standard_ok:

            final_status = "pass"
            standard_qc_pass += 1

        else:

            final_status = "fail"
            standard_qc_fail += 1


            if len(problem_examples) < 20:

                problem_examples.append(
                    {
                        "source_row_number":
                            source_id,

                        "left_shift":
                            row.get(
                                "left_shift",
                                ""
                            ),

                        "specialSNP_chr":
                            row.get(
                                "specialSNP_chr",
                                ""
                            ),

                        "gender_metrics":
                            row.get(
                                "gender_metrics",
                                ""
                            ),

                        "repair_status":
                            original_repair_status,

                        "original_final_qc_status":
                            original_final_status,

                        "required_values_ok":
                            required_values_ok,

                        "all_columns_exist":
                            all_columns_exist
                    }
                )


        # ----------------------------------------------------
        # 4-4. 建立標準輸出列
        # ----------------------------------------------------

        standard_row = {
            "source_row_number":
                source_id,

            "left_shift":
                row["left_shift"],

            "repair_group":
                "shift2-repaired",

            "final_qc_status":
                final_status
        }


        # 複製30個主要資料欄位
        for field in back_data_fields:

            standard_row[field] = (
                row.get(
                    field,
                    ""
                )
            )


        # 寫入標準正式檔
        writer.writerow(
            standard_row
        )


        # ----------------------------------------------------
        # 4-5. 統計群組
        # ----------------------------------------------------

        group_counts[
            (
                standard_row[
                    "specialSNP_chr"
                ],
                standard_row[
                    "gender_metrics"
                ]
            )
        ] += 1


        conversion_counts[
            standard_row[
                "ConversionType"
            ]
        ] += 1


# ============================================================
# 5. 顯示結果
# ============================================================

print("left_shift = 2 標準正式檔建立完成")
print()

print(
    "標準正式檔：",
    shift2_final_fixed_path
)

print()

print("shift2總列數：", total_rows)
print("source ID缺失：", source_id_missing)
print("source ID重複：", source_id_duplicate)

print()
print("原始QC通過：", input_qc_pass)
print("原始QC未通過：", input_qc_fail)

print()
print("標準格式QC通過：", standard_qc_pass)
print("標準格式QC未通過：", standard_qc_fail)


print()
print("染色體與性別群組：")

for group, count in group_counts.most_common():

    print(
        "specialSNP_chr =",
        group[0],
        "| gender_metrics =",
        group[1],
        "| 列數 =",
        count
    )


print()
print("ConversionType分布：")

for conversion, count in (
    conversion_counts.most_common()
):

    print(
        conversion,
        "：",
        count
    )


print()
print("問題範例：")

for example in problem_examples[:5]:

    print(example)

left_shift = 2 標準正式檔建立完成

標準正式檔： /Users/sheena/Desktop/summerintern/split_data/genotype_back_shift2_final_fixed.txt

shift2總列數： 3296
source ID缺失： 0
source ID重複： 0

原始QC通過： 3296
原始QC未通過： 0

標準格式QC通過： 3296
標準格式QC未通過： 0

染色體與性別群組：
specialSNP_chr = X | gender_metrics = female | 列數 = 3058
specialSNP_chr = MT | gender_metrics = all | 列數 = 238

ConversionType分布：
NoMinorHom ： 1681
PolyHighResolution ： 1615

問題範例：


In [59]:
# 載入套件
import csv
import sqlite3
from collections import Counter


# ============================================================
# 1. 設定輸出檔案
# ============================================================

# 四組合併後的最終後段資料
final_back_fixed_path = (
    output_folder
    / "genotype_back_48_77_final_fixed.txt"
)


# 暫存SQLite資料庫
# 使用SQLite可避免一次將680,865列全部放入記憶體
merge_database_path = (
    output_folder
    / "temporary_back_merge.sqlite"
)


# 合併檢查報告
final_back_merge_report_path = (
    output_folder
    / "genotype_back_48_77_final_merge_report.txt"
)


# 若先前有同名暫存資料庫，先刪除
if merge_database_path.exists():
    merge_database_path.unlink()


# ============================================================
# 2. 設定正式輸出欄位
# ============================================================

back_data_fields = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 最終後段資料的標準欄位
final_output_fields = (
    [
        "source_row_number",
        "left_shift"
    ]
    + back_data_fields
    + [
        "repair_group",
        "final_qc_status"
    ]
)


# ============================================================
# 3. 設定四個輸入檔案
# ============================================================

input_groups = [
    {
        "shift": "0",
        "path": shift0_all_fixed_path,
        "fallback_repair_group":
            "shift0-original",
        "expected_rows": 297913
    },
    {
        "shift": "1",
        "path": shift1_final_fixed_path,
        "fallback_repair_group":
            "shift1-repaired-two-missing-fields",
        "expected_rows": 223971
    },
    {
        "shift": "2",
        "path": shift2_final_fixed_path,
        "fallback_repair_group":
            "shift2-repaired",
        "expected_rows": 3296
    },
    {
        "shift": "3",
        "path": shift3_all_fixed_path,
        "fallback_repair_group":
            "shift3-repaired",
        "expected_rows": 155685
    }
]


expected_total_rows = 680865


# ============================================================
# 4. 建立SQLite資料庫
# ============================================================

connection = sqlite3.connect(
    merge_database_path
)

cursor = connection.cursor()


# 將欄位名稱安全地加上雙引號
def quote_sql_identifier(name):

    return '"' + name.replace('"', '""') + '"'


# 建立資料表
column_definitions = [
    '"source_row_number" INTEGER PRIMARY KEY'
]


# 其他欄位全部用TEXT保存
for field in final_output_fields[1:]:

    column_definitions.append(
        f"{quote_sql_identifier(field)} TEXT"
    )


create_table_sql = (
    "CREATE TABLE merged_back ("
    + ", ".join(column_definitions)
    + ")"
)


cursor.execute(create_table_sql)
connection.commit()


# 建立INSERT語法
quoted_output_fields = [
    quote_sql_identifier(field)
    for field in final_output_fields
]


insert_sql = (
    "INSERT INTO merged_back ("
    + ", ".join(quoted_output_fields)
    + ") VALUES ("
    + ", ".join(
        ["?"] * len(final_output_fields)
    )
    + ")"
)


# ============================================================
# 5. 建立合併計數器
# ============================================================

input_row_counts = Counter()
inserted_row_counts = Counter()

duplicate_source_ids = 0
invalid_source_ids = 0
wrong_shift_rows = 0
nonpass_input_rows = 0
missing_required_columns = Counter()

duplicate_examples = []
invalid_examples = []
wrong_shift_examples = []
nonpass_examples = []


# ============================================================
# 6. 將四組資料寫入SQLite
# ============================================================

for group in input_groups:

    expected_shift = group["shift"]
    input_path = group["path"]

    fallback_repair_group = (
        group["fallback_repair_group"]
    )


    print(
        "正在讀取：",
        input_path.name
    )


    # 檢查檔案是否存在
    if not input_path.exists():

        connection.close()

        raise FileNotFoundError(
            f"找不到檔案：{input_path}"
        )


    with input_path.open(
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as input_file:

        reader = csv.DictReader(
            input_file,
            delimiter="\t",
            quoting=csv.QUOTE_NONE
        )


        actual_fieldnames = (
            reader.fieldnames
            if reader.fieldnames is not None
            else []
        )


        # 檢查必要欄位是否存在
        required_input_fields = (
            [
                "source_row_number",
                "left_shift"
            ]
            + back_data_fields
            + [
                "final_qc_status"
            ]
        )


        missing_fields = [
            field
            for field in required_input_fields
            if field not in actual_fieldnames
        ]


        if missing_fields:

            for field in missing_fields:

                missing_required_columns[
                    f"shift{expected_shift}:{field}"
                ] += 1


            connection.close()

            raise ValueError(
                f"shift{expected_shift}缺少欄位："
                f"{missing_fields}"
            )


        # 開始一個交易，提高寫入速度
        connection.execute("BEGIN")


        for row in reader:

            input_row_counts[
                expected_shift
            ] += 1


            # --------------------------------------------
            # 6-1. 檢查source_row_number
            # --------------------------------------------

            source_text = (
                row["source_row_number"]
                .strip()
            )


            try:

                source_number = int(
                    source_text
                )

            except ValueError:

                invalid_source_ids += 1


                if len(invalid_examples) < 20:

                    invalid_examples.append(
                        {
                            "shift":
                                expected_shift,

                            "source_row_number":
                                source_text
                        }
                    )


                continue


            if not (
                1 <= source_number
                <= expected_total_rows
            ):

                invalid_source_ids += 1


                if len(invalid_examples) < 20:

                    invalid_examples.append(
                        {
                            "shift":
                                expected_shift,

                            "source_row_number":
                                source_number,

                            "reason":
                                "超出1至680865"
                        }
                    )


                continue


            # --------------------------------------------
            # 6-2. 檢查left_shift
            # --------------------------------------------

            actual_shift = (
                row["left_shift"]
                .strip()
            )


            if actual_shift != expected_shift:

                wrong_shift_rows += 1


                if len(wrong_shift_examples) < 20:

                    wrong_shift_examples.append(
                        {
                            "source_row_number":
                                source_number,

                            "expected_shift":
                                expected_shift,

                            "actual_shift":
                                actual_shift
                        }
                    )


            # --------------------------------------------
            # 6-3. 檢查原始QC狀態
            # --------------------------------------------

            input_final_status = (
                row["final_qc_status"]
                .strip()
                .lower()
            )


            if input_final_status != "pass":

                nonpass_input_rows += 1


                if len(nonpass_examples) < 20:

                    nonpass_examples.append(
                        {
                            "source_row_number":
                                source_number,

                            "shift":
                                expected_shift,

                            "final_qc_status":
                                input_final_status
                        }
                    )


            # --------------------------------------------
            # 6-4. 決定repair_group
            # --------------------------------------------

            repair_group = (
                row.get(
                    "repair_group",
                    ""
                ).strip()
            )


            if repair_group == "":

                repair_group = (
                    fallback_repair_group
                )


            # --------------------------------------------
            # 6-5. 建立標準輸出列
            # --------------------------------------------

            standard_row = {
                "source_row_number":
                    source_number,

                "left_shift":
                    actual_shift,

                "repair_group":
                    repair_group,

                "final_qc_status":
                    input_final_status
            }


            # 複製30個後段資料欄位
            for field in back_data_fields:

                standard_row[field] = (
                    row.get(
                        field,
                        ""
                    )
                )


            # 依final_output_fields順序產生資料
            insert_values = [
                standard_row[field]
                for field in final_output_fields
            ]


            # --------------------------------------------
            # 6-6. 寫入SQLite
            # --------------------------------------------

            try:

                cursor.execute(
                    insert_sql,
                    insert_values
                )

                inserted_row_counts[
                    expected_shift
                ] += 1


            except sqlite3.IntegrityError:

                # source_row_number為PRIMARY KEY
                # 發生IntegrityError通常代表重複
                duplicate_source_ids += 1


                if len(duplicate_examples) < 20:

                    duplicate_examples.append(
                        {
                            "source_row_number":
                                source_number,

                            "shift":
                                expected_shift
                        }
                    )


        connection.commit()


# ============================================================
# 7. 檢查SQLite中的資料
# ============================================================

cursor.execute(
    """
    SELECT
        COUNT(*),
        MIN(source_row_number),
        MAX(source_row_number)
    FROM merged_back
    """
)

database_count, minimum_source, maximum_source = (
    cursor.fetchone()
)


# 統計SQLite內的left_shift
cursor.execute(
    """
    SELECT left_shift, COUNT(*)
    FROM merged_back
    GROUP BY left_shift
    ORDER BY left_shift
    """
)

database_shift_counts = dict(
    cursor.fetchall()
)


# 統計最終QC
cursor.execute(
    """
    SELECT final_qc_status, COUNT(*)
    FROM merged_back
    GROUP BY final_qc_status
    """
)

database_qc_counts = dict(
    cursor.fetchall()
)


# ============================================================
# 8. 依source_row_number排序並輸出最終TSV
# ============================================================

select_sql = (
    "SELECT "
    + ", ".join(quoted_output_fields)
    + " FROM merged_back "
    + "ORDER BY source_row_number"
)


cursor.execute(select_sql)


missing_source_ids = []
unexpected_source_order = 0

written_rows = 0
expected_source_number = 1


with final_back_fixed_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as output_file:

    writer = csv.writer(
        output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 寫入欄位名稱
    writer.writerow(
        final_output_fields
    )


    # 依原始列號順序寫出
    for database_row in cursor:

        current_source_number = (
            int(database_row[0])
        )


        # 偵測缺少的列號
        while (
            expected_source_number
            < current_source_number
        ):

            if len(missing_source_ids) < 100:

                missing_source_ids.append(
                    expected_source_number
                )


            expected_source_number += 1


        # 檢查目前列號
        if (
            current_source_number
            != expected_source_number
        ):

            unexpected_source_order += 1


        writer.writerow(
            database_row
        )


        written_rows += 1
        expected_source_number = (
            current_source_number + 1
        )


# 檢查最後一列之後是否仍有缺少列號
while expected_source_number <= expected_total_rows:

    if len(missing_source_ids) < 100:

        missing_source_ids.append(
            expected_source_number
        )


    expected_source_number += 1


# 缺少列數
missing_source_count = (
    expected_total_rows
    - database_count
)


# ============================================================
# 9. 判定合併QC
# ============================================================

expected_shift_counts = {
    "0": 297913,
    "1": 223971,
    "2": 3296,
    "3": 155685
}


row_counts_ok = (
    database_count == expected_total_rows
    and written_rows == expected_total_rows
)


shift_counts_ok = (
    database_shift_counts
    == expected_shift_counts
)


source_ids_ok = (
    duplicate_source_ids == 0
    and invalid_source_ids == 0
    and missing_source_count == 0
    and minimum_source == 1
    and maximum_source == expected_total_rows
)


input_qc_ok = (
    nonpass_input_rows == 0
    and database_qc_counts.get(
        "pass",
        0
    ) == expected_total_rows
)


final_merge_ok = (
    row_counts_ok
    and shift_counts_ok
    and source_ids_ok
    and input_qc_ok
    and wrong_shift_rows == 0
)


# ============================================================
# 10. 寫入合併檢查報告
# ============================================================

with final_back_merge_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "Final back 48-77 merge report\n\n"
    )


    report.write(
        f"Expected total rows: "
        f"{expected_total_rows}\n"
    )

    report.write(
        f"Database rows: "
        f"{database_count}\n"
    )

    report.write(
        f"Written rows: "
        f"{written_rows}\n\n"
    )


    report.write(
        f"Minimum source ID: "
        f"{minimum_source}\n"
    )

    report.write(
        f"Maximum source ID: "
        f"{maximum_source}\n"
    )

    report.write(
        f"Duplicate source IDs: "
        f"{duplicate_source_ids}\n"
    )

    report.write(
        f"Invalid source IDs: "
        f"{invalid_source_ids}\n"
    )

    report.write(
        f"Missing source IDs: "
        f"{missing_source_count}\n\n"
    )


    report.write(
        f"Input row counts: "
        f"{dict(input_row_counts)}\n"
    )

    report.write(
        f"Inserted row counts: "
        f"{dict(inserted_row_counts)}\n"
    )

    report.write(
        f"Database shift counts: "
        f"{database_shift_counts}\n"
    )

    report.write(
        f"Database QC counts: "
        f"{database_qc_counts}\n\n"
    )


    report.write(
        f"Wrong shift rows: "
        f"{wrong_shift_rows}\n"
    )

    report.write(
        f"Non-pass input rows: "
        f"{nonpass_input_rows}\n"
    )

    report.write(
        f"Unexpected source order: "
        f"{unexpected_source_order}\n\n"
    )


    report.write(
        f"Row counts OK: "
        f"{row_counts_ok}\n"
    )

    report.write(
        f"Shift counts OK: "
        f"{shift_counts_ok}\n"
    )

    report.write(
        f"Source IDs OK: "
        f"{source_ids_ok}\n"
    )

    report.write(
        f"Input QC OK: "
        f"{input_qc_ok}\n"
    )

    report.write(
        f"Final merge OK: "
        f"{final_merge_ok}\n\n"
    )


    report.write(
        f"First missing source IDs: "
        f"{missing_source_ids}\n"
    )

    report.write(
        f"Duplicate examples: "
        f"{duplicate_examples}\n"
    )

    report.write(
        f"Invalid examples: "
        f"{invalid_examples}\n"
    )

    report.write(
        f"Wrong shift examples: "
        f"{wrong_shift_examples}\n"
    )

    report.write(
        f"Non-pass examples: "
        f"{nonpass_examples}\n"
    )


# ============================================================
# 11. 關閉資料庫
# ============================================================

connection.close()


# 合併完全成功後刪除暫存資料庫
if final_merge_ok and merge_database_path.exists():

    merge_database_path.unlink()


# ============================================================
# 12. 顯示結果
# ============================================================

print("四組後段資料合併完成")
print()

print(
    "最終後段修正版：",
    final_back_fixed_path
)

print(
    "合併檢查報告：",
    final_back_merge_report_path
)

print()

print("預期總列數：", expected_total_rows)
print("資料庫列數：", database_count)
print("實際輸出列數：", written_rows)

print()
print("最小source ID：", minimum_source)
print("最大source ID：", maximum_source)
print("重複source ID：", duplicate_source_ids)
print("無效source ID：", invalid_source_ids)
print("缺少source ID：", missing_source_count)

print()
print("各left_shift列數：")

for shift_value in ["0", "1", "2", "3"]:

    print(
        "left_shift =",
        shift_value,
        "｜列數 =",
        database_shift_counts.get(
            shift_value,
            0
        )
    )


print()
print("最終QC分布：", database_qc_counts)

print()
print("列數正確：", row_counts_ok)
print("shift分布正確：", shift_counts_ok)
print("source ID完整：", source_ids_ok)
print("所有輸入QC通過：", input_qc_ok)

print()
print("最終合併是否通過：", final_merge_ok)

print()
print(
    "前100個缺少的source ID：",
    missing_source_ids
)

正在讀取： genotype_back_shift0_all_fixed.txt
正在讀取： genotype_back_shift1_final_fixed.txt
正在讀取： genotype_back_shift2_final_fixed.txt
正在讀取： genotype_back_shift3_all_fixed.txt
四組後段資料合併完成

最終後段修正版： /Users/sheena/Desktop/summerintern/split_data/genotype_back_48_77_final_fixed.txt
合併檢查報告： /Users/sheena/Desktop/summerintern/split_data/genotype_back_48_77_final_merge_report.txt

預期總列數： 680865
資料庫列數： 680865
實際輸出列數： 680865

最小source ID： 1
最大source ID： 680865
重複source ID： 0
無效source ID： 0
缺少source ID： 0

各left_shift列數：
left_shift = 0 ｜列數 = 297913
left_shift = 1 ｜列數 = 223971
left_shift = 2 ｜列數 = 3296
left_shift = 3 ｜列數 = 155685

最終QC分布： {'pass': 680865}

列數正確： True
shift分布正確： True
source ID完整： True
所有輸入QC通過： True

最終合併是否通過： True

前100個缺少的source ID： []


In [60]:
# 載入套件
import csv


# ============================================================
# 1. 設定前段檢查報告
# ============================================================

front_validation_report_path = (
    output_folder
    / "genotype_front_1_47_validation.txt"
)


# ============================================================
# 2. 建立計數器
# ============================================================

front_total_rows = 0

short_rows = 0
long_rows = 0
correct_length_rows = 0

empty_probeset_id = 0
duplicate_probeset_id = 0

seen_probeset_ids = set()

first_examples = []
problem_examples = []


# ============================================================
# 3. 讀取第1–47欄檔案
# ============================================================

with front_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.reader(
        file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 讀取欄位名稱
    front_header = next(reader)


    front_column_count = len(
        front_header
    )


    # 逐列檢查
    for source_row_number, row in enumerate(
        reader,
        start=1
    ):

        front_total_rows += 1


        # ----------------------------------------------------
        # 3-1. 檢查每列欄位數
        # ----------------------------------------------------

        row_length = len(row)


        if row_length == front_column_count:

            correct_length_rows += 1

        elif row_length < front_column_count:

            short_rows += 1

        else:

            long_rows += 1


        # ----------------------------------------------------
        # 3-2. 檢查probeset_id
        # ----------------------------------------------------

        probeset_id = (
            row[0].strip()
            if len(row) >= 1
            else ""
        )


        if probeset_id == "":

            empty_probeset_id += 1

        elif probeset_id in seen_probeset_ids:

            duplicate_probeset_id += 1

        else:

            seen_probeset_ids.add(
                probeset_id
            )


        # ----------------------------------------------------
        # 3-3. 保存前5筆資料範例
        # ----------------------------------------------------

        if len(first_examples) < 5:

            first_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "row_length":
                        row_length,

                    "probeset_id":
                        probeset_id,

                    "first_5_values":
                        row[:5],

                    "last_5_values":
                        row[-5:]
                }
            )


        # ----------------------------------------------------
        # 3-4. 保存問題範例
        # ----------------------------------------------------

        row_ok = (
            row_length == front_column_count
            and probeset_id != ""
        )


        if (
            not row_ok
            and len(problem_examples) < 20
        ):

            problem_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "row_length":
                        row_length,

                    "expected_length":
                        front_column_count,

                    "probeset_id":
                        probeset_id,

                    "row_start":
                        row[:10],

                    "row_end":
                        row[-10:]
                }
            )


# ============================================================
# 4. 判定是否可以與後段合併
# ============================================================

front_row_count_ok = (
    front_total_rows == 680865
)


front_column_count_ok = (
    front_column_count == 47
)


front_row_structure_ok = (
    correct_length_rows == 680865
    and short_rows == 0
    and long_rows == 0
)


front_probeset_ok = (
    empty_probeset_id == 0
)


front_ready_for_merge = (
    front_row_count_ok
    and front_column_count_ok
    and front_row_structure_ok
    and front_probeset_ok
)


# ============================================================
# 5. 寫入檢查報告
# ============================================================

with front_validation_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "Front columns 1-47 validation\n\n"
    )

    report.write(
        f"Column count: "
        f"{front_column_count}\n"
    )

    report.write(
        f"Total rows: "
        f"{front_total_rows}\n"
    )

    report.write(
        f"Correct-length rows: "
        f"{correct_length_rows}\n"
    )

    report.write(
        f"Short rows: "
        f"{short_rows}\n"
    )

    report.write(
        f"Long rows: "
        f"{long_rows}\n"
    )

    report.write(
        f"Empty probeset_id: "
        f"{empty_probeset_id}\n"
    )

    report.write(
        f"Duplicate probeset_id: "
        f"{duplicate_probeset_id}\n\n"
    )


    report.write(
        f"Row count OK: "
        f"{front_row_count_ok}\n"
    )

    report.write(
        f"Column count OK: "
        f"{front_column_count_ok}\n"
    )

    report.write(
        f"Row structure OK: "
        f"{front_row_structure_ok}\n"
    )

    report.write(
        f"Probeset ID OK: "
        f"{front_probeset_ok}\n"
    )

    report.write(
        f"Ready for merge: "
        f"{front_ready_for_merge}\n\n"
    )


    report.write("=" * 80 + "\n")
    report.write("Header\n")
    report.write("=" * 80 + "\n")

    for index, field in enumerate(
        front_header,
        start=1
    ):

        report.write(
            f"{index}. {field}\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("First examples\n")
    report.write("=" * 80 + "\n")

    for example in first_examples:

        report.write(
            repr(example) + "\n"
        )


    report.write("\n" + "=" * 80 + "\n")
    report.write("Problem examples\n")
    report.write("=" * 80 + "\n")

    for example in problem_examples:

        report.write(
            repr(example) + "\n"
        )


# ============================================================
# 6. 顯示結果
# ============================================================

print("第1–47欄檢查完成")
print()

print(
    "檢查報告：",
    front_validation_report_path
)

print()

print("前段欄位數：", front_column_count)
print("前段總列數：", front_total_rows)
print("欄位數正確的列：", correct_length_rows)
print("欄位不足的列：", short_rows)
print("欄位過多的列：", long_rows)
print("probeset_id空白：", empty_probeset_id)
print("probeset_id重複：", duplicate_probeset_id)

print()
print("列數正確：", front_row_count_ok)
print("欄位數為47：", front_column_count_ok)
print("每列結構正確：", front_row_structure_ok)
print("probeset_id完整：", front_probeset_ok)

print()
print(
    "是否可與後段合併：",
    front_ready_for_merge
)

print()
print("前5筆範例：")

for example in first_examples:
    print(example)

print()
print("前5筆問題範例：")

for example in problem_examples[:5]:
    print(example)

第1–47欄檢查完成

檢查報告： /Users/sheena/Desktop/summerintern/split_data/genotype_front_1_47_validation.txt

前段欄位數： 47
前段總列數： 680865
欄位數正確的列： 680865
欄位不足的列： 0
欄位過多的列： 0
probeset_id空白： 0
probeset_id重複： 0

列數正確： True
欄位數為47： True
每列結構正確： True
probeset_id完整： True

是否可與後段合併： True

前5筆範例：
{'source_row_number': 1, 'row_length': 47, 'probeset_id': 'AFFX-SP-000001', 'first_5_values': ['AFFX-SP-000001', 'AA', 'AA', 'BB', 'AA'], 'last_5_values': ['', '', '', 'Affx-2643493', '100']}
{'source_row_number': 2, 'row_length': 47, 'probeset_id': 'AFFX-SP-000002', 'first_5_values': ['AFFX-SP-000002', 'AB', 'AB', 'AA', 'AA'], 'last_5_values': ['', '', '', 'Affx-7455925', '100']}
{'source_row_number': 3, 'row_length': 47, 'probeset_id': 'AFFX-SP-000003', 'first_5_values': ['AFFX-SP-000003', 'BB', 'AB', 'AA', 'AB'], 'last_5_values': ['', '', '', 'Affx-3102334', '100']}
{'source_row_number': 4, 'row_length': 47, 'probeset_id': 'AFFX-SP-000004', 'first_5_values': ['AFFX-SP-000004', 'AB', 'AB', 'AB', 'AB'], 'last_5_va

合併77欄

In [61]:
# 載入套件
import csv
from collections import Counter


# ============================================================
# 1. 設定輸出檔案
# ============================================================

# 正式77欄資料檔
# 欄位結構：原始第1–47欄 + 修正後第48–77欄
final_77_data_path = (
    output_folder
    / "genotype_77_columns_final_fixed.txt"
)


# 含原始列號、位移類型與QC狀態的稽核版本
final_77_with_qc_path = (
    output_folder
    / "genotype_77_columns_final_fixed_with_qc.txt"
)


# 合併檢查報告
final_77_merge_report_path = (
    output_folder
    / "genotype_77_columns_final_merge_report.txt"
)


# ============================================================
# 2. 設定後段30個正式欄位
# ============================================================

back_data_fields = [
    "FLD",
    "HomFLD",
    "HetSO",
    "HomRO",
    "nMinorAllele",
    "Nclus",
    "n_AA",
    "n_AB",
    "n_BB",
    "n_NC",
    "hemizygous",
    "specialSNP_chr",
    "gender_metrics",
    "ConversionType",
    "BestProbeset",
    "BestandRecommended",
    "HomHet",
    "AA.meanX",
    "AA.meanY",
    "AB.meanX",
    "AB.meanY",
    "BB.meanX",
    "BB.meanY",
    "NC.meanX",
    "NC.meanY",
    "MMD",
    "MinorAlleleFrequency",
    "H.W.p-Value",
    "H.W.chisquared.statistic",
    "Call Modified"
]


# 預期資料列數
expected_rows = 680865


# ============================================================
# 3. 建立計數器
# ============================================================

front_rows_read = 0
back_rows_read = 0
rows_written = 0

front_length_errors = 0
source_order_errors = 0
nonpass_back_rows = 0
invalid_shift_rows = 0

extra_back_rows = 0


# 統計修復群組
shift_counts = Counter()
repair_group_counts = Counter()
final_qc_counts = Counter()


# 保存前20筆問題資料
problem_examples = []


# ============================================================
# 4. 同時讀取前段與修正後後段
# ============================================================

with front_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as front_file, \
final_back_fixed_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as back_file, \
final_77_data_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as clean_output_file, \
final_77_with_qc_path.open(
    "w",
    encoding="utf-8",
    newline=""
) as qc_output_file:

    # 前段使用一般reader
    front_reader = csv.reader(
        front_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 後段使用DictReader
    back_reader = csv.DictReader(
        back_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE
    )


    # 讀取前段欄位名稱
    front_header = next(front_reader)


    # ========================================================
    # 4-1. 檢查欄位結構
    # ========================================================

    if len(front_header) != 47:

        raise ValueError(
            f"前段欄位數不是47，而是{len(front_header)}"
        )


    # 檢查後段需要的欄位是否全部存在
    required_back_fields = (
        [
            "source_row_number",
            "left_shift"
        ]
        + back_data_fields
        + [
            "repair_group",
            "final_qc_status"
        ]
    )


    missing_back_fields = [
        field
        for field in required_back_fields
        if field not in back_reader.fieldnames
    ]


    if missing_back_fields:

        raise ValueError(
            f"最終後段檔缺少欄位：{missing_back_fields}"
        )


    # 檢查前後段是否有重複欄位名稱
    overlapping_fields = sorted(
        set(front_header)
        & set(back_data_fields)
    )


    if overlapping_fields:

        raise ValueError(
            "前段與後段存在重複欄名："
            f"{overlapping_fields}"
        )


    # 正式77欄標題
    final_77_header = (
        front_header
        + back_data_fields
    )


    # 含QC資訊的標題
    final_qc_header = (
        ["source_row_number"]
        + final_77_header
        + [
            "left_shift",
            "repair_group",
            "final_qc_status"
        ]
    )


    # 再次確認欄位數
    if len(final_77_header) != 77:

        raise ValueError(
            f"合併後正式欄位數不是77，"
            f"而是{len(final_77_header)}"
        )


    if len(final_qc_header) != 81:

        raise ValueError(
            f"含QC版本欄位數不是81，"
            f"而是{len(final_qc_header)}"
        )


    # 建立正式資料寫入器
    clean_writer = csv.writer(
        clean_output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 建立QC版本寫入器
    qc_writer = csv.writer(
        qc_output_file,
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        escapechar="\\",
        lineterminator="\n"
    )


    # 寫入欄位名稱
    clean_writer.writerow(
        final_77_header
    )

    qc_writer.writerow(
        final_qc_header
    )


    # ========================================================
    # 4-2. 依原始列序逐列合併
    # ========================================================

    for source_row_number, front_row in enumerate(
        front_reader,
        start=1
    ):

        front_rows_read += 1


        # 從後段取得相同列序資料
        back_row = next(
            back_reader,
            None
        )


        # 後段提前結束
        if back_row is None:

            problem_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "problem":
                        "後段資料提前結束"
                }
            )

            break


        back_rows_read += 1


        # ----------------------------------------------------
        # 檢查前段欄位數
        # ----------------------------------------------------

        front_length_ok = (
            len(front_row) == 47
        )


        if not front_length_ok:

            front_length_errors += 1


        # ----------------------------------------------------
        # 檢查後段source_row_number
        # ----------------------------------------------------

        back_source_text = (
            back_row["source_row_number"]
            .strip()
        )


        try:

            back_source_number = int(
                back_source_text
            )

        except ValueError:

            back_source_number = None


        source_order_ok = (
            back_source_number
            == source_row_number
        )


        if not source_order_ok:

            source_order_errors += 1


        # ----------------------------------------------------
        # 檢查left_shift
        # ----------------------------------------------------

        left_shift = (
            back_row["left_shift"]
            .strip()
        )


        shift_ok = (
            left_shift
            in {"0", "1", "2", "3"}
        )


        if not shift_ok:

            invalid_shift_rows += 1


        # ----------------------------------------------------
        # 檢查後段QC狀態
        # ----------------------------------------------------

        final_qc_status = (
            back_row["final_qc_status"]
            .strip()
            .lower()
        )


        qc_ok = (
            final_qc_status == "pass"
        )


        if not qc_ok:

            nonpass_back_rows += 1


        repair_group = (
            back_row["repair_group"]
            .strip()
        )


        # ----------------------------------------------------
        # 建立後段30欄資料
        # ----------------------------------------------------

        back_values = [
            back_row[field]
            for field in back_data_fields
        ]


        # ----------------------------------------------------
        # 合併第1–47欄與第48–77欄
        # ----------------------------------------------------

        combined_77_row = (
            front_row
            + back_values
        )


        # 檢查合併後欄位數
        combined_length_ok = (
            len(combined_77_row) == 77
        )


        row_ok = (
            front_length_ok
            and source_order_ok
            and shift_ok
            and qc_ok
            and combined_length_ok
        )


        # ----------------------------------------------------
        # 問題範例
        # ----------------------------------------------------

        if (
            not row_ok
            and len(problem_examples) < 20
        ):

            problem_examples.append(
                {
                    "source_row_number":
                        source_row_number,

                    "back_source_number":
                        back_source_number,

                    "front_length":
                        len(front_row),

                    "combined_length":
                        len(combined_77_row),

                    "left_shift":
                        left_shift,

                    "repair_group":
                        repair_group,

                    "final_qc_status":
                        final_qc_status,

                    "front_length_ok":
                        front_length_ok,

                    "source_order_ok":
                        source_order_ok,

                    "shift_ok":
                        shift_ok,

                    "qc_ok":
                        qc_ok,

                    "combined_length_ok":
                        combined_length_ok
                }
            )


        # ----------------------------------------------------
        # 寫入正式77欄檔案
        # ----------------------------------------------------

        clean_writer.writerow(
            combined_77_row
        )


        # ----------------------------------------------------
        # 寫入含QC資訊版本
        # ----------------------------------------------------

        qc_writer.writerow(
            [
                source_row_number
            ]
            + combined_77_row
            + [
                left_shift,
                repair_group,
                final_qc_status
            ]
        )


        rows_written += 1


        # ----------------------------------------------------
        # 統計
        # ----------------------------------------------------

        shift_counts[
            left_shift
        ] += 1

        repair_group_counts[
            repair_group
        ] += 1

        final_qc_counts[
            final_qc_status
        ] += 1


    # ========================================================
    # 4-3. 檢查後段是否還有多餘資料
    # ========================================================

    for _ in back_reader:

        extra_back_rows += 1


# ============================================================
# 5. 最終QC判斷
# ============================================================

expected_shift_counts = {
    "0": 297913,
    "1": 223971,
    "2": 3296,
    "3": 155685
}


row_counts_ok = (
    front_rows_read == expected_rows
    and back_rows_read == expected_rows
    and rows_written == expected_rows
    and extra_back_rows == 0
)


source_order_all_ok = (
    source_order_errors == 0
)


column_structure_ok = (
    front_length_errors == 0
)


shift_distribution_ok = (
    dict(shift_counts)
    == expected_shift_counts
)


all_qc_pass = (
    nonpass_back_rows == 0
    and final_qc_counts.get(
        "pass",
        0
    ) == expected_rows
)


final_77_merge_ok = (
    row_counts_ok
    and source_order_all_ok
    and column_structure_ok
    and shift_distribution_ok
    and all_qc_pass
    and invalid_shift_rows == 0
)


# ============================================================
# 6. 寫入合併報告
# ============================================================

with final_77_merge_report_path.open(
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "Final 77-column genotype merge report\n\n"
    )


    report.write(
        f"Expected rows: {expected_rows}\n"
    )

    report.write(
        f"Front rows read: {front_rows_read}\n"
    )

    report.write(
        f"Back rows read: {back_rows_read}\n"
    )

    report.write(
        f"Rows written: {rows_written}\n"
    )

    report.write(
        f"Extra back rows: {extra_back_rows}\n\n"
    )


    report.write(
        f"Front column count: "
        f"{len(front_header)}\n"
    )

    report.write(
        f"Back data column count: "
        f"{len(back_data_fields)}\n"
    )

    report.write(
        f"Final clean column count: "
        f"{len(final_77_header)}\n"
    )

    report.write(
        f"Final QC column count: "
        f"{len(final_qc_header)}\n\n"
    )


    report.write(
        f"Front length errors: "
        f"{front_length_errors}\n"
    )

    report.write(
        f"Source order errors: "
        f"{source_order_errors}\n"
    )

    report.write(
        f"Invalid shift rows: "
        f"{invalid_shift_rows}\n"
    )

    report.write(
        f"Non-pass back rows: "
        f"{nonpass_back_rows}\n\n"
    )


    report.write(
        f"Shift counts: "
        f"{dict(shift_counts)}\n"
    )

    report.write(
        f"Repair group counts: "
        f"{dict(repair_group_counts)}\n"
    )

    report.write(
        f"Final QC counts: "
        f"{dict(final_qc_counts)}\n\n"
    )


    report.write(
        f"Row counts OK: "
        f"{row_counts_ok}\n"
    )

    report.write(
        f"Source order OK: "
        f"{source_order_all_ok}\n"
    )

    report.write(
        f"Column structure OK: "
        f"{column_structure_ok}\n"
    )

    report.write(
        f"Shift distribution OK: "
        f"{shift_distribution_ok}\n"
    )

    report.write(
        f"All QC pass: "
        f"{all_qc_pass}\n"
    )

    report.write(
        f"Final 77-column merge OK: "
        f"{final_77_merge_ok}\n\n"
    )


    report.write(
        f"Overlapping field names: "
        f"{overlapping_fields}\n"
    )

    report.write(
        f"Problem examples: "
        f"{problem_examples}\n"
    )


# ============================================================
# 7. 顯示結果
# ============================================================

print("第1–47欄與修正後第48–77欄合併完成")
print()

print(
    "正式77欄資料檔：",
    final_77_data_path
)

print(
    "含QC資訊版本：",
    final_77_with_qc_path
)

print(
    "合併檢查報告：",
    final_77_merge_report_path
)

print()

print("預期資料列數：", expected_rows)
print("前段讀取列數：", front_rows_read)
print("後段讀取列數：", back_rows_read)
print("實際輸出列數：", rows_written)
print("後段多餘列數：", extra_back_rows)

print()
print("正式資料欄位數：", len(final_77_header))
print("含QC版本欄位數：", len(final_qc_header))

print()
print("前段欄位數錯誤：", front_length_errors)
print("source順序錯誤：", source_order_errors)
print("無效left_shift：", invalid_shift_rows)
print("後段非pass列數：", nonpass_back_rows)

print()
print("各left_shift列數：")

for shift_value in ["0", "1", "2", "3"]:

    print(
        "left_shift =",
        shift_value,
        "｜列數 =",
        shift_counts.get(
            shift_value,
            0
        )
    )


print()
print("修復群組分布：")

for repair_group, count in (
    repair_group_counts.most_common()
):

    print(
        repair_group,
        "：",
        count
    )


print()
print("最終QC分布：", dict(final_qc_counts))

print()
print("列數正確：", row_counts_ok)
print("source順序正確：", source_order_all_ok)
print("欄位結構正確：", column_structure_ok)
print("shift分布正確：", shift_distribution_ok)
print("全部QC通過：", all_qc_pass)

print()
print(
    "最終77欄合併是否通過：",
    final_77_merge_ok
)

print()
print("問題範例：")

for example in problem_examples[:5]:

    print(example)

第1–47欄與修正後第48–77欄合併完成

正式77欄資料檔： /Users/sheena/Desktop/summerintern/split_data/genotype_77_columns_final_fixed.txt
含QC資訊版本： /Users/sheena/Desktop/summerintern/split_data/genotype_77_columns_final_fixed_with_qc.txt
合併檢查報告： /Users/sheena/Desktop/summerintern/split_data/genotype_77_columns_final_merge_report.txt

預期資料列數： 680865
前段讀取列數： 680865
後段讀取列數： 680865
實際輸出列數： 680865
後段多餘列數： 0

正式資料欄位數： 77
含QC版本欄位數： 81

前段欄位數錯誤： 0
source順序錯誤： 0
無效left_shift： 0
後段非pass列數： 0

各left_shift列數：
left_shift = 0 ｜列數 = 297913
left_shift = 1 ｜列數 = 223971
left_shift = 2 ｜列數 = 3296
left_shift = 3 ｜列數 = 155685

修復群組分布：
shift0-original ： 297913
shift1-repaired-two-missing-fields ： 223971
shift3-repaired ： 155685
shift2-repaired ： 3296

最終QC分布： {'pass': 680865}

列數正確： True
source順序正確： True
欄位結構正確： True
shift分布正確： True
全部QC通過： True

最終77欄合併是否通過： True

問題範例：
